### Tratar dados de monitoramento para as UFs e criar respectivas files 

##### Bibliotecas

In [158]:
import os, time, math, requests, pandas as pd
from datetime import datetime, timedelta, timezone
import pandas as pd
from collections import defaultdict
import re
import numpy as np
from pathlib import Path
import difflib
import unicodedata
from typing import Callable, Any, Optional
from functools import partial
from pandas.api.types import is_object_dtype, is_string_dtype
from collections import Counter
import unicodedata as ud

##### Dicionários e listas 

In [110]:
# Colunas que precisam conter nas files de dados de ESTAÇÃO para cada estado
mqar_campos = [
        "UF","ID_OEMA","CIDADE","ID_MMA","ID_MMA_COMPLETO","POLUENTE","COD_POLUENTE",
        "CD_MUN","COD_UF_IBGE","PROPRIETARIO","PROP_ENTIDADE","OPERADOR","OP_ENTIDADE",
        "LATITUDE","LONGITUDE","MOBILIDADE","CATEGORIA","FUNCIONAMENTO","METODO",
        "MARCA",'INICIO', 'FIM',"FINALIDADE","MONITORAR","FONTE","CALIBRACAO","REALOCACAO",
        "OBS_CALIBRACAO","DADOS_MONITORAMENTO","RECONHECIDA","OBS_GERAIS",
        "STATUS","CERTIFICACAO","REP_ESPACIAL_DECLARADA"
    ]

In [111]:
name_to_uf = {
    "acre":"AC","alagoas":"AL","amapa":"AP","amazonas":"AM","bahia":"BA","ceara":"CE",
    "distrito federal":"DF","espirito santo":"ES","goias":"GO","maranhao":"MA",
    "mato grosso":"MT","mato grosso do sul":"MS","minas gerais":"MG","para":"PA",
    "paraiba":"PB","parana":"PR","pernambuco":"PE","piaui":"PI","rio de janeiro":"RJ",
    "rio grande do norte":"RN","rio grande do sul":"RS","rondonia":"RO","roraima":"RR",
    "santa catarina":"SC","sao paulo":"SP","sergipe":"SE","tocantins":"TO"
}

In [112]:
UF_TO_IBGE = {
    "AC":12,"AL":27,"AP":16,"AM":13,"BA":29,"CE":23,"DF":53,"ES":32,"GO":52,"MA":21,
    "MT":51,"MS":50,"MG":31,"PA":15,"PB":25,"PR":41,"PE":26,"PI":22,"RJ":33,"RN":24,
    "RS":43,"RO":11,"RR":14,"SC":42,"SP":35,"SE":28,"TO":17
}

def sigla_to_ibge(uf): return UF_TO_IBGE[uf.upper()]

In [113]:
# Colunas que precisam conter nas files de dados de MONITORAMENTO para cada estado
mon_cols = ['DATETIME','ANO','MES','DIA','HORA','UNIDADE','QAQC_INTERNO','VALOR']

In [114]:
# Importar planilha com os códigos de poluentes
base = Path.cwd().parent  
out_dir = base / "data" / "dicionarios" 
out_dir.mkdir(parents=True, exist_ok=True)

df_cod = pd.read_csv(out_dir / 'CODIGO_POLUENTES.csv')

In [115]:
# Importar planilha com as respostas do formulário das UFs
base = Path.cwd().parent  
fr_dir = base / "data" 
fr_dir.mkdir(parents=True, exist_ok=True)

forms = pd.read_csv(fr_dir / '2025_Formulário_Coleta_Respostas_UFs.csv')

#Indice das colunas com respostas sobre rede de monitoramento
# for i, c in enumerate(forms.columns):
#    print(f"[{i}] {c}")

idxs = [6,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22] 

##### Concatenar planilhas de estações por UF e Monitoramento_QAr_BR

In [21]:
base = Path.cwd().parent 
df_dir = base / "data" / "DADOS_ESTACOES" 
df_dir.mkdir(parents=True, exist_ok=True)
ufs_dfs = load_csvs(df_dir, prefix=None, recursive=False, limit=None)

Loaded BA_estacoes.csv: (14, 31)
Loaded CE_estacoes.csv: (4, 34)
Loaded DF_estacoes.csv: (9, 31)
Loaded ES_estacoes.csv: (19, 31)
Loaded MA_estacoes.csv: (6, 31)
Loaded MG_estacoes.csv: (67, 28)
Loaded MT_estacoes.csv: (5, 31)
Loaded PB_estacoes.csv: (3, 34)
Loaded PE_estacoes.csv: (4, 34)
Loaded PR_estacoes.csv: (28, 31)
Loaded RJ_estacoes.csv: (100, 35)
Loaded RS_estacoes.csv: (19, 31)
Loaded SC_estacoes.csv: (4, 31)
Loaded SP_estacoes.csv: (101, 31)


In [22]:
for name, d in ufs_dfs.items(): clean_df_all_text(d, in_place=True)

NameError: name 'clean_df_all_text' is not defined

In [ ]:
uf_frame, uf_conflicts = merge_by_id_multi(
    ufs_dfs.copy(),
    id_cols=['ID_OEMA','POLUENTE'],
    source_priority=priority,
    drop_empty_strings=True,
)
uf_frame

In [ ]:
def explode_pollutants(df, col="POLUENTE"):
    out = df.copy()

    # turn "VOC,PM1, PM25 ,PM10" into ["VOC","PM1","PM25","PM10"]
    out[col] = (
        out[col]
        .astype("string")
        .fillna("")
        .apply(lambda s: [p.strip() for p in s.split(",") if p.strip()])
    )

    # explode to one row per pollutant
    out = out.explode(col, ignore_index=True)

    return out.reset_index(drop=True)

In [ ]:
uf_frame = explode_pollutants(uf_frame, col="POLUENTE")

In [ ]:
base = Path.cwd().parent  
out_dir = base / "data" 
out_dir.mkdir(parents=True, exist_ok=True)

df_mma = pd.read_csv(out_dir / "Monitoramento_QAr_BR.csv")

In [ ]:
clean_df_all_text(df_mma, in_place=True)

In [ ]:
print(len(df_mma['ID_OEMA'].unique()))
print(len(uf_frame))

In [ ]:
def replace_vals(df):
    # colunas alvo, só aplica se existirem
    cols = [c for c in ["CATEGORIA", "FUNCIONAMENTO", "STATUS"] if c in df.columns]

    # regex para strings "vazias" comuns
    null_like = r'^\s*(na|n/a|none|null|nan|nat)?\s*$'

    # 1) normaliza vazios em todas as colunas alvo
    for c in cols:
        df[c] = df[c].replace(null_like, pd.NA, regex=True).fillna("Nao declarado")

    # 2) mapeamentos específicos
    if "CATEGORIA" in df.columns:
        df["CATEGORIA"] = df["CATEGORIA"].replace({
            "N": "Nao declarado",
            "D": "Nao declarado",
            "Naodeclarado": "Nao declarado",
            "CertificadaEPA": "Referencia",
            "Certificada EPA": "Referencia",
            "Equivalente": "Referencia"
        })

    if "FUNCIONAMENTO" in df.columns:
        df["FUNCIONAMENTO"] = df["FUNCIONAMENTO"].replace({
            "N": "Nao declarado",
            "D": "Nao declarado",
            "Autmatica": "Automatica",
            "Automatico": "Automatica"
        })

    if "STATUS" in df.columns:
        df["STATUS"] = df["STATUS"].replace({
            "Sim": "Ativa",
            "sim": "Ativa",
            "Não": "Inativa",
            "Nao": "Inativa",
            "nao": "Inativa",
            "NAO": "Inativa",
        })
        # se após o mapeamento ainda sobrou algo vazio, garante "Nao declarado"
        df["STATUS"] = df["STATUS"].replace(null_like, pd.NA, regex=True).fillna("Nao declarado")

    return df

In [ ]:
df_mma = replace_vals(df_mma)
uf_frame = replace_vals(uf_frame)
dfs_dict = {
    'df_mma': df_mma,
    'uf_frame': uf_frame
}

In [ ]:
df_merged, conflicts = merge_by_id_multi(
    dfs_dict.copy(),
    id_cols=['ID_OEMA','POLUENTE'],
    source_priority=priority,
    drop_empty_strings=True,
)
df_merged

In [ ]:
def _erase(s: str) -> str:
    return "".join(c for c in unicodedata.normalize("NFKD", s) if not unicodedata.combining(c))

def _norm(s: str) -> str:
    s = str(s)
    s = re.sub(r"\(.*?\)|\[.*?\]", "", s).replace("µ", "u")
    s = _erase(s)
    s = re.sub(r"\s+", " ", s).strip().lower()
    return s

def build_cod_map(df_cod, col_sinonimos=None):
    m = {}
    for _, r in df_cod.iterrows():
        m[_norm(r["POLUENTE"])] = r["NOME_PASTA"]
        if col_sinonimos and col_sinonimos in df_cod.columns and pd.notna(r[col_sinonimos]):
            for alt in str(r[col_sinonimos]).split("|"):
                alt = alt.strip()
                if alt:
                    m[_norm(alt)] = r["NOME_PASTA"]
    return m

def normalize_pols_cell(text, cod_map, sep=r"[;,/|]+"):
    if pd.isna(text):
        return ""
    parts = re.split(sep, str(text))
    seen, out = set(), []
    for p in parts:
        k = _norm(p)
        name = cod_map.get(k)
        if name and name not in seen:
            seen.add(name); out.append(name)
    return ",".join(sorted(out))

# uso linha a linha
cod_map = build_cod_map(df_cod)  # faça uma vez
df_merged["POLUENTE"] = df_merged["POLUENTE"].apply(lambda s: normalize_pols_cell(s, cod_map))

In [ ]:
def limpar_ipynb_poluente(df, col="POLUENTE"):
    s = df[col].astype(str)

    # remove tokens contendo ".ipynb_checkpoints"
    s = s.str.replace(r'(?i)(^|,)\s*[^,]*ipynb[_-]?checkpoints[^,]*(?=,|$)', '', regex=True)
    # remove tokens contendo ".ipynb"
    s = s.str.replace(r'(?i)(^|,)\s*[^,]*\.ipynb[^,]*(?=,|$)', '', regex=True)

    # compacta vírgulas e espaços
    s = s.str.replace(r'\s*,\s*', ',', regex=True)
    s = s.str.replace(r',+', ',', regex=True).str.strip(', ')

    # dedup dos itens mantendo a ordem
    def _dedup(v):
        if not v:
            return v
        parts = [p for p in v.split(',') if p]
        seen = set()
        out = []
        for p in parts:
            if p not in seen:
                seen.add(p)
                out.append(p)
        return ",".join(out)

    df[col] = s.map(_dedup)
    return df

In [ ]:
limpar_ipynb_poluente(df_merged, col="POLUENTE")

In [ ]:
df_merged = explode_pollutants(df_merged, col="POLUENTE")

In [ ]:
# def replace_POL(df):
#     # colunas alvo, só aplica se existirem
#     cols = [c for c in ["POLUENTE"] if c in df.columns]

#     # regex para strings "vazias" comuns
#     null_like = r'^\s*(na|n/a|none|null|nan|nat)?\s*$'

#     # 2) mapeamentos específicos
#     if "POLUENTE" in df.columns:
#         df["POLUENTE"] = df["POLUENTE"].replace({
#             "PM1": "MP1",
#             "PM1O": "MP10",
#             "5": "O3",
#             "5 E O3": "O3",
#             "ETIL E ORTO": "ETIL",
#             "HCNM E ODORES": "HCNM",
#             "MP10 E MP2": "PTS",
#             "MP10": "MP10",
#             "MP2.5": "MP25",
#             "NO E NOX": "NOX",
#             "NOX E ODORES": "NOX",
#             "PTS E MP10": "PTS",
#             "NOx e Odores": "NOX",
#             "nan": "Nao declarado",
#             "HCnM e Odores": "HCNM",
#             "nan": "Nao declarado"
#         })

#     return df 

In [ ]:
def _to_ascii_upper(s: str) -> str:
    # remove acentos e normaliza subscrito/sobrescrito para dígitos
    subs = str.maketrans("₀₁₂₃₄₅₆₇₈₉", "0123456789")
    sups = str.maketrans("⁰¹²³⁴⁵⁶⁷⁸⁹", "0123456789")
    s = unicodedata.normalize("NFKD", str(s))
    s = "".join(c for c in s if not unicodedata.combining(c))
    s = s.translate(subs).translate(sups).upper().strip()
    s = re.sub(r"\s+", " ", s)
    return s

def _map_token(t: str) -> list[str]:
    """Mapeia um token para 0..n nomes padronizados."""
    t0 = _to_ascii_upper(t)

    # descartes
    if t0 in {"", "NAN", "NA", "NONE", "NULL", "ODOR", "ODORES"}:
        return []

    # PM/MP normalização
    t1 = t0.replace("PM", "MP")  # PM10 -> MP10, PM2.5 -> MP2.5
    t1 = t1.replace("MP1O", "MP10")  # O no lugar de zero

    # MP2.5 variações -> MP25
    if re.fullmatch(r"MP2([.,/]?5)?|MP25|PM2([.,/]?5)?", t0):
        return ["MP25"]

    if t1 in {"MP10"}:
        return ["MP10"]
    if t1 in {"MP1"}:
        return ["MP1"]

    # abreviações comuns
    direct = {
        "NOX": "NOX",
        "NO": "NO",
        "NO2": "NO2",
        "O3": "O3",
        "SO2": "SO2",
        "CO": "CO",
        "CH4": "CH4",
        "PTS": "PTS",
        "VOC": "VOC",
        "HCT": "HCT",
        "HCNM": "HCNM",
        "ERT": "ERT",
        "FMC": "FMC",
        "H2S": "H2S",
        "BENZENO": "BENZENO",
        "TOLUENO": "TOLUENO",
        "ETILBENZENO": "ETILBENZENO",
        "XILENO": "XILENO",
        "OXILENO": "OXILENO",
        "MPXILENO": "MPXILENO",
        "ACETAL": "ACETAL",
        "FORMAL": "FORMAL",
        "CH2O": "FORMAL",   # formaldeído
        "BEN": "BENZENO",
        "BENZ": "BENZENO",
        "TOL": "TOLUENO",
        "ETBEN": "ETILBENZENO",
        "ETILBEN": "ETILBENZENO",
        "MPXIL": "MPXILENO",
        "OXIL": "OXILENO",
        "XIL": "XILENO",
    }
    if t1 in direct:
        return [direct[t1]]

    # caso especial “ETIL,ORTO” já será quebrado pelo split; mas se vier colado:
    if t1 in {"ETIL,ORTO", "ETIL ORTO"}:
        return ["ETILBENZENO", "OXILENO"]

    # fallback: mantemos o token bruto normalizado
    return [t1]

def normalizar_coluna_poluente(df: pd.DataFrame, df_cod: pd.DataFrame, col="POLUENTE") -> pd.DataFrame:
    if col not in df.columns:
        return df.copy()

    # vocabulário permitido
    allowed = set(df_cod["NOME_PASTA"].astype(str).str.upper().str.strip())

    # separadores: vírgula, ;, /, |, e “ e ”
    sep_re = re.compile(r"\s*(?:,|;|/|\||\se\s)\s*", flags=re.I)

    def process_cell(val) -> str:
        if pd.isna(val):
            return ""
        tokens = []
        seen = set()
        # quebra
        parts = [p for p in sep_re.split(str(val)) if p.strip()]
        for p in parts:
            mapped = _map_token(p)
            for m in mapped:
                if m in allowed and m not in seen:
                    seen.add(m)
                    tokens.append(m)
        return ",".join(tokens)

    out = df.copy()
    out[col] = out[col].apply(process_cell)
    return out

def listar_tokens_restantes(df: pd.DataFrame, col="POLUENTE") -> list[str]:
    parts = df[col].astype(str).str.split(",").explode().dropna().str.strip()
    return sorted([t for t in parts.unique() if t])

# -------------------- uso --------------------
# df_cod precisa ter a coluna NOME_PASTA com os nomes oficiais
# df_merged é o seu DF com a coluna POLUENTE
df_merged = normalizar_coluna_poluente(df_merged, df_cod, col="POLUENTE")
print(listar_tokens_restantes(df_merged, "POLUENTE"))

In [ ]:
base = Path.cwd().parent  # .../RQAR_2025_book
out_dir = base / "data" 
out_dir.mkdir(parents=True, exist_ok=True)

out_file = out_dir / "Monitoramento_QAr_BR.csv"
df_merged.to_csv(out_file, index=False, encoding="utf-8")
print("Saved to:", out_file.resolve())

##### Ler e abrir arquivos CSV

In [116]:
def _read_csv(path, sep=None, decimal=None, encoding=None, **kw):
    head = path.read_bytes()[:4096]
    txt  = head.decode(encoding or "utf-8", errors="ignore")
    sep_guess = sep or (";" if txt.count(";") > txt.count(",") else ",")
    dec_guess = decimal or ("," if re.search(r"\d+,\d+", txt) and txt.count(",") > txt.count(".") else ".")
    enc_guess = encoding or ("utf-8-sig" if txt.startswith("\ufeff")
                             else ("latin-1" if ("Ã" in txt or "�" in txt) else "utf-8"))
    try:
        return pd.read_csv(path, sep=sep_guess, decimal=dec_guess, encoding=enc_guess, engine="c", **kw)
    except Exception:
        return pd.read_csv(path, sep=None, engine="python", decimal=dec_guess, encoding=enc_guess, **kw)

def load_csvs(dir_path, prefix=None, recursive=False, limit=None, **read_csv_kwargs):
    pattern = "**/*.csv" if recursive else "*.csv"
    files = sorted(dir_path.glob(pattern))
    if prefix:
        p = prefix.upper()
        files = [f for f in files if f.name.upper().startswith(p)]
    if limit:
        files = files[:int(limit)]
    dfs = {}
    for f in files:
        df = _read_csv(f, **read_csv_kwargs)
        key = f.stem
        # avoid key clashes
        if key in dfs:
            key = f"{f.stem}__{len(dfs)}"
        dfs[key] = df
        print(f"Loaded {f.name}: {df.shape}")
    return dfs

##### Ler e abrir arquivos TXT

In [117]:
def _read_txt(path, sep=None, decimal=None, encoding=None, header=None, **kw):
    head = path.read_bytes()[:4096]
    txt  = head.decode(encoding or "utf-8", errors="ignore")

    enc_guess = encoding or ("utf-8-sig" if txt.startswith("\ufeff")
                             else ("latin-1" if ("Ã" in txt or "�" in txt) else "utf-8"))
    dec_guess = decimal or ("," if re.search(r"\d+,\d+", txt) and txt.count(",") > txt.count(".") else ".")

    if sep is not None:
        sep_guess = sep
    else:
        candidates = ["\t", ";", "|", ","]
        counts = {d: txt.count(d) for d in candidates}
        sep_guess = max(counts, key=counts.get) if max(counts.values()) >= 2 else None

    try:
        if sep_guess is None:
            return pd.read_csv(path, sep=None, engine="python",
                               decimal=dec_guess, encoding=enc_guess,
                               header=header, **kw)
        else:
            return pd.read_csv(path, sep=sep_guess, engine="c",
                               decimal=dec_guess, encoding=enc_guess,
                               header=header, **kw)
    except Exception:
        full = path.read_text(encoding=enc_guess, errors="ignore")
        return pd.DataFrame({"text": full.splitlines()})

def load_txts(dir_path, prefix=None, recursive=False, limit=None, **read_txt_kwargs):
    pattern = "**/*.txt" if recursive else "*.txt"
    files = sorted(dir_path.glob(pattern))
    if prefix:
        p = prefix.upper()
        files = [f for f in files if f.name.upper().startswith(p)]
    if limit:
        files = files[:int(limit)]
    dfs = {}
    for f in files:
        df = _read_txt(f, **read_txt_kwargs)
        key = f.stem
        if key in dfs:
            key = f"{f.stem}__{len(dfs)}"
        dfs[key] = df
        print(f"Loaded {f.name}: {df.shape}")
    return dfs

##### Ler e abrir arquivos XLSX

In [333]:
EXCEL_EXTS = {".xlsx", ".xls", ".xlsm", ".xltx", ".xltm"}

def _excel_engine(ext: str) -> str | None:
    """
    Retorna o engine recomendado para cada extensão.
    .xlsx, .xlsm, .xltx, .xltm -> openpyxl
    .xls -> xlrd (necessita instalar xlrd)
    """
    ext = ext.lower()
    if ext in {".xlsx", ".xlsm", ".xltx", ".xltm"}:
        return "openpyxl"
    if ext == ".xls":
        return "xlrd"  
    return None

In [334]:
def load_excels(dir_path, sheets=0, prefix=None, recursive=False, limit=None, **read_excel_kwargs):
    """
    Lê arquivos Excel de uma pasta e retorna um dicionário {chave: DataFrame}.

    Parâmetros:
      - dir_path: str | Path
        Caminho da pasta onde estão os arquivos.
      - sheets: int | str | list | "all"
        Qual planilha carregar.
        0 carrega a primeira planilha. Pode ser índice (int) ou nome (str).
        Lista para múltiplas planilhas. "all" para todas as planilhas.
      - prefix: str | None
        Se definido, carrega apenas arquivos cujo nome começa com este prefixo.
      - recursive: bool
        Se True, busca nas subpastas.
      - limit: int | None
        Limita a quantidade de arquivos lidos.
      - **read_excel_kwargs:
        Parâmetros extras repassados na função

    Retorno:
      - Dicionário com chave nome base do arquivo
    """
    dir_path = Path(dir_path)
    files = []
    if recursive:
        for ext in EXCEL_EXTS:
            files += list(dir_path.rglob(f"*{ext}"))
    else:
        for ext in EXCEL_EXTS:
            files += list(dir_path.glob(f"*{ext}"))
    files = sorted(files)

    if prefix:
        p = prefix.upper()
        files = [f for f in files if f.name.upper().startswith(p)]
    if limit:
        files = files[:int(limit)]

    dfs = {}
    for f in files:
        eng = _excel_engine(f.suffix)
        try:
            with pd.ExcelFile(f, engine=eng) as xls:
                if sheets == "all":
                    wanted = xls.sheet_names
                elif isinstance(sheets, (list, tuple)):
                    wanted = sheets
                else:
                    wanted = [sheets]

                for s in wanted:
                    df = pd.read_excel(xls, sheet_name=s, **read_excel_kwargs)
                    if sheets == "all" or isinstance(sheets, (list, tuple)):
                        sheet_label = s if isinstance(s, str) else xls.sheet_names[s]
                        key = f"{f.stem}__{sheet_label}"
                    else:
                        key = f.stem
                    if key in dfs:
                        i = 1
                        new_key = f"{key}__{i}"
                        while new_key in dfs:
                            i += 1
                            new_key = f"{key}__{i}"
                        key = new_key

                    dfs[key] = df
                    print(f"Loaded {f.name} [sheet {s}]: {df.shape}")
        except Exception as e:
            print(f"Skip {f.name} due to read error: {e}")
            continue
    return dfs

##### Remove e substitui caracteres específicos 

In [120]:
CLEAN_TABLE = {
    ord("ç"): "c",
    ord("Ç"): "C",
    ord("´"): None,
    ord("~"): None,
    ord("ˆ"): None,
    ord("°"): None,
    ord("`"): None,
    0x0302: None,  # combining ^
}

In [121]:
def clean_text_full(s: object) -> object:
    if pd.isna(s):
        return s
    t = str(s)
    # sua tabela primeiro
    t = t.translate(CLEAN_TABLE)
    # remover acentos de letras precompostas (ex.: ã, á, í)
    t = unicodedata.normalize("NFKD", t)
    t = "".join(ch for ch in t if unicodedata.category(ch) != "Mn")
    return t

def clean_df_all_text(df: pd.DataFrame, cols=None, in_place=False) -> pd.DataFrame:
    out = df if in_place else df.copy()
    if cols is None:
        cols = [c for c in out.columns
                if is_object_dtype(out[c].dtype) or is_string_dtype(out[c].dtype)
                   or isinstance(out[c].dtype, pd.CategoricalDtype)]
    for c in cols:
        s = out[c]
        was_cat = isinstance(s.dtype, pd.CategoricalDtype)
        s2 = s.astype("string").map(lambda v: clean_text_full(v) if pd.notna(v) else v)
        out[c] = s2.astype("category") if was_cat else s2
    return out

##### Transformar coluna Datetime

In [122]:
def convert_column_to_datetime(df, column_name, format=None):
    out = df.copy()

    if (out.index.name or "").upper() == column_name.upper():
        if column_name in out.columns:
            out.reset_index(drop=True, inplace=True)
        else:
            out.reset_index(inplace=True)

    if column_name not in out.columns:
        if pd.api.types.is_datetime64_any_dtype(out.index):
            out[column_name] = out.index
            if not keep_index:
                out.reset_index(drop=True, inplace=True)
        else:
            raise ValueError(f"Column or datetime index '{column_name}' not found.")

    out[column_name] = pd.to_datetime(out[column_name], format=format,  errors="coerce")

    # if out[column_name].notna().any():
    #     out["INICIO"] = int(out[column_name].min().year)
    #     out["FIM"] = int(out[column_name].max().year)
    # else:
    #     out["INICIO"] = None
    #     out["FIM"] = None

    return out

##### Inspecionar colunas de mesmo nome dentro de um dicionário

In [123]:
def _flatten_columns(df):
    """
    Achata colunas MultiIndex em strings simples.
    Exemplo: ("A","B") vira "A | B".
    """
    cols = df.columns
    if isinstance(cols, pd.MultiIndex):
        return [" | ".join([str(x) for x in tup if pd.notna(x)]) for tup in cols]
    return [str(c) for c in cols]

In [124]:
def _normalize(names, lower=True, strip=True):
    """
    Normaliza nomes de colunas para comparação justa.
    - lower: converte para minúsculas
    - strip: remove espaços extras
    """
    out = []
    for n in names:
        s = str(n)
        if strip:
            s = s.strip()
        if lower:
            s = s.lower()
        out.append(s)
    return out

In [125]:
def summarize_columns(dfs: dict, normalize=True):
    """
    Resume colunas presentes em um dicionário {nome_df: DataFrame}.

    Parâmetros
    - dfs: dict[str, pd.DataFrame]
      Dicionário onde a chave é o nome e o valor é um DataFrame.
    - normalize: bool
      Se True, compara usando nomes normalizados (minúsculas e trim).

    Retorno
    - summary_df: pd.DataFrame
      Tabela com uma linha por coluna distinta:
        col            nome da coluna considerada na comparação
        n_present      em quantos DataFrames ela aparece
        n_missing      em quantos não aparece
        is_common      True se aparece em todos
        is_unique      True se aparece em apenas um
        present_in     lista de DataFrames onde aparece
        missing_in     lista de DataFrames onde não aparece
        examples       exemplos de rótulos originais vistos para esta coluna
    - per_df_stats: pd.DataFrame
      Uma linha por DataFrame com:
        n_cols                 quantidade de colunas
        n_common_present       quantas colunas comuns ele possui
        n_common_missing       quantas colunas comuns faltam nele
        extras_count           colunas que só existem em alguns e estão nele
        extras                 lista dessas colunas extras
    """
    # 1) coletar nomes por df
    name_to_cols = {}
    name_to_rawmap = {}
    for name, df in dfs.items():
        cols_raw = _flatten_columns(df)
        cols_cmp = _normalize(cols_raw) if normalize else cols_raw
        name_to_cols[name] = set(cols_cmp)
        # mapeia versão normalizada para exemplos originais
        rawmap = {}
        for raw, cmp in zip(cols_raw, cols_cmp):
            rawmap.setdefault(cmp, set()).add(raw)
        name_to_rawmap[name] = rawmap

    all_cols = set().union(*name_to_cols.values()) if name_to_cols else set()
    df_names = list(name_to_cols.keys())
    n_dfs = len(df_names)

    # 2) frequências por coluna
    records = []
    for col in sorted(all_cols):
        present_in = [n for n in df_names if col in name_to_cols[n]]
        missing_in = [n for n in df_names if col not in name_to_cols[n]]
        # exemplos de rótulos originais
        examples = sorted(set().union(*[name_to_rawmap[n].get(col, set()) for n in df_names]))
        records.append({
            "col": col,
            "n_present": len(present_in),
            "n_missing": n_dfs - len(present_in),
            "is_common": len(present_in) == n_dfs,
            "is_unique": len(present_in) == 1,
            "present_in": present_in,
            "missing_in": missing_in,
            "examples": examples[:5],  # mostra até 5 exemplos
        })
    summary_df = pd.DataFrame(records).sort_values(
        ["is_common", "n_present", "col"], ascending=[False, False, True]
    ).reset_index(drop=True)

    # 3) colunas comuns e extras
    common_set = set(summary_df.loc[summary_df["is_common"], "col"])
    per_df_rows = []
    for name in df_names:
        cols_set = name_to_cols[name]
        n_cols = len(cols_set)
        n_common_present = len(cols_set & common_set)
        n_common_missing = len(common_set - cols_set)
        extras = sorted(c for c in cols_set if c not in common_set)
        per_df_rows.append({
            "df": name,
            "n_cols": n_cols,
            "n_common_present": n_common_present,
            "n_common_missing": n_common_missing,
            "extras_count": len(extras),
            "extras": extras
        })
    per_df_stats = pd.DataFrame(per_df_rows).sort_values("df").reset_index(drop=True)

    return summary_df, per_df_stats

##### Renomear colunas - substituir campos conforme desejado

In [126]:
# manual_map = {"fecha":  "DATETIME",
#              "Date":  "DATETIME",
#              "Fecha":  "DATETIME",
#}

##### Selecionar estações únicas no conjunto de dados

Nota: Quando há junção de mais de uma base de dados de fontes diferentes, utilizar função para manter estações não repetidas e garantir que a maioria das colunas com dados seja mantida.

In [386]:
56+14+4+9+19+6+67+4+5+3+4+28+100+2+19+4+101
445-422

23

In [127]:
def merge_by_id_multi(
    dfs_dict: dict,
    id_cols,              # pode ser "ID_OEMA" ou ["ID_OEMA","UF"]
    source_col="__source",
    source_priority: list[str] | None = None,
    drop_empty_strings: bool = True,
):
    """
    Une vários DataFrames (em um dict) e mantém 1 linha por chave composta.

    Parâmetros:
      - dfs_dict: dict[str, pd.DataFrame]  dicionário nome->DataFrame.
      - id_cols: str | list/tuple          coluna(s) que formam a chave. Ex.: "ID_OEMA" ou ["ID_OEMA","UF"].
      - source_col: str                    coluna auxiliar com o nome da fonte.
      - source_priority: list[str] | None  prioridade entre fontes para resolver conflitos.
      - drop_empty_strings: bool           converte strings vazias em NaN antes de mesclar.

    Retorna:
      - df_merged: DataFrame final com 1 linha por chave.
      - df_conflicts: DataFrame listando conflitos por coluna e chave.
    """
    if isinstance(id_cols, str):
        id_cols = (id_cols,)
    id_set = set(id_cols)

    # 1) concatena e marca fonte
    frames = []
    for name, df in dfs_dict.items():
        if isinstance(df, pd.DataFrame):
            tmp = df.copy()
            tmp[source_col] = name
            frames.append(tmp)
    if not frames:
        return pd.DataFrame(), pd.DataFrame()
    big = pd.concat(frames, ignore_index=True, sort=False)

    # garante que todas as colunas de chave existam
    for c in id_cols:
        if c not in big.columns:
            big[c] = pd.NA

    # 2) normalização leve
    if drop_empty_strings:
        for c in big.columns:
            if big[c].dtype.kind in "OUS":
                big[c] = big[c].astype("string").str.strip().replace({"": pd.NA})

    def pick_value(series, sources):
        mask = series.notna()
        vals = series[mask].astype(object).tolist()
        srcs = sources[mask].astype(str).tolist()
        if not vals:
            return pd.NA, None, []
        if len(set(map(str, vals))) == 1:
            return vals[0], srcs[0], list(dict.fromkeys(srcs))
        if source_priority:
            pri = {s: i for i, s in enumerate(source_priority)}
            best = min(range(len(srcs)), key=lambda i: pri.get(srcs[i], 10**9))
            return vals[best], srcs[best], list(dict.fromkeys(srcs))
        return vals[0], srcs[0], list(dict.fromkeys(srcs))

    # 3) reduz por chave composta
    data_cols = [c for c in big.columns if c not in id_set | {source_col}]
    rows, conflicts = [], []
    for key_vals, g in big.groupby(list(id_cols), dropna=True, sort=False):
        if len(id_cols) == 1:
            key_vals = (key_vals,)
        out = {c: v for c, v in zip(id_cols, key_vals)}

        for c in data_cols:
            s = g[c] if c in g.columns else pd.Series([pd.NA] * len(g))
            winner, winner_src, all_srcs = pick_value(s, g[source_col])
            out[c] = winner

            non_null = [str(v) for v in s.dropna().tolist()]
            if len(set(non_null)) > 1:
                confl = {col: val for col, val in zip(id_cols, key_vals)}
                confl.update({
                    "col": c,
                    "values": sorted(set(non_null)),
                    "sources": all_srcs,
                    "chosen": str(winner),
                    "chosen_source": winner_src,
                })
                conflicts.append(confl)

        rows.append(out)

    df_merged = pd.DataFrame(rows)
    ordered = list(id_cols) + [c for c in df_merged.columns if c not in id_set]
    df_merged = df_merged[ordered]

    df_conflicts = pd.DataFrame(conflicts)
    return df_merged, df_conflicts

##### Unir estações quando dentro de um dict - Múltiplas planilhas

Nota: Maneira mais simplificada de unir diferentes dfs dentro de um dict, quando sabemos que não existem linhas duplicadas ou informações repetidas. Quando há dúvida, rodar a função anterior.

In [128]:
def merge_station_dfs(dfs_dict, uf):
    frames = []
    for station_name, d in dfs_dict.items():
        df = d.copy()

        # station
        df["ESTACAO"] = station_name # LEMBRAR DE DAR UM DROP
        frames.append(df)

    if not frames:
        raise ValueError("No data frames to merge.")
    return pd.concat(frames, ignore_index=True, sort=False)

##### Corrigir e identificar problemas com Datetime column

In [129]:
TARGET_FMT = "%d/%m/%Y %H:%M"

def fix_datetime_df(df, col, keep_dt_col=True):
    out = df.copy()
    out.rename(columns=lambda c: str(c).strip().lstrip("\ufeff"), inplace=True)

    s = (out[col].astype(str)
                  .str.strip()
                  .str.replace("\xa0", " ", regex=False)   # nbsp
                  .str.replace("T", " ", regex=False)
                  .str.replace("Z", "", regex=False))

    dt = pd.Series(pd.NaT, index=out.index, dtype="datetime64[ns]")

    # put your actual input formats first
    tries = [
        ("%H:%M %m/%d/%Y", False),       # 02:00 1/1/2019
        ("%H:%M:%S %m/%d/%Y", False),    # 02:00:00 1/1/2019
        ("%d/%m/%Y %H:%M", True),
        ("%d/%m/%Y %H:%M:%S", True),
        ("%Y-%m-%d %H:%M", False),
        ("%Y-%m-%d %H:%M:%S", False),
        ("%d-%b-%Y %H:%M", True),
        ("%d-%b-%Y %H:%M:%S", True),
    ]

    for fmt, dayfirst in tries:
        m = dt.isna()
        if not m.any():
            break
        dt.loc[m] = pd.to_datetime(s[m], format=fmt, errors="coerce", dayfirst=dayfirst)

    # final fallback
    m = dt.isna()
    if m.any():
        try:
            dt.loc[m] = pd.to_datetime(s[m], format="mixed", dayfirst=True, errors="coerce")
        except TypeError:
            dt.loc[m] = pd.to_datetime(s[m], dayfirst=True, errors="coerce")

    if keep_dt_col:
        out[f"{col}_DT"] = dt

    out[col] = np.where(dt.notna(), dt.dt.strftime(TARGET_FMT), np.nan)
    return out

In [130]:
# Diagnóstico das colunas ou linhas com problema
def diag_bad_stations(df_all, col="DATETIME", top=5):
    out = []
    for st, g in df_all.groupby("ESTACAO"):
        rate = g[col].notna().mean()
        if rate < 0.99:
            bad = g.loc[g[col].isna(), col].astype(str).head(top).tolist()
            out.append((st, rate, bad))
    for st, rate, examples in out:
        print(f"{st}: parsed {rate:.1%}  examples not parsed -> {examples}")

##### Criar início e fim das medições por estação

In [131]:
# def station_year_bounds(g, datetime_col: str | None):
#     if not datetime_col or datetime_col not in g.columns:
#         return None, None
#     ts = g[datetime_col].dropna()
#     ini = int(ts.min().year) if not ts.empty else None
#     fim = int(ts.max().year) if not ts.empty else None
#     return ini, fim

def station_year_bounds(g: pd.DataFrame, datetime_col: Optional[str]):
    if not datetime_col or datetime_col not in g.columns:
        return pd.NA, pd.NA
    s = pd.to_datetime(g[datetime_col], dayfirst=True)
    if s.notna().any():
        return int(s.min().year), int(s.max().year)
    return pd.NA, pd.NA

##### Criar coluna de poluentes mapeando os nomes

In [132]:
# Função para mapear e converter apenas os nomes dos poluentes quando no CABEÇALHO DAS COLUNAS
def id_pol(df, df_cod, column_name, drop_after_underscore=True):
    nomes = []
    for col in df.columns:
        if col == column_name:
            continue  
        
        # limpa parênteses -> "CO(µg/m³)" -> "CO"
        new_col = re.sub(r"\(.*?\)", "", col).strip()
        # keep only left side before first underscore if you want
        if drop_after_underscore and "_" in new_col:
            new_col = new_col.split("_", 1)[0]

        # verifica no dicionário
        linha = df_cod[df_cod["POLUENTE"].str.strip() == new_col]
        if not linha.empty:
            nomes.append(linha["NOME_PASTA"].values[0])
    
    return ",".join(sorted(set(nomes)))  # retorna só os nomes únicos

In [133]:
# Função para mapear e converter apenas os nomes dos poluentes quando NAS LINHAS DOS DFS
def id_pol_from_rows(
    df,
    df_cod,
    cols_cand=("POLUENTE","POLUENTES","Poluente","pollutant","nome_poluente","NomePoluente"),
    find_all=True,
    col_sinonimos=None,
    sep=r"[;,/|]+",
):
    """
    Identifica poluentes quando os nomes estão nas LINHAS e retorna nomes únicos
    padronizados conforme df_cod['NOME_PASTA'], separados por vírgula.

    Parâmetros:
      - df: DataFrame de entrada com as respostas.
      - df_cod: DataFrame de dicionário com colunas obrigatórias:
          'POLUENTE'  nome canônico
          'NOME_PASTA' nome padronizado desejado
        Opcional:
          col_sinonimos  nome de coluna em df_cod com sinônimos separados por '|'
      - cols_cand: possíveis nomes de coluna em df que contêm o texto dos poluentes.
      - find_all: se True, quando não encontrar cols_cand, varre todas as colunas de texto.
      - sep: regex para dividir listas no texto, ex "CO, NO2; PM10".

    Retorno:
      - str com nomes únicos padronizados, separados por vírgula.
    """

    # Normalização simples de rótulos para comparação
    def _erase(s: str) -> str:
        return "".join(c for c in unicodedata.normalize("NFKD", s) if not unicodedata.combining(c))

    def _norm(s: str) -> str:
        s = str(s)
        s = re.sub(r"\(.*?\)|\[.*?\]", "", s)  # remove unidades entre () ou []
        s = s.replace("µ", "u")
        s = _erase(s)
        s = re.sub(r"\s+", " ", s).strip().lower()
        return s

    # Mapa normalizado -> NOME_PASTA, incluindo sinônimos se houver
    cod_map = {}
    for _, row in df_cod.iterrows():
        cod_map[_norm(row["POLUENTE"])] = row["NOME_PASTA"]
        if col_sinonimos and col_sinonimos in df_cod.columns and pd.notna(row[col_sinonimos]):
            for alt in str(row[col_sinonimos]).split("|"):
                alt = alt.strip()
                if alt:
                    cod_map[_norm(alt)] = row["NOME_PASTA"]

    # Coleta de text candidatos
    text = []
    col_pol = next((c for c in df.columns if str(c).strip() in cols_cand), None)
    if col_pol is not None:
        text = df[col_pol].dropna().astype(str).tolist()
    elif find_all:
        for c in df.columns:
            if df[c].dtype.kind in "OUS":
                text += df[c].dropna().astype(str).tolist()

    if not text:
        return ""

    # Quebra por sep e mapeia para nomes padronizados
    seen = set()
    out = []
    sep_re = re.compile(sep)
    for t in text:
        partes = [p.strip() for p in sep_re.split(t)] if sep_re.search(t) else [t.strip()]
        for p in partes:
            if not p:
                continue
            k = _norm(p)
            nome = cod_map.get(k)
            if nome and nome not in seen:
                seen.add(nome)
                out.append(nome)

    return ",".join(sorted(out))

##### Criar planilha final com os campos desejados para DADOS ESTAÇÕES, quando df final está incompleto

In [134]:
def build_inventory_rows(df_all,
                         df_cod,
                         uf,
                         datetime_col: str | None=None,
                         drop_after_underscore=True,
                         mode: str = "rows"):
    """
    Monta linhas do inventário por estação.

    Parâmetros:
      - df_all: DataFrame já unido, com colunas 'ESTACAO' e opcionalmente a coluna de data-hora.
      - df_cod: dicionário de poluentes com colunas 'POLUENTE' e 'NOME_PASTA' (e opcional 'SINONIMOS').
      - uf: sigla UF, ex. 'RJ'.
      - datetime_col: nome da coluna de data-hora ou None (quando não há).
      - mode:
          'columns' -> detectar poluentes no CABEÇALHO (usa sua função id_pol(...))
          'rows'    -> detectar poluentes nas LINHAS (usa id_pol_from_rows(...))

    Retorno:
      - DataFrame com uma linha por estação, incluindo POLUENTE, INICIO e FIM.
    """ 
    rows = []
    for station, g in df_all.groupby("ESTACAO", sort=True):
        inicio, fim = station_year_bounds(g, datetime_col)
            
        if mode == "columns":
            # usa função que lê do cabeçalho; passar um nome para ignorar
            col_to_ignore = datetime_col 
            pols = id_pol(g, df_cod, column_name=col_to_ignore, drop_after_underscore=True)
        else:
            # lê poluentes a partir das LINHAS; NÃO passe datetime_col aqui
            pols = id_pol_from_rows(g, df_cod)
            
        rows.append({
            "UF": uf,
            "ID_OEMA": station,
            "CIDADE": "",
            "ID_MMA": "",  # será preenchido depois
            "ID_MMA_COMPLETO": "",
            "POLUENTE": pols,
            "COD_POLUENTE": "",  # deixamos vazio
            "CD_MUN": "",
            "COD_UF_IBGE": sigla_to_ibge(uf),
            "PROPRIETARIO": "",
            "PROP_ENTIDADE": "",
            "OPERADOR": "",
            "OP_ENTIDADE": "",
            "LATITUDE": "",
            "LONGITUDE": "",
            "MOBILIDADE": "",
            "CATEGORIA": "",
            "FUNCIONAMENTO": "",
            "METODO": "",
            "MARCA": "",
            "FINALIDADE": "",
            "MONITORAR": "",
            "FONTE": "",
            "CALIBRACAO": "",
            "REALOCACAO": "",
            "OBS_CALIBRACAO": "",
            "INICIO": inicio,
            "FIM": fim,
            "DADOS_MONITORAMENTO": "",
            "RECONHECIDA": "",
            "OBS_GERAIS": "",
            "CERTIFICACAO": "",
            "STATUS": "",
            "REP_ESPACIAL_DECLARADA": ""
        })

    return pd.DataFrame(rows)

##### Criar planilha final com os campos desejados para DADOS ESTAÇÕES, quando df final contém colunas desejadas

In [135]:
def pick_group_value(s: pd.Series, strategy="first_non_null", sep=" | "):
    """
    Escolhe um valor representativo dentro de um grupo.
    - s: Série com valores do grupo
    - strategy: "first_non_null" | "mode" | "concat_unique"
    - sep: separador para concatenação
    """
    ss = s.dropna()
    if ss.empty:
        return None
    if strategy == "mode":
        m = ss.mode()
        return m.iloc[0] if not m.empty else ss.iloc[0]
    if strategy == "concat_unique":
        vals = pd.unique(ss.astype(str).str.strip())
        return sep.join([v for v in vals if v])
    return ss.iloc[0]  # padrão

In [136]:
def build_inventory_flexible(
    df_all: pd.DataFrame,
    uf: str,
    df_cod,                         # usado pelas funções de poluentes
    group_col: str = "ID_OEMA",
    datetime_col: Optional[str] = None,
    mode: str = "rows",
    field_map: Optional[dict[str, Any]] = None,
    drop_after_underscore=True
) -> pd.DataFrame:
    """
    PT-BR:
      - df_all: DataFrame com ao menos group_col.
      - uf: sigla da UF.
      - df_cod: dicionário de poluentes.
      - group_col: chave do agrupamento.
      - datetime_col: coluna de data-hora para INICIO/FIM (opcional).
      - mode: "rows" usa id_pol_from_rows, "columns" usa id_pol.
      - field_map: regras extras, ex. {"CIDADE": ("col","CIDADE",{"strategy":"mode"})}
    """
    if group_col not in df_all.columns:
        raise ValueError(f"Coluna de agrupamento '{group_col}' não existe.")

    # funções de poluentes por modo
    if mode == "columns":
        pol_func = lambda g: id_pol(g, df_cod, column_name=datetime_col, drop_after_underscore=True)
    else:
        pol_func = lambda g: id_pol_from_rows(g, df_cod)

    # regras base: valores por grupo usando ("func", ...)
    base_rules = {
        "UF": uf,
        "ID_OEMA": ("group_key",),
        "POLUENTE": ("func", pol_func),
        "COD_UF_IBGE": sigla_to_ibge(uf),
        "INICIO": ("func", lambda g: station_year_bounds(g, datetime_col)[0]) if datetime_col else "",
        "FIM":    ("func", lambda g: station_year_bounds(g, datetime_col)[1]) if datetime_col else "",
        "CIDADE": "",
        "ID_MMA": "",
        "ID_MMA_COMPLETO": "",
        "COD_POLUENTE": "",
        "CD_MUN": "",
        "PROPRIETARIO": "",
        "PROP_ENTIDADE": "",
        "OPERADOR": "",
        "OP_ENTIDADE": "",
        "LATITUDE": "",
        "LONGITUDE": "",
        "MOBILIDADE": "",
        "CATEGORIA": "",
        "FUNCIONAMENTO": "",
        "METODO": "",
        "MARCA": "",
        "FINALIDADE": "",
        "MONITORAR": "",
        "FONTE": "",
        "CALIBRACAO": "",
        "REALOCACAO": "",
        "OBS_CALIBRACAO": "",
        "DADOS_MONITORAMENTO": "",
        "RECONHECIDA": "",
        "OBS_GERAIS": "",
        "CERTIFICACAO": "",
        "STATUS": "",
        "REP_ESPACIAL_DECLARADA": "",
    }

    rules = {**base_rules, **(field_map or {})}

    rows = []
    for group_key, g in df_all.groupby(group_col, sort=True):
        out = {}
        for field, rule in rules.items():
            # literal
            if not isinstance(rule, tuple):
                out[field] = rule
                continue

            kind = rule[0]
            if kind == "group_key":
                out[field] = group_key
            elif kind == "col":
                colname = rule[1]
                opts = rule[2] if len(rule) > 2 and isinstance(rule[2], dict) else {}
                out[field] = pick_group_value(g[colname], **opts) if colname in g.columns else None
            elif kind == "func":
                func = rule[1] if len(rule) > 1 else None
                out[field] = func(g) if callable(func) else None
            else:
                out[field] = None

        rows.append(out)

    return pd.DataFrame(rows)

##### Criar ID_MMA e ID_MMA_COMPLETO com base nas datas de inicio e fim das operações

NOTA: Sempre antes de criar ID_MMA COMPLETO na planilha final dos DADOS ESTAÇÕES, precisa garantir que as estações estejam com os campos início e fim preenchidos, para criar a sequência dos códigos - que depende do período de funcionamento

In [156]:
# def assign_id_mma(df, uf_col="UF"):
#     out = df.sort_values([uf_col, "INICIO", "ID_OEMA"], na_position="last").reset_index(drop=True)
#     seq = out.groupby(uf_col).cumcount().add(1).astype(str).str.zfill(4)
#     out["ID_MMA"] = out[uf_col] + seq
#     out["ID_MMA_COMPLETO"] = out["ID_MMA"]
#     return out

def _ascii_lower(s: str) -> str:
    s = str(s)
    s = ud.normalize("NFKD", s)
    s = "".join(ch for ch in s if not ud.combining(ch))
    return s.lower()

def assign_id_mma(df, uf_col="UF", date_col="INICIO"):
    out = df.copy()
    name_key = out["ID_OEMA"].astype(str).map(_ascii_lower)

    out = (
        out.assign(_name_key=name_key)
           .sort_values([uf_col, date_col, "_name_key"], kind="mergesort", na_position="last")
           .drop(columns="_name_key")
           .reset_index(drop=True)
    )

    seq = out.groupby(uf_col).cumcount().add(1).astype(str).str.zfill(4)
    out["ID_MMA"] = out[uf_col] + seq
    out["ID_MMA_COMPLETO"] = out["ID_MMA"]
    return out

##### Conferir após criação da planilha principal, se não permaneceram linhas duplicadas 

In [138]:
# Linhas duplicadas por chave (ex: "ID_OEMA")
def get_duplicate_rows(df: pd.DataFrame, key: str = "ID_OEMA") -> pd.DataFrame:
    '''  - df: DataFrame final
         - key: nome da coluna que quer usar como filtro'''
    
    s = df[key].astype("string").str.strip()
    mask = s.duplicated(keep=False)
    return df.loc[mask].sort_values(key)

# Linhas que correspondem a um valor específico da chave
# - value: valor a filtrar (ex.: "RJ_est_0123")
# def get_rows_for_id(df: pd.DataFrame, value, key: str = "ID_OEMA") -> pd.DataFrame:
#     s = df[key].astype("string").str.strip()
#     return df.loc[s.eq(str(value).strip())]

# Relatório por chave com número de linhas e colunas diferentes
def get_duplicates_report(df: pd.DataFrame, key: str = "ID_OEMA") -> pd.DataFrame:
    s = df[key].astype("string").str.strip()
    dups = df.loc[s.duplicated(keep=False)]
    rows = []
    for k, g in dups.groupby(key):
        nun = g.nunique(dropna=False)
        diff_cols = nun[nun > 1].index.tolist()
        rows.append({"ID": k, "n_rows": len(g), "diff_cols": diff_cols})
    rep = pd.DataFrame(rows).sort_values(["n_rows","ID"], ascending=[False, True])
    return rep

# Linhas duplicadas por chave dupla (ex: ["ID_OEMA","UF"])
def get_duplicate_rows_multi(df: pd.DataFrame, keys: list[str]) -> pd.DataFrame:
    # normaliza espaços nas chaves
    norm = df.copy()
    for k in keys:
        norm[k] = norm[k].astype("string").str.strip()
    mask = norm.duplicated(subset=keys, keep=False)
    return df.loc[mask].sort_values(keys)

# Todas as linhas de uma chave composta específica
def get_rows_for_keys(df: pd.DataFrame, keys: list[str], values: list) -> pd.DataFrame:
    m = pd.Series(True, index=df.index)
    for c, v in zip(keys, values):
        m &= df[c].astype("string").str.strip().eq(str(v).strip())
    return df.loc[m]

##### Salvar planilha com DADOS ESTAÇÕES

In [139]:
def save_UF_estacoes_csv(df_final, uf):
    base = Path.cwd().parent  
    out_dir = base / "data" / "DADOS_ESTACOES"
    out_dir.mkdir(parents=True, exist_ok=True)
    
    out_file = out_dir / f"{uf}_estacoes.csv"
    df_final.to_csv(out_file, index=False, encoding="utf-8")
    print("Saved to:", out_file.resolve())
    return

##### a) Ceará

In [399]:
# Substituir pelo estado desejado
uf = "CE" 

In [400]:
# Conferir respostas do formulário 
ce_forms = forms[forms["Unidade da Federação: "].str.contains(uf, case=False, na=False)]
ce_forms.iloc[:, idxs].head()

,Unidade da Federação:,Em sua UF há estações de monitoramento da qualidade do ar com equipamentos certificados como de referência ou equivalentes? \n,"Em caso afirmativo, a UF é responsável pela operação de algumas dessas estações?",Informe a quantidade de estações de operação própria:,Em sua UF há estações instaladas como condicionante de licença ambiental?,Informe a quantidade de estações instaladas por licença ambiental:,Em sua UF há estações de monitoramento indicativas (com sensores de baixo custo)?,Informe a quantidade de estações indicativas:,"Em 2024, houve mudanças na rede de monitoramento da qualidade do ar na UF?","Em caso afirmativo, indique quais mudanças ocorreram:",Quais são as perspectivas de curto prazo (1 ano) para a rede de monitoramento?,Deseja incluir observações sobre a rede de monitoramento da qualidade do ar? \n,Os dados de monitoramento da qualidade do ar são disponibilizados ao público?,"Em caso afirmativo, informe o endereço eletrônico para acesso:",Os dados de monitoramento passam por algum tratamento e formatação antes de serem disponibilizados?,"Em caso afirmativo, informe o método utilizado: \n(Exemplo: validação automática com base em padrões da CETESB)"
5,Ceará (CE),Sim,Sim,2,Não,0,Sim,8,Sim,Inativação de estações,Não há perspectivas de curto prazo,A Semace tem 2 estações de monitoramento da qu...,Não,NaN,Não,X


In [414]:
# Importar arquivos do estado
base = Path.cwd().parent 
df_dir = base / "data" / "DADOS_BRUTOS" 
df_dir.mkdir(parents=True, exist_ok=True)
print(df_dir)

ce_dir = df_dir / uf

/home/nobre/Notebooks/RQAR_2025_book/data/DADOS_BRUTOS


In [424]:
ce_dfs = load_csvs(ce_dir)

Loaded CIPP.csv: (21114, 25)
Loaded UM Parada Pecem.csv: (692, 24)
Loaded UM Parque Alto Alegre.csv: (557, 24)
Loaded UM UFC Reitoria.csv: (1922, 25)


In [425]:
for name, df in ce_dfs.items():
    print(f"\n=== {name} ===")
    display(df.head())


=== CIPP ===


,Date,BEN(),CH3-C6H5-CH3(µg/m³),CH4(ppm),CO(µg/m³),DD(grados),EBE(),HC(ppm),HR(%),LL(l/m²),...,PM10(µg/m³),PM10-MAN(µg/m³),PRB(hPa),RS(W/m²),SO2(µg/m³),SPM(),TIN(ºC),TMP(ºC),TOL(),VV(m/s)
0,25/08/2016 14:00,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,25/08/2016 15:00,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,25/08/2016 16:00,1.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,25/08/2016 17:00,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,25/08/2016 18:00,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN



=== UM Parada Pecem ===


,Date,BEN(),CH4(ppm),CO(µg/m³),DD(grados),EBE(),HC(ppm),HR(%),LL(l/m²),NMH(),...,PM10(µg/m³),PM25(),PRB(hPa),RS(W/m²),SO2(µg/m³),SPM(),TIN(ºC),TMP(ºC),TOL(),VV(m/s)
0,2019-07-25 07:00,0.1,2.0,276.0,188.0,0.2,2.5,74.0,0.0,0.52,...,7.0,NaN,1011.0,135.0,14.0,31.0,25.0,27.0,0.2,3.0
1,2019-07-25 08:00,0.2,2.0,276.0,193.0,0.3,2.6,69.0,0.0,0.53,...,7.0,NaN,1012.0,362.0,14.0,34.0,25.0,28.0,0.2,4.2
2,2019-07-25 09:00,0.1,1.9,228.0,200.0,0.1,2.4,60.0,0.0,0.52,...,4.0,NaN,1012.0,566.0,11.0,31.0,26.0,31.0,0.2,4.7
3,2019-07-25 10:00,0.1,1.9,228.0,190.0,0.1,2.4,56.0,0.0,0.53,...,16.0,NaN,1012.0,760.0,8.6,62.0,26.0,32.0,0.2,5.5
4,2019-07-25 11:00,0.1,1.9,252.0,192.0,0.1,2.4,54.0,0.0,0.50,...,13.0,NaN,1011.0,898.0,8.7,60.0,26.0,33.0,0.2,5.8



=== UM Parque Alto Alegre ===


,Date,BEN(),CH4(ppm),CO(µg/m³),DD(grados),EBE(),HC(ppm),HR(%),LL(l/m²),NMH(),...,PM10(µg/m³),PM25(),PRB(hPa),RS(W/m²),SO2(µg/m³),SPM(),TIN(ºC),TMP(ºC),TOL(),VV(m/s)
0,13/09/2019 15:00,0.12,2.2,927,120,0.10,3.6,42,0.0,1.4,...,153,NaN,1006,736,7.1,239,28,32,0.12,5.2
1,13/09/2019 16:00,0.12,2.2,1042,124,0.09,3.6,49,0.0,1.4,...,166,48.0,1006,438,10.0,253,28,31,0.12,4.5
2,13/09/2019 17:00,0.11,2.2,1099,130,0.09,3.7,54,0.0,1.5,...,150,52.0,1006,184,5.4,231,29,30,0.11,4.2
3,13/09/2019 18:00,0.11,2.2,1030,148,0.08,3.6,65,0.0,1.4,...,79,31.0,1006,15,4.0,137,29,28,0.10,2.3
4,13/09/2019 19:00,0.10,2.2,1042,138,0.08,3.6,75,0.0,1.4,...,53,20.0,1006,1,3.3,86,29,27,0.10,2.1



=== UM UFC Reitoria ===


,Fecha,BEN(),CH4(ppm),CO(µg/m³),DD(grados),EBE(),HC(ppm),HR(%),LL(l/m²),NMH(),...,PM10(µg/m³),PM25(),PRB(hPa),RS(W/m²),SO2(µg/m³),SPM(),TIN(ºC),TMP(ºC),TOL(),VV(m/s)
0,2019-02-04 08:00,0.0,0.0,0.0,208.0,0.0,0.0,72.0,0.1,0.0,...,3.0,3.0,1010,132.0,5.4,0.0,29,29.0,0.0,0.87
1,2019-02-04 09:00,0.0,0.0,792.0,210.0,0.0,0.0,75.0,0.0,0.0,...,3.0,3.0,1010,82.0,5.4,0.0,27,28.0,0.0,1.10
2,2019-02-04 10:00,0.0,0.0,1008.0,216.0,0.0,0.0,68.0,0.0,0.0,...,3.0,3.0,1010,333.0,5.4,0.0,25,29.0,0.0,0.79
3,2019-02-04 11:00,0.0,0.0,0.0,222.0,0.0,0.0,0.0,0.0,0.0,...,3.0,3.0,500,0.0,5.4,0.0,1,-30.0,0.0,1.30
4,2019-02-04 12:00,0.0,0.0,0.0,222.0,0.0,0.0,0.0,0.0,0.0,...,3.0,3.0,500,0.0,5.4,0.0,1,-30.0,0.0,1.30


In [426]:
# Renomear todas as colunas com data e hora para datetime
manual_map = {"fecha":  "DATETIME",
              "Date":  "DATETIME",
              "Fecha":  "DATETIME",
}

ce_dfs = {name: df.rename(columns=manual_map) for name, df in ce_dfs.items()}
ce_dfs

{'CIPP':                DATETIME  BEN()  CH3-C6H5-CH3(µg/m³)  CH4(ppm)  CO(µg/m³)  \
 0      25/08/2016 14:00    1.0                  NaN       NaN        NaN   
 1      25/08/2016 15:00    1.0                  NaN       NaN        NaN   
 2      25/08/2016 16:00    1.0                  0.0       NaN        NaN   
 3      25/08/2016 17:00    1.0                  NaN       NaN        NaN   
 4      25/08/2016 18:00    1.0                  NaN       NaN        NaN   
 ...                 ...    ...                  ...       ...        ...   
 21109  28/08/2023 11:00   25.0                  0.0       2.0      378.0   
 21110  28/08/2023 12:00   25.0                  0.0       1.0        NaN   
 21111  28/08/2023 13:00   24.0                  0.0       1.0        NaN   
 21112  28/08/2023 14:00   24.0                  0.0       1.0        NaN   
 21113  29/08/2023 09:00   25.0                  0.0       1.0        NaN   
 
        DD(grados)  EBE()  HC(ppm)  HR(%)  LL(l/m²)  ...  PM10(µg/

In [427]:
# Transformar coluna datetime para formato desejado
ce_dfs = {
    name: convert_column_to_datetime(d, column_name="DATETIME", format="%d/%m/%Y %H:%M")
    for name, d in ce_dfs.copy().items()
}

In [428]:
# Unir dataframes em único > Precisa ser feito antes de rodar a rotina de criação da planilha de estações
ce_frame = merge_station_dfs(ce_dfs.copy(), uf)
ce_frame.head()

,DATETIME,BEN(),CH3-C6H5-CH3(µg/m³),CH4(ppm),CO(µg/m³),DD(grados),EBE(),HC(ppm),HR(%),LL(l/m²),...,RS(W/m²),SO2(µg/m³),SPM(),TIN(ºC),TMP(ºC),TOL(),VV(m/s),ESTACAO,PM25(),O-XI(µg/m³)
0,2016-08-25 14:00:00,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,CIPP,NaN,NaN
1,2016-08-25 15:00:00,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,CIPP,NaN,NaN
2,2016-08-25 16:00:00,1.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,CIPP,NaN,NaN
3,2016-08-25 17:00:00,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,CIPP,NaN,NaN
4,2016-08-25 18:00:00,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,CIPP,NaN,NaN


In [429]:
# Corrigir problemas com datetime
ce_frame = fix_datetime_df(ce_frame, "DATETIME")
diag_bad_stations(ce_frame, "DATETIME")

UM Parada Pecem: parsed 0.0%  examples not parsed -> ['NaT', 'NaT', 'NaT', 'NaT', 'NaT']
UM UFC Reitoria: parsed 0.0%  examples not parsed -> ['NaT', 'NaT', 'NaT', 'NaT', 'NaT']


In [431]:
ce_dfs = build_inventory_rows(ce_frame.copy(), df_cod, uf, datetime_col='DATETIME', mode="columns")
ce_dfs = assign_id_mma(ce_dfs)
ce_dfs.head()

,UF,ID_OEMA,CIDADE,ID_MMA,ID_MMA_COMPLETO,POLUENTE,COD_POLUENTE,CD_MUN,COD_UF_IBGE,PROPRIETARIO,...,REALOCACAO,OBS_CALIBRACAO,INICIO,FIM,DADOS_MONITORAMENTO,RECONHECIDA,OBS_GERAIS,CERTIFICACAO,STATUS,REP_ESPACIAL_DECLARADA
0,CE,CIPP,,CE0001,CE0001,"BENZENO,CH4,CO,HCNM,HCT,MP10,MP25,NO,NO2,NOX,O...",,,23,,...,,,2016,2023,,,,,,
1,CE,UM Parque Alto Alegre,,CE0002,CE0002,"BENZENO,CH4,CO,HCNM,HCT,MP10,MP25,NO,NO2,NOX,O...",,,23,,...,,,2019,2019,,,,,,
2,CE,UM Parada Pecem,,CE0003,CE0003,"BENZENO,CH4,CO,HCNM,HCT,MP10,MP25,NO,NO2,NOX,O...",,,23,,...,,,<NA>,<NA>,,,,,,
3,CE,UM UFC Reitoria,,CE0004,CE0004,"BENZENO,CH4,CO,HCNM,HCT,MP10,MP25,NO,NO2,NOX,O...",,,23,,...,,,<NA>,<NA>,,,,,,


In [432]:
save_UF_estacoes_csv(ce_dfs, uf)

Saved to: /home/nobre/Notebooks/RQAR_2025_book/data/DADOS_ESTACOES/CE_estacoes.csv


##### b) Rio de Janeiro

In [232]:
# Substituir pelo estado desejado
uf = "RJ" 

In [233]:
# Conferir respostas do formulário 
rj_forms = forms[forms["Unidade da Federação: "].str.contains(uf, case=False, na=False)]
rj_forms.iloc[:, idxs].head()

,Unidade da Federação:,Em sua UF há estações de monitoramento da qualidade do ar com equipamentos certificados como de referência ou equivalentes? \n,"Em caso afirmativo, a UF é responsável pela operação de algumas dessas estações?",Informe a quantidade de estações de operação própria:,Em sua UF há estações instaladas como condicionante de licença ambiental?,Informe a quantidade de estações instaladas por licença ambiental:,Em sua UF há estações de monitoramento indicativas (com sensores de baixo custo)?,Informe a quantidade de estações indicativas:,"Em 2024, houve mudanças na rede de monitoramento da qualidade do ar na UF?","Em caso afirmativo, indique quais mudanças ocorreram:",Quais são as perspectivas de curto prazo (1 ano) para a rede de monitoramento?,Deseja incluir observações sobre a rede de monitoramento da qualidade do ar? \n,Os dados de monitoramento da qualidade do ar são disponibilizados ao público?,"Em caso afirmativo, informe o endereço eletrônico para acesso:",Os dados de monitoramento passam por algum tratamento e formatação antes de serem disponibilizados?,"Em caso afirmativo, informe o método utilizado: \n(Exemplo: validação automática com base em padrões da CETESB)"
22,Rio de Janeiro (RJ),Sim,Não,46,Sim,144,Não,0,Sim,"Expansão da rede (novas estações), Reativação ...",Ampliação da rede estadual,O questionário foi preenchido com base nas est...,Sim,https://portalsigqar.inea.rj.gov.br/,Sim,Validação automática e também pela equipe de a...


In [234]:
# Importar arquivos do estado
base = Path.cwd().parent 
df_dir = base / "data" / "DADOS_ESTACOES" 
df_dir.mkdir(parents=True, exist_ok=True)
print(df_dir)

rj_dir = df_dir / uf 
os.listdir(rj_dir)
#print(rj_dir)

/home/nobre/Notebooks/RQAR_2025_book/data/DADOS_ESTACOES


['RJ_estacoes_nao_preenchidas.xlsx',
 'RJ_estacoes_enviada.xlsx',
 'RJ_estacoes.csv']

In [235]:
''' Neste caso, a inspeção das pastas indiciou que existiam arquivos .XLSX e .CSV, 
    necessitando aplicar duas operações diferentes para leitura dos arquivos'''

rj_exls = load_excels(rj_dir) # Está organizado para ler sempre a primeira planilha
rj_csvs = load_csvs(rj_dir)

Loaded RJ_estacoes_enviada.xlsx [sheet 0]: (64, 29)
Loaded RJ_estacoes_nao_preenchidas.xlsx [sheet 0]: (57, 31)
Loaded RJ_estacoes.csv: (121, 31)


In [236]:
''' Neste caso, como são dois dicionários diferentes - uma para arquivos .XLSX e uma para arquivos .CSV,
    foi necessário fazer um merge entre eles. Esse passo vai existir nessas condições'''

rj_dfs = rj_csvs | rj_exls 

In [237]:
for name, df in rj_dfs.items():
    print(f"\n=== {name} ===")
    display(df.head())


=== RJ_estacoes ===


,UF,COD_UF_IBGE,ID_OEMA,ID_MMA,CIDADE,CD_MUN,CATEGORIA,FUNCIONAMENTO,PROPRIETARIO,PROP_ENTIDADE,...,FIM,STATUS,CALIBRACAO,OBS_CALIBRACAO,MONITORAR,FONTE,OBS_GERAIS,DADOS_MONITORAMENTO,RECONHECIDA,REP_ESPACIAL_DECLARADA
0,RJ,33,RJ - Largo do Bodegão,RJ0012,Rio de Janeiro,3304557.0,Referencia,Automatica,NaN,NaN,...,NaN,Ativa,NaN,NaN,NaN,Consulta 2024,NaN,NaN,NaN,NaN
1,RJ,33,BR - São Bernardo,RJ0018,Belford Roxo,3300456.0,Referencia,Automatica,NaN,NaN,...,NaN,Ativa,NaN,NaN,NaN,Consulta 2024,NaN,NaN,NaN,NaN
2,RJ,33,NI - Monteiro Lobato,RJ0019,Nova Iguaçu,3303500.0,Referencia,Automatica,NaN,NaN,...,NaN,Ativa,NaN,NaN,NaN,Consulta 2024,NaN,NaN,NaN,NaN
3,RJ,33,RJ - Campo dos Afonsos,RJ0020,Rio de Janeiro,3304557.0,Referencia,Automatica,NaN,NaN,...,NaN,Ativa,NaN,NaN,NaN,Consulta 2024,NaN,NaN,NaN,NaN
4,RJ,33,RJ - Taquara,RJ0021,Rio de Janeiro,3304557.0,Referencia,Automatica,NaN,NaN,...,NaN,Ativa,NaN,NaN,NaN,Consulta 2024,NaN,NaN,NaN,NaN



=== RJ_estacoes_enviada ===


,UF,COD_UF_IBGE,ID_OEMA,ID_MMA,CIDADE,CD_MUN,CATEGORIA,FUNCIONAMENTO,PROPRIETARIO,PROP_ENTIDADE,...,FINALIDADE,REP_ESPACIAL,INICIO,FIM,STATUS,CALIBRACAO,OBS_CALIBRACAO,MONITORAR,FONTE,OBS_GERAIS
0,RJ,33,BM - Boa Sorte,RJ0073,Barra Mansa,3300407,Referencia,Automática,Saint Gobain,Privada,...,Licenciamento Ambiental,Bairro,NaN,NaN,Ativa,Sim,NaN,Não,Coleta Interna,NaN
1,RJ,33,BM - Sesi,RJ0074,Barra Mansa,3300407,Referencia,Automática,Saint Gobain,Privada,...,Licenciamento Ambiental,Bairro,NaN,NaN,Ativa,Sim,NaN,Não,Coleta Interna,NaN
2,RJ,33,BM - Bocaininha,RJ0075,Barra Mansa,3300407,Referencia,Automática,Arcelormittal Barra Mansa,Privada,...,Licenciamento Ambiental,Bairro,NaN,NaN,Ativa,Sim,NaN,Não,Coleta Interna,NaN
3,RJ,33,BM - Roberto Silveira,RJ0076,Barra Mansa,3300407,Referencia,Automática,Arcelormittal Barra Mansa,Privada,...,Licenciamento Ambiental,Bairro,NaN,NaN,Ativa,Sim,NaN,Não,Coleta Interna,NaN
4,RJ,33,BM - Vista Alegre,RJ0077,Barra Mansa,3300407,Referencia,Automática,Arcelormittal Barra Mansa,Privada,...,Licenciamento Ambiental,Bairro,NaN,NaN,Ativa,Sim,NaN,Não,Coleta Interna,NaN



=== RJ_estacoes_nao_preenchidas ===


,UF,COD_UF_IBGE,ID_OEMA,ID_MMA,CIDADE,CD_MUN,CATEGORIA,FUNCIONAMENTO,PROPRIETARIO,PROP_ENTIDADE,...,FIM,STATUS,CALIBRACAO,OBS_CALIBRACAO,MONITORAR,FONTE,OBS_GERAIS,DADOS_MONITORAMENTO,RECONHECIDA,REP_ESPACIAL_DECLARADA
0,RJ,33,E. Móvel - Barra Mansa,RJ0299,Barra Mansa,3300407.0,Referencia,Automatica,INEA,Pública,...,NaN,Inativa,x,NaN,Não,NaN,NaN,NaN,NaN,NaN
1,RJ,33,E. Móvel - Belford Roxo,RJ0298,Belford Roxo,3300456.0,Referencia,Automatica,INEA,Pública,...,NaN,Inativa,x,NaN,Não,NaN,NaN,NaN,NaN,NaN
2,RJ,33,Estação Meteorológica - Ute Campos,RJ0613,Campos dos Goytacazes,3301009.0,Referencia,Automatica,Furnas S.A,Privada,...,NaN,Inativa,x,NaN,Não,NaN,NaN,NaN,NaN,NaN
3,RJ,33,Cg - Meteorológica Euclidelândia 1,RJ0053,Cantagalo,3301108.0,Referencia,Automatica,Votorantim Cimentos S.A,Privada,...,NaN,Ativa,x,NaN,Não,NaN,NaN,NaN,NaN,NaN
4,RJ,33,DC - Vila São Luiz,RJ0033,Duque de Caxias,3301702.0,Referencia,Automatica,REDUC,Privada,...,NaN,Inativa,x,NaN,Não,Consulta 2024,NaN,NaN,NaN,NaN


In [238]:
# Encontrar colunas diferentes e iguais dentro dos dicts 
summary, stats = summarize_columns(rj_dfs, normalize=True)

# Colunas iguais em todos
cols_comuns = summary.loc[summary.is_common, "col"].tolist()
print("Qtd colunas comuns:", len(cols_comuns))
print("Algumas comuns:", cols_comuns[:10])

# Colunas diferentes em todos
cols_diff = summary.loc[~ summary.is_common, "col"].tolist()
print("Qtd colunas diferentes:", len(cols_diff))
print("Algumas diferentes:", cols_diff[:10])

# Estatísticas por DataFrame
stats[["df","n_cols","n_common_present","n_common_missing","extras_count"]].head()

Qtd colunas comuns: 28
Algumas comuns: ['calibracao', 'categoria', 'cd_mun', 'cidade', 'cod_uf_ibge', 'fim', 'finalidade', 'fonte', 'funcionamento', 'id_mma']
Qtd colunas diferentes: 4
Algumas diferentes: ['dados_monitoramento', 'reconhecida', 'rep_espacial_declarada', 'rep_espacial']


,df,n_cols,n_common_present,n_common_missing,extras_count
0,RJ_estacoes,31,28,0,3
1,RJ_estacoes_enviada,29,28,0,1
2,RJ_estacoes_nao_preenchidas,31,28,0,3


In [239]:
# Limpar caracteres indesejados
for name, d in rj_dfs.items(): clean_df_all_text(d, in_place=True)

In [240]:
# Depois de rodar pela primeira vez e identificar os conflitos, posso escolher a base de dados prioritária para sobrepor informações
priority = ["RJ_estacoes_enviada", "RJ_estacoes_nao_preenchidas", "RJ_estacoes"]

rj_frame, conflicts = merge_by_id_multi(
    rj_dfs.copy(),
    id_cols=['ID_OEMA','POLUENTE'],
    source_priority=priority,
    drop_empty_strings=True,
)
rj_frame

,ID_OEMA,POLUENTE,UF,COD_UF_IBGE,ID_MMA,CIDADE,CD_MUN,CATEGORIA,FUNCIONAMENTO,PROPRIETARIO,...,STATUS,CALIBRACAO,OBS_CALIBRACAO,MONITORAR,FONTE,OBS_GERAIS,DADOS_MONITORAMENTO,RECONHECIDA,REP_ESPACIAL_DECLARADA,REP_ESPACIAL
0,RJ - Largo do Bodegao,"MP10,NO,.ipynb_checkpoints,HCT,BENZENO,PTS,CH4...",RJ,33,RJ0012,Rio de Janeiro,3304557.0,Referencia,Automatica,<NA>,...,Ativa,<NA>,<NA>,<NA>,Consulta 2024,<NA>,<NA>,<NA>,<NA>,<NA>
1,BR - Sao Bernardo,"MP10,NO,.ipynb_checkpoints,HCT,CO,CH4,O3,NO2,H...",RJ,33,RJ0018,Belford Roxo,3300456.0,Referencia,Automatica,<NA>,...,Ativa,<NA>,<NA>,<NA>,Consulta 2024,<NA>,<NA>,<NA>,<NA>,<NA>
2,NI - Monteiro Lobato,"MP10,NO,HCT,CO,CH4,O3,SO2,NO2,HCNM,NOX",RJ,33,RJ0019,Nova Iguacu,3303500.0,Referencia,Automatica,INEA,...,Ativa,Sim,<NA>,Nao,Coleta Interna,<NA>,<NA>,<NA>,<NA>,Bairro
3,RJ - Campo dos Afonsos,"NO,O3,NO2,NOX",RJ,33,RJ0020,Rio de Janeiro,3304557.0,Referencia,Automatica,INEA,...,Ativa,Sim,<NA>,Nao,Coleta Interna,<NA>,<NA>,<NA>,<NA>,Bairro
4,RJ - Taquara,"MP10,NO,HCT,CO,CH4,O3,SO2,NO2,HCNM,NOX",RJ,33,RJ0021,Rio de Janeiro,3304557.0,Referencia,Automatica,INEA,...,Ativa,Sim,<NA>,Nao,Coleta Interna,<NA>,<NA>,<NA>,<NA>,Bairro
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
102,Mt - Praia Do Saco,"MP10,PTS",RJ,33,RJ0068,Mangaratiba,3302601.0,Referencia,Automatica,Vale S. A,...,Ativa,Sim,<NA>,Nao,Coleta Interna,<NA>,<NA>,<NA>,<NA>,Urbana
103,RJ - Largo do Bodegao,"MP10,NO,HCT,BENZENO,PTS,CH4,O3,ETILBENZENO,SO2...",RJ,33,RJ0012,Rio de Janeiro,3304557.0,Referencia,Automatica,Ternium Brasil,...,Ativa,Sim,<NA>,Nao,Coleta Interna,<NA>,<NA>,<NA>,<NA>,Bairro
104,RJ - Manguinhos,"MP10,MP2,5,NO,HCT,CO,CH4,O3,SO2,NO2,HCNM,NOX,MP25",RJ,33,RJ0029,Rio de Janeiro,3304557.0,Referencia,Automatica,Refinaria de Manguinhos (Refit),...,Ativa,Sim,<NA>,Nao,Coleta Interna,<NA>,<NA>,<NA>,<NA>,Urbana
105,RJ - Joao XXIII (Caminhao),"NO,HCT,BENZENO,CH4,O3,ETILBENZENO,SO2,NO2,TOLU...",RJ,33,RJ0216,Rio de Janeiro,3304557.0,Referencia,Automatica,INEA,...,Inativa,x,<NA>,Nao,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>


In [241]:
rj_frame.columns

Index(['ID_OEMA', 'POLUENTE', 'UF', 'COD_UF_IBGE', 'ID_MMA', 'CIDADE',
       'CD_MUN', 'CATEGORIA', 'FUNCIONAMENTO', 'PROPRIETARIO', 'PROP_ENTIDADE',
       'OPERADOR', 'OP_ENTIDADE', 'LATITUDE', 'LONGITUDE', 'MOBILIDADE',
       'REALOCACAO', 'MARCA', 'METODO', 'FINALIDADE', 'INICIO', 'FIM',
       'STATUS', 'CALIBRACAO', 'OBS_CALIBRACAO', 'MONITORAR', 'FONTE',
       'OBS_GERAIS', 'DADOS_MONITORAMENTO', 'RECONHECIDA',
       'REP_ESPACIAL_DECLARADA', 'REP_ESPACIAL'],
      dtype='object')

In [242]:
# Lista de colunas alvo que já existem no df (na ordem desejada)
cols = [
    "ID_OEMA","ID_MMA","CIDADE","CD_MUN","CATEGORIA",
    "FUNCIONAMENTO","PROPRIETARIO","PROP_ENTIDADE","OPERADOR","OP_ENTIDADE",
    "LATITUDE","LONGITUDE","MOBILIDADE","REALOCACAO","MARCA","METODO","FINALIDADE",
    "STATUS","CALIBRACAO","OBS_CALIBRACAO","MONITORAR","FONTE",
    "OBS_GERAIS","DADOS_MONITORAMENTO","RECONHECIDA","REP_ESPACIAL_DECLARADA",
    "REP_ESPACIAL"
]

def make_field_map(cols,
                   prefer_mode=("CIDADE",),     # PT-BR: colunas que preferem moda
                   use_group_key=("ID_OEMA",)): # PT-BR: colunas que vêm da chave do grupo
    fm = {}
    for c in cols:
        if c in use_group_key:
            fm[c] = ("group_key",)
        elif c in prefer_mode:
            fm[c] = ("col", c, {"strategy": "mode"})
        else:
            fm[c] = ("col", c, {"strategy": "first_non_null"})
    return fm

field_map = make_field_map(cols)
teste = build_inventory_flexible(
    rj_frame.copy(),
    uf,
    df_cod,
    group_col="ID_OEMA",
    datetime_col=None,  # ou "DATETIME" se desejar calcular INICIO/FIM
    mode= "rows",
    field_map=field_map
)
teste

,UF,ID_OEMA,POLUENTE,COD_UF_IBGE,INICIO,FIM,CIDADE,ID_MMA,ID_MMA_COMPLETO,COD_POLUENTE,...,CALIBRACAO,REALOCACAO,OBS_CALIBRACAO,DADOS_MONITORAMENTO,RECONHECIDA,OBS_GERAIS,CERTIFICACAO,STATUS,REP_ESPACIAL_DECLARADA,REP_ESPACIAL
0,RJ,BM - Boa Sorte,"MP10,PTS",33,,,Barra Mansa,RJ0073,,,...,Sim,Nao,None,None,None,None,,Ativa,None,Bairro
1,RJ,BM - Bocaininha,"MP10,PTS",33,,,Barra Mansa,RJ0075,,,...,Sim,Nao,None,None,None,None,,Ativa,None,Bairro
2,RJ,BM - Roberto Silveira,"MP10,PTS",33,,,Barra Mansa,RJ0076,,,...,Sim,Nao,None,None,None,None,,Ativa,None,Bairro
3,RJ,BM - Sesi,"MP10,PTS",33,,,Barra Mansa,RJ0074,,,...,Sim,Nao,None,None,None,None,,Ativa,None,Bairro
4,RJ,BM - Vista Alegre,"MP10,PTS",33,,,Barra Mansa,RJ0077,,,...,Sim,Nao,None,None,None,None,,Ativa,None,Bairro
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,RJ,Sp - Piranema,"CH4,CO,HCNM,HCT,MP10,NO,NO2,NOX,O3,SO2",33,,,Seropedica,RJ0058,,,...,Sim,Nao,None,None,None,None,,Ativa,None,Urbana
96,RJ,VR - Belmonte,"BENZENO,CH4,CO,ETILBENZENO,HCNM,HCT,MP10,MP25,...",33,,,Volta Redonda,RJ0069,,,...,Sim,Nao,None,None,None,None,,Ativa,None,Bairro
97,RJ,VR - Nossa Sra. das Gracas (Van),"MP10,MP25,NO,NO2,NOX,PTS,SO2",33,,,Volta Redonda,RJ0637,,,...,Sim,Nao,None,None,None,None,,Ativa,None,Bairro
98,RJ,VR - Retiro,"BENZENO,CH4,CO,ETILBENZENO,HCNM,HCT,MP10,MP25,...",33,,,Volta Redonda,RJ0070,,,...,Sim,Nao,None,None,None,None,,Ativa,None,Bairro


In [278]:
save_UF_estacoes_csv(teste, uf)

Saved to: /home/nobre/Notebooks/RQAR_2025_book/data/DADOS_ESTACOES/RJ_estacoes.csv


In [244]:
print("IDs orig:", rj_frame["ID_OEMA"].astype("string").str.strip().dropna().nunique())
print("IDs inv :", teste["ID_OEMA"].astype("string").str.strip().dropna().nunique())

# ver se POLUENTE varia por linha (não deve ser igual para todas)
print(teste["POLUENTE"].nunique(), "poluente(s) distintos")


IDs orig: 100
IDs inv : 100
45 poluente(s) distintos


In [243]:
dups_df = get_duplicate_rows(rj_frame, key="ID_OEMA")
dups_df

,ID_OEMA,POLUENTE,UF,COD_UF_IBGE,ID_MMA,CIDADE,CD_MUN,CATEGORIA,FUNCIONAMENTO,PROPRIETARIO,...,STATUS,CALIBRACAO,OBS_CALIBRACAO,MONITORAR,FONTE,OBS_GERAIS,DADOS_MONITORAMENTO,RECONHECIDA,REP_ESPACIAL_DECLARADA,REP_ESPACIAL
1,BR - Sao Bernardo,"MP10,NO,.ipynb_checkpoints,HCT,CO,CH4,O3,NO2,H...",RJ,33,RJ0018,Belford Roxo,3300456.0,Referencia,Automatica,<NA>,...,Ativa,<NA>,<NA>,<NA>,Consulta 2024,<NA>,<NA>,<NA>,<NA>,<NA>
93,BR - Sao Bernardo,"MP10,NO,HCT,CO,CH4,O3,NO2,HCNM,NOX",RJ,33,RJ0018,Belford Roxo,3300456.0,Referencia,Automatica,INEA,...,Ativa,Sim,<NA>,Nao,Coleta Interna,<NA>,<NA>,<NA>,<NA>,Bairro
28,Cg - Macuco,"MP10,NO,PTS,O3,NO2,NOX,MP25",RJ,33,RJ0051,Macuco,3302452.0,Referencia,Automatica,<NA>,...,Ativa,<NA>,<NA>,<NA>,Consulta 2024,<NA>,<NA>,<NA>,<NA>,<NA>
101,Cg - Macuco,"MP10,NO,O3,NO2,NOX,MP25",RJ,33,RJ0051,Macuco,3302452.0,Referencia,Automatica,CSN Cimento - Cantagalo,...,Ativa,Sim,<NA>,Nao,Coleta Interna,<NA>,<NA>,<NA>,<NA>,Urbana
19,Itb - Porto das Caixas,"MP10,NO,HCT,BENZENO,CO,CH4,O3,ETILBENZENO,SO2,...",RJ,33,RJ0038,Itaborai,3301900.0,Referencia,Automatica,<NA>,...,Ativa,<NA>,<NA>,<NA>,Consulta 2024,<NA>,<NA>,<NA>,<NA>,<NA>
96,Itb - Porto das Caixas,"MP10,PTS,NO,HCT,BENZENO,CO,CH4,O3,ETILBENZENO,...",RJ,33,RJ0038,Itaborai,3301900.0,Referencia,Automatica,Complexo de Energias Boaventura,...,Ativa,Sim,<NA>,Nao,Coleta Interna,<NA>,<NA>,<NA>,<NA>,Bairro
37,Itg - Ilha Da Madeira,"MP10,PTS,MP25",RJ,33,RJ0066,Itaguai,3302007.0,Referencia,Automatica,<NA>,...,Ativa,<NA>,<NA>,<NA>,Consulta 2024,<NA>,<NA>,<NA>,<NA>,<NA>
99,Itg - Ilha Da Madeira,"MP10,PTS",RJ,33,RJ0066,Itaguai,3302007.0,Referencia,Automatica,Vale S. A,...,Ativa,Sim,<NA>,Nao,Coleta Interna,<NA>,<NA>,<NA>,<NA>,Urbana
39,Mt - Praia Do Saco,"MP10,PTS,MP25",RJ,33,RJ0068,Mangaratiba,3302601.0,Referencia,Automatica,<NA>,...,Ativa,<NA>,<NA>,<NA>,Consulta 2024,<NA>,<NA>,<NA>,<NA>,<NA>
102,Mt - Praia Do Saco,"MP10,PTS",RJ,33,RJ0068,Mangaratiba,3302601.0,Referencia,Automatica,Vale S. A,...,Ativa,Sim,<NA>,Nao,Coleta Interna,<NA>,<NA>,<NA>,<NA>,Urbana


In [129]:
# # mapeia campos extras que quer pegar das linhas
# field_map = {
#     "ID_OEMA": ("col", "ID_OEMA", {"strategy": "first_non_null"}),
#     "CIDADE":  ("col", "CIDADE", {"strategy": "mode"}),
# }

# teste = build_inventory_flexible(
#     rj_frame.copy(),
#     uf,
#     df_cod,
#     group_col="ID_OEMA",
#     datetime_col=None,
#     mode= "rows",       
#     field_map=field_map
# )
# teste

,UF,ID_OEMA,CIDADE,ID_MMA,ID_MMA_COMPLETO,POLUENTE,COD_POLUENTE,CD_MUN,COD_UF_IBGE,PROPRIETARIO,...,REALOCACAO,OBS_CALIBRACAO,INICIO,FIM,DADOS_MONITORAMENTO,RECONHECIDA,OBS_GERAIS,CERTIFICACAO,STATUS,REP_ESPACIAL_DECLARADA
0,RJ,BM - Boa Sorte,Barra Mansa,,,"BENZENO,CH4,CO,ETILBENZENO,HCNM,HCT,MP10,MP25,...",,,33,,...,,,None,None,,,,,,
1,RJ,BM - Bocaininha,Barra Mansa,,,"BENZENO,CH4,CO,ETILBENZENO,HCNM,HCT,MP10,MP25,...",,,33,,...,,,None,None,,,,,,
2,RJ,BM - Roberto Silveira,Barra Mansa,,,"BENZENO,CH4,CO,ETILBENZENO,HCNM,HCT,MP10,MP25,...",,,33,,...,,,None,None,,,,,,
3,RJ,BM - Sesi,Barra Mansa,,,"BENZENO,CH4,CO,ETILBENZENO,HCNM,HCT,MP10,MP25,...",,,33,,...,,,None,None,,,,,,
4,RJ,BM - Vista Alegre,Barra Mansa,,,"BENZENO,CH4,CO,ETILBENZENO,HCNM,HCT,MP10,MP25,...",,,33,,...,,,None,None,,,,,,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
119,RJ,VR - Belmonte,Volta Redonda,,,"BENZENO,CH4,CO,ETILBENZENO,HCNM,HCT,MP10,MP25,...",,,33,,...,,,None,None,,,,,,
120,RJ,VR - Meteorológica Ilha das Águas Cruas,Volta Redonda,,,"BENZENO,CH4,CO,ETILBENZENO,HCNM,HCT,MP10,MP25,...",,,33,,...,,,None,None,,,,,,
121,RJ,VR - Nossa Sra. das Graças (Van),Volta Redonda,,,"BENZENO,CH4,CO,ETILBENZENO,HCNM,HCT,MP10,MP25,...",,,33,,...,,,None,None,,,,,,
122,RJ,VR - Retiro,Volta Redonda,,,"BENZENO,CH4,CO,ETILBENZENO,HCNM,HCT,MP10,MP25,...",,,33,,...,,,None,None,,,,,,


In [301]:
#save_UF_estacoes_csv(rj_dfs, uf)

Saved to: /home/nobre/Notebooks/RQAR_2025_book/data/DADOS_ESTACOES/CE_estacoes.csv


##### c) Paraíba

In [433]:
# Substituir pelo estado desejado
uf = "PB" 

In [434]:
# Conferir respostas do formulário 
pb_forms = forms[forms["Unidade da Federação: "].str.contains(uf, case=False, na=False)]
pb_forms.iloc[:, idxs].head()

,Unidade da Federação:,Em sua UF há estações de monitoramento da qualidade do ar com equipamentos certificados como de referência ou equivalentes? \n,"Em caso afirmativo, a UF é responsável pela operação de algumas dessas estações?",Informe a quantidade de estações de operação própria:,Em sua UF há estações instaladas como condicionante de licença ambiental?,Informe a quantidade de estações instaladas por licença ambiental:,Em sua UF há estações de monitoramento indicativas (com sensores de baixo custo)?,Informe a quantidade de estações indicativas:,"Em 2024, houve mudanças na rede de monitoramento da qualidade do ar na UF?","Em caso afirmativo, indique quais mudanças ocorreram:",Quais são as perspectivas de curto prazo (1 ano) para a rede de monitoramento?,Deseja incluir observações sobre a rede de monitoramento da qualidade do ar? \n,Os dados de monitoramento da qualidade do ar são disponibilizados ao público?,"Em caso afirmativo, informe o endereço eletrônico para acesso:",Os dados de monitoramento passam por algum tratamento e formatação antes de serem disponibilizados?,"Em caso afirmativo, informe o método utilizado: \n(Exemplo: validação automática com base em padrões da CETESB)"
20,Paraíba (PB),Sim,Sim,4,Não,0,Sim,14,Sim,Expansão da rede (novas estações),Início do monitoramento com estações certificadas,A Paraíba possui quatorze monitores indicativo...,Não,NaN,Não,X


In [559]:
# Importar arquivos do estado
base = Path.cwd().parent 
df_dir = base / "data" / "DADOS_BRUTOS" 
df_dir.mkdir(parents=True, exist_ok=True)

pb_dir = df_dir / uf
print(pb_dir)

/home/nobre/Notebooks/RQAR_2025_book/data/DADOS_BRUTOS/PB


In [560]:
pb_dfs = load_txts(pb_dir)

Loaded Estação 1.txt: (2810, 8)
Loaded Estação 2.txt: (2395, 8)
Loaded Estação 3.txt: (3175, 8)


In [561]:
for name, df in pb_dfs.items():
    print(f"\n=== {name} ===")
    display(df.head())


=== Estação 1 ===


,0,1,2,3,4,5,6,7
0,2024-08-22T20:00:00,23040019,SUDEMA,-999999.00,-999999.00,-999999.00,-999999.00,-999999.0
1,2024-08-22T21:00:00,23040019,SUDEMA,54.25,42.17,17.42,15.33,12.8
2,2024-08-22T22:00:00,23040019,SUDEMA,54.92,43.92,19.67,17.58,12.8
3,2024-08-22T23:00:00,23040019,SUDEMA,47.75,39.75,19.50,17.92,12.7
4,2024-08-23T00:00:00,23040019,SUDEMA,79.67,60.00,21.08,18.08,12.7



=== Estação 2 ===


,0,1,2,3,4,5,6,7
0,2024-08-22T19:00:00,23040016,SUDEMA,46.40,35.60,16.40,14.40,12.5
1,2024-08-22T20:00:00,23040016,SUDEMA,63.67,47.33,19.67,16.67,12.5
2,2024-08-22T21:00:00,23040016,SUDEMA,57.17,43.58,19.67,17.00,12.4
3,2024-08-22T22:00:00,23040016,SUDEMA,53.42,41.67,19.92,17.75,12.4
4,2024-08-22T23:00:00,23040016,SUDEMA,53.33,41.75,20.25,17.42,12.4



=== Estação 3 ===


,0,1,2,3,4,5,6,7
0,2024-08-21T16:00:00,23040023,SUDEMA,22.22,21.22,18.78,14.56,12.7
1,2024-08-21T17:00:00,23040023,SUDEMA,28.92,27.25,24.08,18.50,12.5
2,2024-08-21T18:00:00,23040023,SUDEMA,31.58,29.75,26.58,20.50,12.4
3,2024-08-21T19:00:00,23040023,SUDEMA,36.42,34.50,30.25,22.83,12.4
4,2024-08-21T20:00:00,23040023,SUDEMA,41.42,38.50,32.92,24.58,12.3


NOTA: Informações fornecidas pela UF

Localização:

Estação 1: 7°2'4.75"S 34°50'34.73"W

Estação 2: 7°5'11.20"S 34°50'56.19"W

Estação 3: 7°2'50.65"S 34°57'23.56"W

Formato do arquivo de dados:[Data]T[Hora];[Serial];[NOME DO ÓRGÃO];[PTS];[PM10];[PM2.5];[PM1];[VOLTAGEM DA BATERIA];

In [562]:
# Adicionar informações de cabeçalho 
# Manual_map foi montado com base nas informações fornecidas pelo estado sobre os arquivos .txt
manual_map = [
    "DATETIME", "serial", "PROPRIETARIO", "PTS",
    "MP10", "MP25", "MP1", "voltagem_bateria"
]

pb_dfs = {name: df.set_axis(manual_map, axis=1) for name, df in pb_dfs.items()}

In [563]:
pb_frame = merge_station_dfs(pb_dfs.copy(), uf)

In [564]:
# Limpar linhas de datetime para aceitar o formato
pb_frame["DATETIME"] = (pb_frame["DATETIME"].astype(str).str.replace("T", " "))

In [569]:
# Criar planilha final 
pb_dfs = build_inventory_rows(pb_frame.copy(), df_cod, uf, datetime_col='DATETIME', mode="columns")
pb_dfs["PROPRIETARIO"] = pb_frame["PROPRIETARIO"]

/tmp/ipykernel_70933/4086501568.py:12: UserWarning: Parsing dates in %Y-%m-%d %H:%M:%S format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  s = pd.to_datetime(g[datetime_col], dayfirst=True)
/tmp/ipykernel_70933/4086501568.py:12: UserWarning: Parsing dates in %Y-%m-%d %H:%M:%S format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  s = pd.to_datetime(g[datetime_col], dayfirst=True)
/tmp/ipykernel_70933/4086501568.py:12: UserWarning: Parsing dates in %Y-%m-%d %H:%M:%S format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  s = pd.to_datetime(g[datetime_col], dayfirst=True)


In [574]:
# Adicionar colunas de latitude e longitude
lat_map = {
    "Estação 1": -7.034652778,
    "Estação 2": -7.086444444,
    "Estação 3": -7.047402778,
}
lon_map = {
    "Estação 1": -34.842980556,
    "Estação 2": -34.848941667,
    "Estação 3": -34.956544444,
}

pb_dfs["LATITUDE"]  = pb_dfs["ID_OEMA"].map(lat_map)
pb_dfs["LONGITUDE"] = pb_dfs["ID_OEMA"].map(lon_map)

In [572]:
pb_dfs = assign_id_mma(pb_dfs)
pb_dfs.head()

,UF,ID_OEMA,CIDADE,ID_MMA,ID_MMA_COMPLETO,POLUENTE,COD_POLUENTE,CD_MUN,COD_UF_IBGE,PROPRIETARIO,...,REALOCACAO,OBS_CALIBRACAO,INICIO,FIM,DADOS_MONITORAMENTO,RECONHECIDA,OBS_GERAIS,CERTIFICACAO,STATUS,REP_ESPACIAL_DECLARADA
0,PB,Estação 1,,PB0001,PB0001,"MP1 ,MP10,MP25,PTS",,,25,SUDEMA,...,,,2024,2024,,,,,,
1,PB,Estação 2,,PB0002,PB0002,"MP1 ,MP10,MP25,PTS",,,25,SUDEMA,...,,,2024,2024,,,,,,
2,PB,Estação 3,,PB0003,PB0003,"MP1 ,MP10,MP25,PTS",,,25,SUDEMA,...,,,2024,2024,,,,,,


In [573]:
save_UF_estacoes_csv(pb_dfs, uf)

Saved to: /home/nobre/Notebooks/RQAR_2025_book/data/DADOS_ESTACOES/PB_estacoes.csv


##### d) Pernambuco

1. Dados enviados pela UF

In [42]:
# Substituir pelo estado desejado
uf = "PE" 

In [43]:
# Conferir respostas do formulário 
pe_forms = forms[forms["Unidade da Federação: "].str.contains(uf, case=False, na=False)]
pe_forms.iloc[:, idxs].head()

,Unidade da Federação:,Em sua UF há estações de monitoramento da qualidade do ar com equipamentos certificados como de referência ou equivalentes? \n,"Em caso afirmativo, a UF é responsável pela operação de algumas dessas estações?",Informe a quantidade de estações de operação própria:,Em sua UF há estações instaladas como condicionante de licença ambiental?,Informe a quantidade de estações instaladas por licença ambiental:,Em sua UF há estações de monitoramento indicativas (com sensores de baixo custo)?,Informe a quantidade de estações indicativas:,"Em 2024, houve mudanças na rede de monitoramento da qualidade do ar na UF?","Em caso afirmativo, indique quais mudanças ocorreram:",Quais são as perspectivas de curto prazo (1 ano) para a rede de monitoramento?,Deseja incluir observações sobre a rede de monitoramento da qualidade do ar? \n,Os dados de monitoramento da qualidade do ar são disponibilizados ao público?,"Em caso afirmativo, informe o endereço eletrônico para acesso:",Os dados de monitoramento passam por algum tratamento e formatação antes de serem disponibilizados?,"Em caso afirmativo, informe o método utilizado: \n(Exemplo: validação automática com base em padrões da CETESB)"
16,Pernambuco (PE),Sim,Não,0,Sim,4,Sim,11,Não,Não se aplica – não existem estações na UF,Ampliação da rede estadual,As 11 estações de baixo custo são operadas pel...,Sim,https://monitorar.mma.gov.br/mapa,Sim,Validação automática realizada na concepção do...
33,Sergipe (SE),Não,Não se aplica - não existem estações de referê...,0,Não,0,Não,0,Não se aplica - não existem estações na UF,Não se aplica – não existem estações na UF,Início do monitoramento com estações de baixo ...,Existem monitoramento nas saídas das chaminés ...,Não se aplica - não existem estações na UF,NaN,Não se aplica - não existem estações na UF,X


In [44]:
# Importar arquivos do estado
base = Path.cwd().parent 
df_dir = base / "data" / "DADOS_BRUTOS" 
df_dir.mkdir(parents=True, exist_ok=True)

pe_dir = df_dir / uf
print(pe_dir)

/home/nobre/Notebooks/RQAR_2025_book/data/DADOS_BRUTOS/PE


In [89]:
pe_dfs = load_excels(pe_dir,sheets=0)

Loaded Multi Station Report _2019.xlsx [sheet 0]: (8774, 25)
Loaded Multi Station Report _2020.xlsx [sheet 0]: (8798, 25)
Loaded Multi Station Report _2021.xlsx [sheet 0]: (8774, 25)
Loaded Multi Station Report _2022.xlsx [sheet 0]: (8774, 25)
Loaded Multi Station Report _2023.xlsx [sheet 0]: (8774, 25)
Loaded Multi Station Report_2024.xlsx [sheet 0]: (8798, 25)


In [90]:
from IPython.display import display as ipy_display

for name, df in pe_dfs.items():
    print(f"\n=== {name} ===")
    ipy_display(df.head())


=== Multi Station Report _2019 ===


,"Type: Average, Time Base: Hourly, Yearly: 1/1/2019 - 12/31/2019",Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,...,Unnamed: 15,Unnamed: 16,Unnamed: 17,Unnamed: 18,Unnamed: 19,Unnamed: 20,Unnamed: 21,Unnamed: 22,Unnamed: 23,Unnamed: 24
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Date Time,RNEST ESCOLA IPOJUCA,NaN,NaN,NaN,NaN,NaN,RNEST EDCUPE,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,RNEST CPRH,NaN,NaN,NaN,NaN
2,NaN,PM10_1Hr,O3_ug/m3,NO2_ug/m3,NH3_ug/m3,SO2_ug/m3,CO_ppm,PM10_1Hr,NO2_ug/m3,SO2_ug/m3,...,CO_ppm,SO2_ug/m3,O3_ug/m3,NO2_ug/m3,H2S_ug/m3,CO_ppm,PM10,SO2_ug/m3,NO2_ug/m3,O3_ug/m3
3,NaN,ug/m3,ug/m3,ug/m3,ug/m3,ug/m3,ppm,ug/m3,ug/m3,ug/m3,...,ppm,ug/m3,ug/m3,ug/m3,ug/m3,ppm,ug/m3,ug/m3,ug/m3,ug/m3
4,01:00 1/1/2019,25,23.95,10.45,4.7,14.54,0.41,2,0.27,5.14,...,NoData,NoData,NoData,NaN,NaN,NaN,NaN,NaN,NaN,NaN



=== Multi Station Report _2020 ===


,"Type: Average, Time Base: Hourly, Yearly: 1/1/2020 - 12/31/2020",Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,...,Unnamed: 15,Unnamed: 16,Unnamed: 17,Unnamed: 18,Unnamed: 19,Unnamed: 20,Unnamed: 21,Unnamed: 22,Unnamed: 23,Unnamed: 24
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Date Time,RNEST ESCOLA IPOJUCA,NaN,NaN,NaN,NaN,NaN,RNEST EDCUPE,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,RNEST CPRH,NaN,NaN,NaN,NaN
2,NaN,PM10_1Hr,O3_ug/m3,NO2_ug/m3,NH3_ug/m3,SO2_ug/m3,CO_ppm,PM10_1Hr,NO2_ug/m3,SO2_ug/m3,...,CO_ppm,SO2_ug/m3,O3_ug/m3,NO2_ug/m3,H2S_ug/m3,CO_ppm,PM10,SO2_ug/m3,NO2_ug/m3,O3_ug/m3
3,NaN,ug/m3,ug/m3,ug/m3,ug/m3,ug/m3,ppm,ug/m3,ug/m3,ug/m3,...,ppm,ug/m3,ug/m3,ug/m3,ug/m3,ppm,ug/m3,ug/m3,ug/m3,ug/m3
4,01:00 1/1/2020,17,13.24,15.28,7.93,4.16,0.4,NoData,NoData,NoData,...,0.32,5.16,32.63,NoData,4.22,0.24,NoData,NoData,NoData,NoData



=== Multi Station Report _2021 ===


...

,"Type: Average, Time Base: Hourly, Yearly: 1/1/2022 - 12/31/2022",Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,...,Unnamed: 15,Unnamed: 16,Unnamed: 17,Unnamed: 18,Unnamed: 19,Unnamed: 20,Unnamed: 21,Unnamed: 22,Unnamed: 23,Unnamed: 24
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Date Time,RNEST ESCOLA IPOJUCA,NaN,NaN,NaN,NaN,NaN,RNEST EDCUPE,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,RNEST CPRH,NaN,NaN,NaN,NaN
2,NaN,PM10_1Hr,O3_ug/m3,NO2_ug/m3,NH3_ug/m3,SO2_ug/m3,CO_ppm,PM10_1Hr,NO2_ug/m3,SO2_ug/m3,...,CO_ppm,SO2_ug/m3,O3_ug/m3,NO2_ug/m3,H2S_ug/m3,CO_ppm,PM10,SO2_ug/m3,NO2_ug/m3,O3_ug/m3
3,NaN,ug/m3,ug/m3,ug/m3,ug/m3,ug/m3,ppm,ug/m3,ug/m3,ug/m3,...,ppm,ug/m3,ug/m3,ug/m3,ug/m3,ppm,ug/m3,ug/m3,ug/m3,ug/m3
4,01:00 1/1/2022,75.26,34.507,4.873,1.1,1.03,0.383,21.09,2.098,1.799,...,0.099,BelowR,30.842,6.939,BelowR,0,9,3.97,0.817,22.229



=== Multi Station Report _2023 ===


,"Type: Average, Time Base: Hourly, Yearly: 1/1/2023 - 12/31/2023",Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,...,Unnamed: 15,Unnamed: 16,Unnamed: 17,Unnamed: 18,Unnamed: 19,Unnamed: 20,Unnamed: 21,Unnamed: 22,Unnamed: 23,Unnamed: 24
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Date Time,RNEST ESCOLA IPOJUCA,NaN,NaN,NaN,NaN,NaN,RNEST EDCUPE,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,RNEST CPRH,NaN,NaN,NaN,NaN
2,NaN,PM10_1Hr,O3_ug/m3,NO2_ug/m3,NH3_ug/m3,SO2_ug/m3,CO_ppm,PM10_1Hr,NO2_ug/m3,SO2_ug/m3,...,CO_ppm,SO2_ug/m3,O3_ug/m3,NO2_ug/m3,H2S_ug/m3,CO_ppm,PM10,SO2_ug/m3,NO2_ug/m3,O3_ug/m3
3,NaN,ug/m3,ug/m3,ug/m3,ug/m3,ug/m3,ppm,ug/m3,ug/m3,ug/m3,...,ppm,ug/m3,ug/m3,ug/m3,ug/m3,ppm,ug/m3,ug/m3,ug/m3,ug/m3
4,01:00 1/1/2023,12.34,14.437,5.783,0.436,13.797,0.311,6.7,2.935,0.581,...,0.096,0.956,27.068,7.399,1.38,NoData,18.1,3.577,6.019,32.458



=== Multi Station Report_2024 ===


,"Type: Average, Time Base: Hourly, Yearly: 1/1/2024 - 12/31/2024",Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,...,Unnamed: 15,Unnamed: 16,Unnamed: 17,Unnamed: 18,Unnamed: 19,Unnamed: 20,Unnamed: 21,Unnamed: 22,Unnamed: 23,Unnamed: 24
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Date Time,RNEST ESCOLA IPOJUCA,NaN,NaN,NaN,NaN,NaN,RNEST EDCUPE,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,RNEST CPRH,NaN,NaN,NaN,NaN
2,NaN,PM10_1Hr,O3_ug/m3,NO2_ug/m3,NH3_ug/m3,SO2_ug/m3,CO_ppm,PM10_1Hr,NO2_ug/m3,SO2_ug/m3,...,CO_ppm,SO2_ug/m3,O3_ug/m3,NO2_ug/m3,H2S_ug/m3,CO_ppm,PM10,SO2_ug/m3,NO2_ug/m3,O3_ug/m3
3,NaN,ug/m3,ug/m3,ug/m3,ug/m3,ug/m3,ppm,ug/m3,ug/m3,ug/m3,...,ppm,ug/m3,ug/m3,ug/m3,ug/m3,ppm,ug/m3,ug/m3,ug/m3,ug/m3
4,01:00 1/1/2024,NoData,NoData,NoData,NoData,NoData,NoData,27.9,3.643,1.934,...,0.226,2.679,4.656,10.74,1.919,NoData,25,2.545,InVld,38.996


In [91]:
# Limpar dataframes pois contém informações extras nas linhas que não são utilizadas

bad_rows = {
    "Data Precent","Avg","STD","Num",
    "Maximum","Max Date","Max Time",
    "Minimum","Min Date","Min Time",
}

for name, df in pe_dfs.items():
    first_col = df.columns[0]
    mask = df[first_col].astype(str).str.strip().isin(bad_rows)
    pe_dfs[name] = df.loc[~mask].copy()

In [92]:
# Criar um dataframe por df dentro do dict
pe_out = {} 

for name, df in pe_dfs.copy().items():

    # 1) find the header start row
    idx = df.index[df.iloc[:, 0].astype(str).str.strip().eq("Date Time")]
    start = int(idx[0]) if len(idx) else 0
    sub = df.iloc[start:].reset_index(drop=True)
    
    # df is your raw frame
    station = df.iloc[1].ffill()          # row 1, forward-fill across columns
    pollutant = df.iloc[2].astype(str)                # row 2
    
    new_cols = ["DATETIME"] + [f"{st}|{po}" for st, po in zip(station[1:], pollutant[1:])]

    out = sub.iloc[4:].copy()                       # data rows
    out.columns = new_cols

    pe_out[name] = out  

In [93]:
pe_frame = {
    name: fix_datetime_df(d, col="DATETIME", keep_dt_col=False)
    for name, d in pe_out.copy().items()
}

Nota: Nesse caso, as planilhas de estações seguem ordem cronológica de datetime, então o merge precisa conferir se a estação já existia ou se foi criada em um novo ano.

In [65]:
def coalesce_duplicate_columns(df: pd.DataFrame) -> pd.DataFrame:
    """If the same column name appears multiple times, keep one and fill with first non-null."""
    counts = Counter(df.columns)
    dups = [c for c, n in counts.items() if n > 1]
    for c in dups:
        cols = [k for k in df.columns if k == c]
        df[c] = df[cols].bfill(axis=1).iloc[:, 0]  # take first non-null left-to-right
        df.drop(columns=cols[1:], inplace=True)
    return df

def merge_by_datetime_union(dfs_dict: dict, keep_source=True):
    frames = []
    for name, df in dfs_dict.items():
        t = df.copy()
        if keep_source:
            t["__source"] = name  # year or filename

        frames.append(t)

    if not frames:
        return pd.DataFrame()

    out = pd.concat(frames, axis=0, ignore_index=True, sort=True)  # union of columns
    out = coalesce_duplicate_columns(out)

    # keep DATETIME first if present
    if "DATETIME" in out.columns:
        cols = ["DATETIME"] + [c for c in out.columns if c != "DATETIME"]
        out = out[cols]

    return out

In [95]:
pe_all = merge_by_datetime_union(pe_frame.copy(), keep_source=True)
pe_all

,DATETIME,RNEST CPRH|CO_ppm,RNEST CPRH|NO2_ug/m3,RNEST CPRH|O3_ug/m3,RNEST CPRH|PM10,RNEST CPRH|SO2_ug/m3,RNEST EDCUPE|CO_ppm,RNEST EDCUPE|H2S_ug/m3,RNEST EDCUPE|NH3_ug/m3,RNEST EDCUPE|NO2_ug/m3,...,RNEST ESCOLA IPOJUCA|O3_ug/m3,RNEST ESCOLA IPOJUCA|PM10_1Hr,RNEST ESCOLA IPOJUCA|SO2_ug/m3,RNEST IFPE|CO_ppm,RNEST IFPE|H2S_ug/m3,RNEST IFPE|NO2_ug/m3,RNEST IFPE|O3_ug/m3,RNEST IFPE|PM10,RNEST IFPE|SO2_ug/m3,__source
0,01/01/2019 02:00,NaN,NaN,NaN,NaN,NaN,1.92,28.49,0.31,0.27,...,24.22,17,15.58,NoData,NaN,NaN,NoData,NoData,NoData,Multi Station Report _2019
1,01/01/2019 03:00,NaN,NaN,NaN,NaN,NaN,4.79,23.78,0.31,0.29,...,26.02,21,15.1,NoData,NaN,NaN,NoData,NoData,NoData,Multi Station Report _2019
2,01/01/2019 04:00,NaN,NaN,NaN,NaN,NaN,8.99,18.19,0.33,0.3,...,24.89,17,14.95,NoData,NaN,NaN,NoData,NoData,NoData,Multi Station Report _2019
3,01/01/2019 05:00,NaN,NaN,NaN,NaN,NaN,9.99,10.68,0.33,0.35,...,23.13,16,14.69,NoData,NaN,NaN,NoData,NoData,NoData,Multi Station Report _2019
4,01/01/2019 06:00,NaN,NaN,NaN,NaN,NaN,10.54,6.37,0.32,0.41,...,16.94,22,14.65,NoData,NaN,NaN,NoData,NoData,NoData,Multi Station Report _2019
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
52597,31/12/2024 20:00,0.536,5.977,33.563,27.2,2.271,0.23,7.792,1.086,3.497,...,13.115,42.06,5.282,0.07,0.679,Maintain,13.1,26.93,2.124,Multi Station Report_2024
52598,31/12/2024 21:00,0.586,4.141,24.413,32.6,1.626,0.226,7.185,0.777,3.899,...,12.967,34.04,5.09,0.054,1.034,Maintain,13.244,18.33,1.97,Multi Station Report_2024
52599,31/12/2024 22:00,0.583,4.532,18.867,23.9,1.637,0.218,5.131,0.757,3.39,...,13.802,31.75,4.937,0.061,2.259,Maintain,14.253,11.31,1.864,Multi Station Report_2024
52600,31/12/2024 23:00,0.624,5.767,19.587,22.1,1.453,0.246,5.039,0.79,3.582,...,12.195,31.09,4.722,0.051,3.026,Maintain,13.887,10.38,1.927,Multi Station Report_2024


In [66]:
def make_station_layout(df_or_dict, keep_source=True):
  # accept dicts of DataFrames too
    df = (pd.concat(df_or_dict.values(), ignore_index=True, sort=False)
          if isinstance(df_or_dict, dict) else df_or_dict)

    id_vars = ["DATETIME"]
    if keep_source and "__source" in df.columns:
        id_vars.append("__source")

    long = df.melt(id_vars=id_vars, var_name="pair", value_name="VALUE")
    long = long[long["pair"].astype(str).str.contains(r"\|", na=False)].copy()

    # critical line: force string and split on literal "|"
    long["pair"] = long["pair"].astype("string")
    long[["ESTACAO", "POLLUTANT"]] = long["pair"].str.split(
        pat="|", n=1, expand=True, regex=False
    )
    long.drop(columns=["pair"], inplace=True)
    long["ESTACAO"] = long["ESTACAO"].str.strip()
    long["POLLUTANT"] = long["POLLUTANT"].str.strip()

    index_cols = ["DATETIME", "ESTACAO"] + (["__source"] if "__source" in id_vars else [])
    wide = (
        long.pivot_table(index=index_cols, columns="POLLUTANT", values="VALUE", aggfunc="first")
            .reset_index()
            .sort_values(["DATETIME", "ESTACAO"], kind="stable")
    )
    wide.columns.name = None
    return wide

In [97]:
pe_all = make_station_layout(pe_all, keep_source=True)  # keeps "NoData"
pe_all.head()

,DATETIME,ESTACAO,__source,CO_ppm,H2S_ug/m3,NH3_ug/m3,NO2_ug/m3,O3_ug/m3,PM10,PM10_1Hr,SO2_ug/m3
0,01/01/2019 02:00,RNEST EDCUPE,Multi Station Report _2019,1.92,28.49,0.31,0.27,2.83,NaN,18,4.83
1,01/01/2019 02:00,RNEST ESCOLA IPOJUCA,Multi Station Report _2019,0.41,NaN,4.71,11.13,24.22,NaN,17,15.58
2,01/01/2019 02:00,RNEST IFPE,Multi Station Report _2019,NoData,NaN,NaN,NaN,NoData,NoData,NaN,NoData
3,01/01/2019 03:00,RNEST EDCUPE,Multi Station Report _2019,4.79,23.78,0.31,0.29,4.92,NaN,8,5.04
4,01/01/2019 03:00,RNEST ESCOLA IPOJUCA,Multi Station Report _2019,0.35,NaN,4.92,9.91,26.02,NaN,21,15.1


Nota: Nota-se que existem anos que não há medição por poluente, discriminado como NoData. Neste caso, não podemos contabilizar o poluente na estação, por isso deve-se aplicar um filtro.

In [106]:
pe_station = build_inventory_rows(pe_all.copy(), df_cod, uf, datetime_col='DATETIME', mode="columns", drop_after_underscore=True)
pe_station

,UF,ID_OEMA,CIDADE,ID_MMA,ID_MMA_COMPLETO,POLUENTE,COD_POLUENTE,CD_MUN,COD_UF_IBGE,PROPRIETARIO,...,REALOCACAO,OBS_CALIBRACAO,INICIO,FIM,DADOS_MONITORAMENTO,RECONHECIDA,OBS_GERAIS,CERTIFICACAO,STATUS,REP_ESPACIAL_DECLARADA
0,PE,RNEST CPRH,,,,"CO,H2S,MP10,NH3,NO2,O3,SO2",,,26,,...,,,2020,2024,,,,,,
1,PE,RNEST EDCUPE,,,,"CO,H2S,MP10,NH3,NO2,O3,SO2",,,26,,...,,,2019,2024,,,,,,
2,PE,RNEST ESCOLA IPOJUCA,,,,"CO,H2S,MP10,NH3,NO2,O3,SO2",,,26,,...,,,2019,2024,,,,,,
3,PE,RNEST IFPE,,,,"CO,H2S,MP10,NH3,NO2,O3,SO2",,,26,,...,,,2019,2024,,,,,,


In [109]:
pe_station = assign_id_mma(pe_station)

In [108]:
save_UF_estacoes_csv(pe_station, uf)

Saved to: /home/nobre/Notebooks/RQAR_2025_book/data/DADOS_ESTACOES/PE_estacoes.csv


##### e) Roraima

In [140]:
# Substituir pelo estado desejado
uf = "RR" 

In [141]:
# Conferir respostas do formulário 
rr_forms = forms[forms["Unidade da Federação: "].str.contains("RR" , case=False, na=False)]
rr_forms.iloc[:, idxs].head()

,Unidade da Federação:,Em sua UF há estações de monitoramento da qualidade do ar com equipamentos certificados como de referência ou equivalentes? \n,"Em caso afirmativo, a UF é responsável pela operação de algumas dessas estações?",Informe a quantidade de estações de operação própria:,Em sua UF há estações instaladas como condicionante de licença ambiental?,Informe a quantidade de estações instaladas por licença ambiental:,Em sua UF há estações de monitoramento indicativas (com sensores de baixo custo)?,Informe a quantidade de estações indicativas:,"Em 2024, houve mudanças na rede de monitoramento da qualidade do ar na UF?","Em caso afirmativo, indique quais mudanças ocorreram:",Quais são as perspectivas de curto prazo (1 ano) para a rede de monitoramento?,Deseja incluir observações sobre a rede de monitoramento da qualidade do ar? \n,Os dados de monitoramento da qualidade do ar são disponibilizados ao público?,"Em caso afirmativo, informe o endereço eletrônico para acesso:",Os dados de monitoramento passam por algum tratamento e formatação antes de serem disponibilizados?,"Em caso afirmativo, informe o método utilizado: \n(Exemplo: validação automática com base em padrões da CETESB)"
27,Roraima (RR),Não,Não se aplica - não existem estações de referê...,0,Não,0,Sim,2,Não se aplica - não existem estações na UF,Não se aplica – não existem estações na UF,Não há perspectivas de curto prazo,NaN,Não se aplica - não existem estações na UF,NaN,Não se aplica - não existem estações na UF,x


In [142]:
# Importar arquivos do estado
base = Path.cwd().parent 
df_dir = base / "data" / "DADOS_BRUTOS" 
df_dir.mkdir(parents=True, exist_ok=True)
print(df_dir)

rr_dir = df_dir / uf
os.listdir(rr_dir)

/home/nobre/Notebooks/RQAR_2025_book/data/DADOS_BRUTOS


['FEMARH-ENEVA.xlsx', 'Medições (29).xlsx', 'FAZENDA-ENEVA.xlsx']

In [143]:
rr_dfs = load_excels(rr_dir)

Loaded FAZENDA-ENEVA.xlsx [sheet 0]: (6576, 31)
Loaded FEMARH-ENEVA.xlsx [sheet 0]: (7296, 37)
Loaded Medições (29).xlsx [sheet 0]: (30542, 101)


In [144]:
for name, df in rr_dfs.items():
    print(f"\n=== {name} ===")
    display(df.head())


=== FAZENDA-ENEVA ===


,Data,PM25,Status_PM25,PM10,Status_PM10,CO,Status_CO,SO2,Status_SO2,O3,...,DirVento,Status_DirVento,RH,Status_RH,Rain,Status_Rain,Radiação,Status_Radiação,Press_Ar,Status_Press_Ar
0,2024-12-01 00:00:00,0.009,Ok,-9999.0,InVld,0.119,Ok,0.182,Ok,6.25,...,-9999.0,InVld,-9999.0,InVld,-9999.0,InVld,-9999.0,InVld,-9999.0,InVld
1,2024-12-01 01:00:00,7.000,Ok,-9999.0,InVld,0.124,Ok,0.095,Ok,5.93,...,-9999.0,InVld,-9999.0,InVld,-9999.0,InVld,-9999.0,InVld,-9999.0,InVld
2,2024-12-01 02:00:00,6.000,Ok,-9999.0,InVld,0.132,Ok,0.290,Ok,6.02,...,-9999.0,InVld,-9999.0,InVld,-9999.0,InVld,-9999.0,InVld,-9999.0,InVld
3,2024-12-01 03:00:00,17.000,Ok,-9999.0,InVld,0.153,Ok,0.310,Ok,5.57,...,-9999.0,InVld,-9999.0,InVld,-9999.0,InVld,-9999.0,InVld,-9999.0,InVld
4,2024-12-01 04:00:00,22.000,Ok,-9999.0,InVld,0.100,Ok,0.300,Ok,3.82,...,-9999.0,InVld,-9999.0,InVld,-9999.0,InVld,-9999.0,InVld,-9999.0,InVld



=== FEMARH-ENEVA ===


,Data,TempAr,Status_TempAr,VelVento,Status_VelVento,DirVento,Status_DirVento,RH,Status_RH,Rain,...,NMHC,Status_NMHC,CH4,Status_CH4,NO,Status_NO,NO2,Status_NO2,NOX,Status_NOX
0,2024-11-01 00:00:00,28.49,Ok,0.00,Ok,79.78,Ok,82.47,Ok,0.0,...,0.93,Ok,1.21,Ok,-9999.00,NoData,-9999.00,NoData,-9999.00,NoData
1,2024-11-01 01:00:00,28.53,Ok,0.00,Ok,108.00,Ok,85.42,Ok,0.0,...,0.94,Ok,1.25,Ok,2.98,Ok,0.25,Ok,3.23,Ok
2,2024-11-01 02:00:00,28.94,Ok,0.07,Ok,86.32,Ok,77.41,Ok,0.0,...,0.94,Ok,1.18,Ok,2.47,Ok,0.31,Ok,2.78,Ok
3,2024-11-01 03:00:00,28.25,Ok,0.13,Ok,79.69,Ok,79.34,Ok,0.0,...,0.93,Ok,1.17,Ok,2.84,Ok,0.25,Ok,3.09,Ok
4,2024-11-01 04:00:00,27.43,Ok,0.15,Ok,78.89,Ok,82.10,Ok,0.0,...,0.94,Ok,1.16,Ok,2.61,Ok,0.12,Ok,2.73,Ok



=== Medições (29) ===


,Data e Hora,AZULÃO GERAÇÃO DE ENERGIA S.A. - ENEVA S.A. - UTE Jaguatirica II,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,...,Unnamed: 91,Unnamed: 92,Unnamed: 93,Unnamed: 94,Unnamed: 95,Unnamed: 96,Unnamed: 97,Unnamed: 98,Unnamed: 99,Unnamed: 100
0,NaT,Estação Fazenda Carolina,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaT,Qualidade do Ar,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaT,Ar Ambiente,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaT,"Partículas Respiráveis (<2,5µm)",NaN,Monóxido de Carbono,NaN,NaN,NaN,Dióxido de Nitrogênio,NaN,NaN,...,Direção Escalar do Vento,NaN,Umidade Relativa,NaN,Radiação Solar Global,NaN,Pressão Atmosférica,NaN,Precipitação Pluviométrica,NaN
4,NaT,Média:1 h/ Freq.: horária/ Alt.: 4.0 m,NaN,Média:1 h/ Freq.: horária/ Alt.: 3.0 m,NaN,Média:1 h/ Freq.: horária/ Alt.: 3.0 m,NaN,Média:1 h/ Freq.: horária/ Alt.: 3.0 m,NaN,Média:1 h/ Freq.: horária/ Alt.: 3.0 m,...,Média:1 h/ Freq.: horária/ Alt.: 10.0 m,NaN,Média:1 h/ Freq.: horária/ Alt.: 1.0 m,NaN,Média:1 h/ Freq.: horária/ Alt.: 10.0 m,NaN,Média:1 h/ Freq.: horária/ Alt.: 1.0 m,NaN,Média:1 h/ Freq.: horária/ Alt.: 3.0 m,NaN


Nota: Como os 3 dataframes possuem formatos diferentes, é necessário tratá-los conforme suas particularidades

In [145]:
# 1) Fazer alterações somente no df Medições (29) dentro de rr_dfs dict
data = rr_dfs.copy()
key = "Medições (29)"              # exact key in rr_dfs
df = data[key].copy()

# 1) find the header start row
idx = df.index[df.iloc[:, 0].astype(str).str.strip().eq("Data e Hora")]
start = int(idx[0]) if len(idx) else 0
sub = df.iloc[start:].reset_index(drop=True)

# df is your raw frame
station = df.iloc[0].ffill()          # row 1, forward-fill across columns
pollutant = df.iloc[3].astype(str)                # row 2

new_cols = ["DATETIME"] + [f"{st}|{po}" for st, po in zip(station[1:], pollutant[1:])]

out = sub.iloc[7:].copy()                       # data rows
out.columns = new_cols
out["__source"] = key

data[key]=out
out

,DATETIME,"Estação Fazenda Carolina|Partículas Respiráveis (<2,5µm)",Estação Fazenda Carolina|nan,Estação Fazenda Carolina|Monóxido de Carbono,Estação Fazenda Carolina|nan,Estação Fazenda Carolina|nan,Estação Fazenda Carolina|nan,Estação Fazenda Carolina|Dióxido de Nitrogênio,Estação Fazenda Carolina|nan,Estação Fazenda Carolina|nan,...,Estação FEMARH|nan,Estação FEMARH|Umidade Relativa,Estação FEMARH|nan,Estação FEMARH|Radiação Solar Global,Estação FEMARH|nan,Estação FEMARH|Pressão Atmosférica,Estação FEMARH|nan,Estação FEMARH|Precipitação Pluviométrica,Estação FEMARH|nan,__source
7,2021-06-13 19:30:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Medições (29)
8,2021-06-14 16:30:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,IE,113.7,IE,1298,IE,1016.9,IE,0,IE,Medições (29)
9,2021-06-14 17:30:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,IU,69.6,,34.6,,1000,,0,,Medições (29)
10,2021-06-14 18:30:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,IU,69.1,,2.6,,1000.5,,0,,Medições (29)
11,2021-06-14 19:30:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,IU,68.1,,1.7,,1001.4,,0,,Medições (29)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
30537,2024-12-19 11:30:00,5,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Medições (29)
30538,2024-12-19 12:30:00,7,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Medições (29)
30539,2024-12-19 13:30:00,5,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Medições (29)
30540,2024-12-19 14:30:00,2,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Medições (29)


In [146]:
# 2) Fazer alterações nos dfs "FAZENDA-ENEVA" e "FEMARH-ENEVA" dentro de rr_dfs dict
manual_map = {"Data": "DATETIME"}

for key in ["FAZENDA-ENEVA", "FEMARH-ENEVA"]:
    val = data.get(key)
    if val is None:
        continue

    if isinstance(val, dict):
        # rename inside nested dict
        new = {}
        for name, obj in val.items():
            if isinstance(obj, pd.DataFrame):
                new[name] = obj.rename(columns=manual_map)
        data[key] = new

    elif isinstance(val, pd.DataFrame):
        data[key] = val.rename(columns=manual_map)

    elif isinstance(val, pd.Series):
        data[key] = val.rename("DATETIME") if val.name == "Data" else val

In [147]:
for name, df in data.items():
    print(f"\n=== {name} ===")
    display(df.head())


=== FAZENDA-ENEVA ===


,DATETIME,PM25,Status_PM25,PM10,Status_PM10,CO,Status_CO,SO2,Status_SO2,O3,...,DirVento,Status_DirVento,RH,Status_RH,Rain,Status_Rain,Radiação,Status_Radiação,Press_Ar,Status_Press_Ar
0,2024-12-01 00:00:00,0.009,Ok,-9999.0,InVld,0.119,Ok,0.182,Ok,6.25,...,-9999.0,InVld,-9999.0,InVld,-9999.0,InVld,-9999.0,InVld,-9999.0,InVld
1,2024-12-01 01:00:00,7.000,Ok,-9999.0,InVld,0.124,Ok,0.095,Ok,5.93,...,-9999.0,InVld,-9999.0,InVld,-9999.0,InVld,-9999.0,InVld,-9999.0,InVld
2,2024-12-01 02:00:00,6.000,Ok,-9999.0,InVld,0.132,Ok,0.290,Ok,6.02,...,-9999.0,InVld,-9999.0,InVld,-9999.0,InVld,-9999.0,InVld,-9999.0,InVld
3,2024-12-01 03:00:00,17.000,Ok,-9999.0,InVld,0.153,Ok,0.310,Ok,5.57,...,-9999.0,InVld,-9999.0,InVld,-9999.0,InVld,-9999.0,InVld,-9999.0,InVld
4,2024-12-01 04:00:00,22.000,Ok,-9999.0,InVld,0.100,Ok,0.300,Ok,3.82,...,-9999.0,InVld,-9999.0,InVld,-9999.0,InVld,-9999.0,InVld,-9999.0,InVld



=== FEMARH-ENEVA ===


,DATETIME,TempAr,Status_TempAr,VelVento,Status_VelVento,DirVento,Status_DirVento,RH,Status_RH,Rain,...,NMHC,Status_NMHC,CH4,Status_CH4,NO,Status_NO,NO2,Status_NO2,NOX,Status_NOX
0,2024-11-01 00:00:00,28.49,Ok,0.00,Ok,79.78,Ok,82.47,Ok,0.0,...,0.93,Ok,1.21,Ok,-9999.00,NoData,-9999.00,NoData,-9999.00,NoData
1,2024-11-01 01:00:00,28.53,Ok,0.00,Ok,108.00,Ok,85.42,Ok,0.0,...,0.94,Ok,1.25,Ok,2.98,Ok,0.25,Ok,3.23,Ok
2,2024-11-01 02:00:00,28.94,Ok,0.07,Ok,86.32,Ok,77.41,Ok,0.0,...,0.94,Ok,1.18,Ok,2.47,Ok,0.31,Ok,2.78,Ok
3,2024-11-01 03:00:00,28.25,Ok,0.13,Ok,79.69,Ok,79.34,Ok,0.0,...,0.93,Ok,1.17,Ok,2.84,Ok,0.25,Ok,3.09,Ok
4,2024-11-01 04:00:00,27.43,Ok,0.15,Ok,78.89,Ok,82.10,Ok,0.0,...,0.94,Ok,1.16,Ok,2.61,Ok,0.12,Ok,2.73,Ok



=== Medições (29) ===


,DATETIME,"Estação Fazenda Carolina|Partículas Respiráveis (<2,5µm)",Estação Fazenda Carolina|nan,Estação Fazenda Carolina|Monóxido de Carbono,Estação Fazenda Carolina|nan,Estação Fazenda Carolina|nan,Estação Fazenda Carolina|nan,Estação Fazenda Carolina|Dióxido de Nitrogênio,Estação Fazenda Carolina|nan,Estação Fazenda Carolina|nan,...,Estação FEMARH|nan,Estação FEMARH|Umidade Relativa,Estação FEMARH|nan,Estação FEMARH|Radiação Solar Global,Estação FEMARH|nan,Estação FEMARH|Pressão Atmosférica,Estação FEMARH|nan,Estação FEMARH|Precipitação Pluviométrica,Estação FEMARH|nan,__source
7,2021-06-13 19:30:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Medições (29)
8,2021-06-14 16:30:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,IE,113.7,IE,1298,IE,1016.9,IE,0,IE,Medições (29)
9,2021-06-14 17:30:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,IU,69.6,,34.6,,1000,,0,,Medições (29)
10,2021-06-14 18:30:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,IU,69.1,,2.6,,1000.5,,0,,Medições (29)
11,2021-06-14 19:30:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,IU,68.1,,1.7,,1001.4,,0,,Medições (29)


In [148]:
rr_frame = {
    name: fix_datetime_df(d, col="DATETIME", keep_dt_col=False)
    for name, d in data.copy().items()
}

In [149]:
# 1) Fazer alterações somente no df Medições (29) dentro de rr_dfs dict
key = "Medições (29)"              
df = rr_frame[key].copy()

medicoes = make_station_layout(df, keep_source=True)

In [151]:
medicoes

,DATETIME,ESTACAO,__source,Direção Escalar do Vento,Dióxido de Enxofre,Dióxido de Nitrogênio,Frequência Elétrica de Entrada,Hidrocarbonetos Não Metano,Hidrocarbonetos Totais,Metano,...,"Partículas Respiráveis (<2,5µm)",Precipitação Pluviométrica,Pressão Atmosférica,Radiação Solar Global,Temperatura,Tensão Elétrica,Umidade Relativa,Velocidade Escalar do Vento,nan,Óxidos de Nitrogênio
0,01/01/2022 00:30,Estação FEMARH,Medições (29),72,0.00107,0.0001,60,-9999,-9999,-9999,...,17,0,1001.4,0,30,217.9,74.9,2.5,,0.0007
1,01/01/2022 00:30,Estação Fazenda Carolina,Medições (29),72.8,0.00067,0.0004,60,0.44,2.79,2.35,...,18,0,1004.1,0,25.8,NaN,78,2.4,,0.0026
2,01/01/2022 01:30,Estação FEMARH,Medições (29),71.9,0.00109,0,60,-9999,-9999,-9999,...,17,0,1001.4,0,29.4,218.3,76.7,2.3,VU,0.0006
3,01/01/2022 01:30,Estação Fazenda Carolina,Medições (29),73.8,0.00072,0.0004,60,0.44,2.8,2.36,...,15,0,1004.1,0,26,NaN,80.1,1.4,,0.0027
4,01/01/2022 02:30,Estação FEMARH,Medições (29),71.5,0.00108,0,60,-9999,-9999,-9999,...,13,0,1000.9,0,29.1,217.8,78.5,2.4,,0.0007
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
58356,31/12/2023 21:30,Estação Fazenda Carolina,Medições (29),64.5,0.00023,0.0006,60,NaN,NaN,NaN,...,985,0,1004.5,4.3,26.3,NaN,57.6,2.2,IU,0.0018
58357,31/12/2023 22:30,Estação FEMARH,Medições (29),69.1,0.00116,NaN,60,NaN,NaN,NaN,...,9,0,1001.7,0,24.5,215.1,70,1.7,,NaN
58358,31/12/2023 22:30,Estação Fazenda Carolina,Medições (29),56.4,0.00028,0.0007,60,NaN,NaN,NaN,...,985,0,1005.3,4.3,26.2,NaN,59.1,3.6,IU,0.0019
58359,31/12/2023 23:30,Estação FEMARH,Medições (29),69.5,0.00107,NaN,60,NaN,NaN,NaN,...,6,0,1001.5,0,24.9,214.2,73.1,1.8,,NaN


In [160]:
rr_station = build_inventory_rows(medicoes.copy(), df_cod, uf, datetime_col='DATETIME', mode="columns", drop_after_underscore=True)

In [161]:
rr_station = assign_id_mma(rr_station)
rr_station

,UF,ID_OEMA,CIDADE,ID_MMA,ID_MMA_COMPLETO,POLUENTE,COD_POLUENTE,CD_MUN,COD_UF_IBGE,PROPRIETARIO,...,REALOCACAO,OBS_CALIBRACAO,INICIO,FIM,DADOS_MONITORAMENTO,RECONHECIDA,OBS_GERAIS,CERTIFICACAO,STATUS,REP_ESPACIAL_DECLARADA
0,RR,Estação Fazenda Carolina,,RR0001,RR0001,"CH4,CO,HCT,NO,NO2,NOX,O3,SO2",,,14,,...,,,2021,2024,,,,,,
1,RR,Estação FEMARH,,RR0002,RR0002,"CH4,CO,HCT,NO,NO2,NOX,O3,SO2",,,14,,...,,,2021,2024,,,,,,


In [162]:
save_UF_estacoes_csv(rr_station, uf)

Saved to: /home/nobre/Notebooks/RQAR_2025_book/data/DADOS_ESTACOES/RR_estacoes.csv


##### f) Acre

In [164]:
# Substituir pelo estado desejado
uf = "AC" 

In [165]:
# Conferir respostas do formulário 
ac_forms = forms[forms["Unidade da Federação: "].str.contains(uf , case=False, na=False)]
ac_forms.iloc[:, idxs].head()

,Unidade da Federação:,Em sua UF há estações de monitoramento da qualidade do ar com equipamentos certificados como de referência ou equivalentes? \n,"Em caso afirmativo, a UF é responsável pela operação de algumas dessas estações?",Informe a quantidade de estações de operação própria:,Em sua UF há estações instaladas como condicionante de licença ambiental?,Informe a quantidade de estações instaladas por licença ambiental:,Em sua UF há estações de monitoramento indicativas (com sensores de baixo custo)?,Informe a quantidade de estações indicativas:,"Em 2024, houve mudanças na rede de monitoramento da qualidade do ar na UF?","Em caso afirmativo, indique quais mudanças ocorreram:",Quais são as perspectivas de curto prazo (1 ano) para a rede de monitoramento?,Deseja incluir observações sobre a rede de monitoramento da qualidade do ar? \n,Os dados de monitoramento da qualidade do ar são disponibilizados ao público?,"Em caso afirmativo, informe o endereço eletrônico para acesso:",Os dados de monitoramento passam por algum tratamento e formatação antes de serem disponibilizados?,"Em caso afirmativo, informe o método utilizado: \n(Exemplo: validação automática com base em padrões da CETESB)"
7,Acre (AC),Não,Não,0,Não,0,Sim,26,Não,Inativação de estações,Não há perspectivas de curto prazo,NaN,Sim,https://map.purpleair.com/air-quality-raw-pm25...,Sim,Apenas seleção do Sensor que possui dados comp...
29,Acre (AC),Não,Não se aplica - não existem estações de referê...,0,Sim,1,Não,0,Não se aplica - não existem estações na UF,Não se aplica – não existem estações na UF,Não há perspectivas de curto prazo,NaN,Não se aplica - não existem estações na UF,NaN,Não se aplica - não existem estações na UF,X


In [172]:
# Importar arquivos do estado
base = Path.cwd().parent 
df_dir = base / "data" / "DADOS_BRUTOS" 
df_dir.mkdir(parents=True, exist_ok=True)
print(df_dir)

ac_dir = df_dir / uf
os.listdir(ac_dir)

/home/nobre/Notebooks/RQAR_2025_book/data/DADOS_BRUTOS


['.ipynb_checkpoints',
 'AC_MONITOR DE  QUALIDADE DO AR MÉDIA - BANCO DE DADOS - SENSOR PURPLE AIR BAIXO CUSTO - Ylza Lima.xlsx',
 'Erro',
 'AC_PM2.5_sensoresAB_2023 - Ylza Lima.csv']

In [177]:
# Como foram apresentadas particularidades nesse arquivo csv, foi necessário fazer uma alteração na função read,
# para transformar manualmente o tipo de encoding

def _read_csv(path, sep=None, decimal=None, encoding="cp1252", **kw):
    head = path.read_bytes()[:4096]
    txt  = head.decode(encoding or "utf-8", errors="ignore")
    sep_guess = sep or (";" if txt.count(";") > txt.count(",") else ",")
    dec_guess = decimal or ("," if re.search(r"\d+,\d+", txt) and txt.count(",") > txt.count(".") else ".")
    enc_guess = encoding or ("utf-8-sig" if txt.startswith("\ufeff")
                             else ("latin-1" if ("Ã" in txt or "�" in txt) else "utf-8"))
    try:
        return pd.read_csv(path, sep=sep_guess, decimal=dec_guess, encoding=enc_guess, engine="c", **kw)
    except Exception:
        return pd.read_csv(path, sep=None, engine="python", decimal=dec_guess, encoding=enc_guess, **kw)

In [256]:
ac_csv = load_csvs(ac_dir)
for name, df in ac_csv.items():
    print(f"\n=== {name} ===")
    display(df.head())

Loaded AC_PM2.5_sensoresAB_2023 - Ylza Lima.csv: (394, 57)

=== AC_PM2.5_sensoresAB_2023 - Ylza Lima ===


,BD - QUALIDADE DO AR - MÉDIA DIÁRIA PM2.5 (?g/m3) - 2023 (limite OMS: 15 ?g/m3),Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,...,Unnamed: 47,Unnamed: 48,Unnamed: 49,Unnamed: 50,Unnamed: 51,Unnamed: 52,Unnamed: 53,Unnamed: 54,Unnamed: 55,Unnamed: 56
0,NaN,Rio Branco,Rio Branco,Rio Branco,Rio Branco,Rio Branco,Rio Branco,Rio Branco,Rio Branco,Bujari,...,Epitaciolândia,Epitaciolândia,Brasiléia,Brasiléia,Brasiléia,Brasiléia,Santa Rosa do Purus,Santa Rosa do Purus,Porto Walter,Porto Walter
1,NaN,Ministério Público do Estado do Acre (SEDE) A,Ministério Público do Estado do Acre (SEDE) B,UFAC A,UFAC B,RB-BACKUP (Estação particular FB) A,RB-BACKUP (Estação particular FB) B,AcreBioClima - UFAC A,AcreBioClima - UFAC B,MPAC_BJR_01_promotoria A,...,MPAC_EPL_02_escola.joao.pedro A,MPAC_EPL_02_escola.joao.pedro B,MPAC_BRL_02_radio fm 90.3 A,MPAC_BRL_02_radio fm 90.3 B,MPAC_BRL_01_promotoria A,MPAC_BRL_01_promotoria B,MPAC_SRP_01_prefeitura A,MPAC_SRP_01_prefeitura B,MPAC_PTW_01_prefeitura A,MPAC_PTW_01_prefeitura B
2,01/01/2023,NaN,NaN,"1,09","1,15",NaN,NaN,NaN,NaN,"0,00",...,"1,88","2,22",NaN,NaN,"1,34","1,41","2,10","1,81",NaN,NaN
3,02/01/2023,NaN,NaN,"1,43","1,51",NaN,NaN,NaN,NaN,"0,00",...,"3,03","3,36",NaN,NaN,"2,26","2,22","3,25","2,87",NaN,NaN
4,03/01/2023,NaN,NaN,"0,73","0,78",NaN,NaN,NaN,NaN,"0,00",...,"2,09","2,27",NaN,NaN,"1,56","1,49","1,53","1,46",NaN,NaN


In [290]:
ac_long = pd.DataFrame(list(ac_csv.values())[0])
ac_long.head()

,BD - QUALIDADE DO AR - MÉDIA DIÁRIA PM2.5 (?g/m3) - 2023 (limite OMS: 15 ?g/m3),Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,...,Unnamed: 47,Unnamed: 48,Unnamed: 49,Unnamed: 50,Unnamed: 51,Unnamed: 52,Unnamed: 53,Unnamed: 54,Unnamed: 55,Unnamed: 56
0,NaN,Rio Branco,Rio Branco,Rio Branco,Rio Branco,Rio Branco,Rio Branco,Rio Branco,Rio Branco,Bujari,...,Epitaciolândia,Epitaciolândia,Brasiléia,Brasiléia,Brasiléia,Brasiléia,Santa Rosa do Purus,Santa Rosa do Purus,Porto Walter,Porto Walter
1,NaN,Ministério Público do Estado do Acre (SEDE) A,Ministério Público do Estado do Acre (SEDE) B,UFAC A,UFAC B,RB-BACKUP (Estação particular FB) A,RB-BACKUP (Estação particular FB) B,AcreBioClima - UFAC A,AcreBioClima - UFAC B,MPAC_BJR_01_promotoria A,...,MPAC_EPL_02_escola.joao.pedro A,MPAC_EPL_02_escola.joao.pedro B,MPAC_BRL_02_radio fm 90.3 A,MPAC_BRL_02_radio fm 90.3 B,MPAC_BRL_01_promotoria A,MPAC_BRL_01_promotoria B,MPAC_SRP_01_prefeitura A,MPAC_SRP_01_prefeitura B,MPAC_PTW_01_prefeitura A,MPAC_PTW_01_prefeitura B
2,01/01/2023,NaN,NaN,"1,09","1,15",NaN,NaN,NaN,NaN,"0,00",...,"1,88","2,22",NaN,NaN,"1,34","1,41","2,10","1,81",NaN,NaN
3,02/01/2023,NaN,NaN,"1,43","1,51",NaN,NaN,NaN,NaN,"0,00",...,"3,03","3,36",NaN,NaN,"2,26","2,22","3,25","2,87",NaN,NaN
4,03/01/2023,NaN,NaN,"0,73","0,78",NaN,NaN,NaN,NaN,"0,00",...,"2,09","2,27",NaN,NaN,"1,56","1,49","1,53","1,46",NaN,NaN


In [299]:
# Como a planilha de dados possui diversas particularidades, precisa passar por uma limpeza

df = ac_long.copy()

# 1) find the header start row
idx = df.index[(df.iloc[:, 0].isna()) & (df.notna().sum(axis=1) >= 3)]
start = int(idx[0]) if len(idx) else 0
sub = df.iloc[start:].reset_index(drop=True)

# 2) build column names: row 1 has stations from col 1 onward
stations = (
    sub.iloc[1, 1:]           # row with station names, skip first col
       .astype(str).str.strip()
       .tolist()
)

new_cols = ["DATETIME"] + stations 

ac_dfc = sub.iloc[2:].copy() 
ac_dfc.columns = new_cols 
ac_dfc.head()

,DATETIME,Ministério Público do Estado do Acre (SEDE) A,Ministério Público do Estado do Acre (SEDE) B,UFAC A,UFAC B,RB-BACKUP (Estação particular FB) A,RB-BACKUP (Estação particular FB) B,AcreBioClima - UFAC A,AcreBioClima - UFAC B,MPAC_BJR_01_promotoria A,...,MPAC_EPL_02_escola.joao.pedro A,MPAC_EPL_02_escola.joao.pedro B,MPAC_BRL_02_radio fm 90.3 A,MPAC_BRL_02_radio fm 90.3 B,MPAC_BRL_01_promotoria A,MPAC_BRL_01_promotoria B,MPAC_SRP_01_prefeitura A,MPAC_SRP_01_prefeitura B,MPAC_PTW_01_prefeitura A,MPAC_PTW_01_prefeitura B
2,01/01/2023,NaN,NaN,"1,09","1,15",NaN,NaN,NaN,NaN,"0,00",...,"1,88","2,22",NaN,NaN,"1,34","1,41","2,10","1,81",NaN,NaN
3,02/01/2023,NaN,NaN,"1,43","1,51",NaN,NaN,NaN,NaN,"0,00",...,"3,03","3,36",NaN,NaN,"2,26","2,22","3,25","2,87",NaN,NaN
4,03/01/2023,NaN,NaN,"0,73","0,78",NaN,NaN,NaN,NaN,"0,00",...,"2,09","2,27",NaN,NaN,"1,56","1,49","1,53","1,46",NaN,NaN
5,04/01/2023,NaN,NaN,"1,57","1,66",NaN,NaN,NaN,NaN,"0,00",...,"1,21","1,37",NaN,NaN,"1,08","1,05","1,78","1,68",NaN,NaN
6,05/01/2023,NaN,NaN,"0,00","0,00",NaN,NaN,NaN,NaN,"0,00",...,"0,62","1,05",NaN,NaN,"0,00","0,02","0,00","0,00",NaN,NaN


In [300]:
station_cols = ['Ministério Público do Estado do Acre (SEDE) A',
       'Ministério Público do Estado do Acre (SEDE) B', 'UFAC A', 'UFAC B',
       'RB-BACKUP (Estação particular FB) A',
       'RB-BACKUP (Estação particular FB) B', 'AcreBioClima - UFAC A',
       'AcreBioClima - UFAC B', 'MPAC_BJR_01_promotoria A',
       'MPAC_BJR_01_promotoria B', 'MPAC_SNG_01_promotoria A',
       'MPAC_SNG_01_promotoria B', 'MPAC_PTA_01_Sec.infraestrutura A',
       'MPAC_PTA_01_Sec.infraestrutura B', 'MPAC_ACL_01_promotoria A',
       'MPAC_ACL_01_promotoria B', 'MPAC_CPX_01_qpm A', 'MPAC_CPX_01_qpm B',
       'MPAC_XAP_02_promotoria A', 'MPAC_XAP_02_promotoria B',
       'MPAC_ABR_01_promotoria A', 'MPAC_ABR_01_promotoria B',
       'MPAC_ABR_02_SEMSA A', 'MPAC_ABR_02_SEMSA B',
       'MPAC_PLC_01_promotoria A', 'MPAC_PLC_01_promotoria B',
       'MPAC_SNM_01_ifac A', 'MPAC_SNM_01_ifac B', 'MPAC_SNM_02_promotoria A',
       'MPAC_SNM_02_promotoria B', 'MPAC_MNU_01_promotoria A',
       'MPAC_MNU_01_promotoria B', 'MPAC_FIJ_01_promotoria A',
       'MPAC_FIJ_01_promotoria B', 'MPAC_TRC_02_ifac A', 'MPAC_TRC_02_ifac B',
       'MPAC_JRD_01_prefeitura A', 'MPAC_JRD_01_prefeitura B',
       'MPAC_RDA_01_prefeitura A', 'MPAC_RDA_01_prefeitura B',
       'MPAC_CZS_02_ciosp A', 'MPAC_CZS_02_ciosp B', 'UFACFloresta A',
       'UFACFloresta B', 'MPAC_MTH_01_semec A', 'MPAC_MTH_01_semec B',
       'MPAC_EPL_02_escola.joao.pedro A', 'MPAC_EPL_02_escola.joao.pedro B',
       'MPAC_BRL_02_radio fm 90.3 A', 'MPAC_BRL_02_radio fm 90.3 B',
       'MPAC_BRL_01_promotoria A', 'MPAC_BRL_01_promotoria B',
       'MPAC_SRP_01_prefeitura A', 'MPAC_SRP_01_prefeitura B',
       'MPAC_PTW_01_prefeitura A', 'MPAC_PTW_01_prefeitura B']

ac_full = ac_dfc.melt(
    id_vars=['DATETIME'],   # keep these as is
    value_vars=station_cols,                      # columns to unpivot
    var_name='ESTACAO',                           # new col with station names
    value_name='VALOR'                            # new col with numeric values
)
ac_full

,DATETIME,ESTACAO,VALOR
0,01/01/2023,Ministério Público do Estado do Acre (SEDE) A,NaN
1,02/01/2023,Ministério Público do Estado do Acre (SEDE) A,NaN
2,03/01/2023,Ministério Público do Estado do Acre (SEDE) A,NaN
3,04/01/2023,Ministério Público do Estado do Acre (SEDE) A,NaN
4,05/01/2023,Ministério Público do Estado do Acre (SEDE) A,NaN
...,...,...,...
21947,NaN,MPAC_PTW_01_prefeitura B,NaN
21948,NaN,MPAC_PTW_01_prefeitura B,NaN
21949,NaN,MPAC_PTW_01_prefeitura B,NaN
21950,NaN,MPAC_PTW_01_prefeitura B,NaN


In [318]:
ac_station = build_inventory_rows(ac_full.copy(), df_cod, uf, datetime_col='DATETIME', mode="rows", drop_after_underscore=True)
ac_station.head()

,UF,ID_OEMA,CIDADE,ID_MMA,ID_MMA_COMPLETO,POLUENTE,COD_POLUENTE,CD_MUN,COD_UF_IBGE,PROPRIETARIO,...,REALOCACAO,OBS_CALIBRACAO,INICIO,FIM,DADOS_MONITORAMENTO,RECONHECIDA,OBS_GERAIS,CERTIFICACAO,STATUS,REP_ESPACIAL_DECLARADA
0,AC,AcreBioClima - UFAC A,,,,,,,12,,...,,,2023,2023,,,,,,
1,AC,AcreBioClima - UFAC B,,,,,,,12,,...,,,2023,2023,,,,,,
2,AC,MPAC_ABR_01_promotoria A,,,,,,,12,,...,,,2023,2023,,,,,,
3,AC,MPAC_ABR_01_promotoria B,,,,,,,12,,...,,,2023,2023,,,,,,
4,AC,MPAC_ABR_02_SEMSA A,,,,,,,12,,...,,,2023,2023,,,,,,


In [319]:
df = ac_long.copy()

# 1) find the header start row
idx = df.index[(df.iloc[:, 0].isna()) & (df.notna().sum(axis=1) >= 3)]
start = int(idx[0]) if len(idx) else 0
sub = df.iloc[start:].reset_index(drop=True)

# From your cleaned 'sub' with two header rows:
cities   = sub.iloc[0, 1:].astype(str).tolist()      
stations = sub.iloc[1, 1:].astype(str).tolist()     

# 1) station -> city
station_to_city = dict(zip(stations, cities))        

# 2) map onto your dataframe
ac_station["CIDADE"] = ac_station["ID_OEMA"].map(station_to_city)

In [322]:
cols = {
    "POLUENTE": "MP25",
    "CATEGORIA": "Indicativa",
    "MARCA": "PurpleAir",
}

targets = list(cols)

# normalize empty strings and whitespace to NA
ac_station[targets] = ac_station[targets].replace(r"^\s*$", pd.NA, regex=True)

# fill only where missing
for c, v in cols.items():
    ac_station.loc[ac_station[c].isna(), c] = v

In [323]:
ac_station = assign_id_mma(ac_station)
ac_station.head()

,UF,ID_OEMA,CIDADE,ID_MMA,ID_MMA_COMPLETO,POLUENTE,COD_POLUENTE,CD_MUN,COD_UF_IBGE,PROPRIETARIO,...,REALOCACAO,OBS_CALIBRACAO,INICIO,FIM,DADOS_MONITORAMENTO,RECONHECIDA,OBS_GERAIS,CERTIFICACAO,STATUS,REP_ESPACIAL_DECLARADA
0,AC,AcreBioClima - UFAC A,Rio Branco,AC0001,AC0001,MP25,,,12,,...,,,2023,2023,,,,,,
1,AC,AcreBioClima - UFAC B,Rio Branco,AC0002,AC0002,MP25,,,12,,...,,,2023,2023,,,,,,
2,AC,Ministério Público do Estado do Acre (SEDE) A,Rio Branco,AC0003,AC0003,MP25,,,12,,...,,,2023,2023,,,,,,
3,AC,Ministério Público do Estado do Acre (SEDE) B,Rio Branco,AC0004,AC0004,MP25,,,12,,...,,,2023,2023,,,,,,
4,AC,MPAC_ABR_01_promotoria A,Assis Brasil,AC0005,AC0005,MP25,,,12,,...,,,2023,2023,,,,,,


In [324]:
save_UF_estacoes_csv(ac_station, uf)

Saved to: /home/nobre/Notebooks/RQAR_2025_book/data/DADOS_ESTACOES/AC_estacoes.csv


##### g) Mato Grosso do Sul

In [325]:
# Substituir pelo estado desejado
uf = "MS" 

In [326]:
# Conferir respostas do formulário 
ms_forms = forms[forms["Unidade da Federação: "].str.contains(uf, case=False, na=False)]
ms_forms.iloc[:, idxs].head()

,Unidade da Federação:,Em sua UF há estações de monitoramento da qualidade do ar com equipamentos certificados como de referência ou equivalentes? \n,"Em caso afirmativo, a UF é responsável pela operação de algumas dessas estações?",Informe a quantidade de estações de operação própria:,Em sua UF há estações instaladas como condicionante de licença ambiental?,Informe a quantidade de estações instaladas por licença ambiental:,Em sua UF há estações de monitoramento indicativas (com sensores de baixo custo)?,Informe a quantidade de estações indicativas:,"Em 2024, houve mudanças na rede de monitoramento da qualidade do ar na UF?","Em caso afirmativo, indique quais mudanças ocorreram:",Quais são as perspectivas de curto prazo (1 ano) para a rede de monitoramento?,Deseja incluir observações sobre a rede de monitoramento da qualidade do ar? \n,Os dados de monitoramento da qualidade do ar são disponibilizados ao público?,"Em caso afirmativo, informe o endereço eletrônico para acesso:",Os dados de monitoramento passam por algum tratamento e formatação antes de serem disponibilizados?,"Em caso afirmativo, informe o método utilizado: \n(Exemplo: validação automática com base em padrões da CETESB)"
19,Mato Grosso do Sul (MS),Sim,Não,0,Sim,4,Não,0,Não,Não se aplica – não existem estações na UF,Início do monitoramento com estações de baixo ...,Existe proposta de instalação de equipamentos ...,Sim,https://monitorar.mma.gov.br/mapa e https://ww...,Sim,Validação automática e revisão de segurança se...


In [329]:
# Importar arquivos do estado
base = Path.cwd().parent 
df_dir = base / "data" / "DADOS_BRUTOS" 
df_dir.mkdir(parents=True, exist_ok=True)
print(df_dir)

ms_dir = df_dir / uf

/home/nobre/Notebooks/RQAR_2025_book/data/DADOS_BRUTOS


In [335]:
ms_dfs = load_excels(ms_dir)

Loaded MS_2024.xlsx [sheet 0]: (212048, 6)
Loaded MS_Dados RMQAR 2022.xlsx [sheet 0]: (130273, 6)
Loaded MS_monitoramento_2023.xlsx [sheet 0]: (212048, 6)


In [366]:
for name, df in ms_dfs.items():
    print(f"\n=== {name} ===")
    display(df.head())


=== MS_2024 ===


,Estação,Parâmetro Nome,Sigla,Valor Medido,Data,Hora
0,Suzano - Ribas do Rio Pardo,Partículas Totais em Suspensão,PTS,718.0,08/10/2024,18:00
1,Suzano - Ribas do Rio Pardo,Partículas Totais em Suspensão,PTS,717.0,06/07/2024,19:00
2,Suzano - Ribas do Rio Pardo,Partículas Totais em Suspensão,PTS,714.0,02/09/2024,19:00
3,Suzano - Ribas do Rio Pardo,Partículas Totais em Suspensão,PTS,707.0,23/09/2024,18:00
4,Suzano - Ribas do Rio Pardo,Partículas Totais em Suspensão,PTS,705.0,26/07/2024,21:00



=== MS_Dados RMQAR 2022 ===


,Estação,Sigla,Parâmetro Nome,Valor Medido,Data,Hora
0,Eldorado Brasil Celulose S.A.,NaN,Chuva,1.368663,01/01/2022,00:00
1,Eldorado Brasil Celulose S.A.,NaN,Chuva,0.815571,01/01/2022,01:00
2,Eldorado Brasil Celulose S.A.,NaN,Chuva,1.535701,01/01/2022,02:00
3,Eldorado Brasil Celulose S.A.,NaN,Chuva,1.438458,01/01/2022,03:00
4,Eldorado Brasil Celulose S.A.,NaN,Chuva,0.885628,01/01/2022,04:00



=== MS_monitoramento_2023 ===


,Estação,Parâmetro Nome,Sigla,Valor Medido,Data,Hora
0,Suzano - Ribas do Rio Pardo,Partículas Totais em Suspensão,PTS,718.0,08/10/2024,18:00
1,Suzano - Ribas do Rio Pardo,Partículas Totais em Suspensão,PTS,717.0,06/07/2024,19:00
2,Suzano - Ribas do Rio Pardo,Partículas Totais em Suspensão,PTS,714.0,02/09/2024,19:00
3,Suzano - Ribas do Rio Pardo,Partículas Totais em Suspensão,PTS,707.0,23/09/2024,18:00
4,Suzano - Ribas do Rio Pardo,Partículas Totais em Suspensão,PTS,705.0,26/07/2024,21:00


In [370]:
for name, df in ms_dfs.items():
    df["DATETIME"] = pd.to_datetime(df["Data"] + " " + df["Hora"], format="%d/%m/%Y %H:%M")

In [373]:
ms_full = merge_by_datetime_union(ms_dfs, keep_source=True)
ms_full.head()

,DATETIME,Data,Estação,Hora,Parâmetro Nome,Sigla,Valor Medido,__source
0,2024-10-08 18:00:00,08/10/2024,Suzano - Ribas do Rio Pardo,18:00,Partículas Totais em Suspensão,PTS,718.0,MS_2024
1,2024-07-06 19:00:00,06/07/2024,Suzano - Ribas do Rio Pardo,19:00,Partículas Totais em Suspensão,PTS,717.0,MS_2024
2,2024-09-02 19:00:00,02/09/2024,Suzano - Ribas do Rio Pardo,19:00,Partículas Totais em Suspensão,PTS,714.0,MS_2024
3,2024-09-23 18:00:00,23/09/2024,Suzano - Ribas do Rio Pardo,18:00,Partículas Totais em Suspensão,PTS,707.0,MS_2024
4,2024-07-26 21:00:00,26/07/2024,Suzano - Ribas do Rio Pardo,21:00,Partículas Totais em Suspensão,PTS,705.0,MS_2024


In [374]:
manual_map = {"Estação":  "ESTACAO",
              "Sigla":  "POLUENTE"
}

ms_full = ms_full.rename(columns=manual_map) 

In [375]:
ms_full = convert_column_to_datetime(ms_full, column_name="DATETIME", format="%d/%m/%Y %H:%M")
# ms_full = ms_full.sort_values("DATETIME", ascending=False).reset_index(drop=True)
ms_full.head()

,DATETIME,Data,ESTACAO,Hora,Parâmetro Nome,POLUENTE,Valor Medido,__source
0,2024-10-08 18:00:00,08/10/2024,Suzano - Ribas do Rio Pardo,18:00,Partículas Totais em Suspensão,PTS,718.0,MS_2024
1,2024-07-06 19:00:00,06/07/2024,Suzano - Ribas do Rio Pardo,19:00,Partículas Totais em Suspensão,PTS,717.0,MS_2024
2,2024-09-02 19:00:00,02/09/2024,Suzano - Ribas do Rio Pardo,19:00,Partículas Totais em Suspensão,PTS,714.0,MS_2024
3,2024-09-23 18:00:00,23/09/2024,Suzano - Ribas do Rio Pardo,18:00,Partículas Totais em Suspensão,PTS,707.0,MS_2024
4,2024-07-26 21:00:00,26/07/2024,Suzano - Ribas do Rio Pardo,21:00,Partículas Totais em Suspensão,PTS,705.0,MS_2024


In [378]:
ms_station = build_inventory_rows(ms_full.copy(), df_cod, uf, datetime_col='DATETIME', mode="rows")
ms_station = assign_id_mma(ms_station)
ms_station

,UF,ID_OEMA,CIDADE,ID_MMA,ID_MMA_COMPLETO,POLUENTE,COD_POLUENTE,CD_MUN,COD_UF_IBGE,PROPRIETARIO,...,REALOCACAO,OBS_CALIBRACAO,INICIO,FIM,DADOS_MONITORAMENTO,RECONHECIDA,OBS_GERAIS,CERTIFICACAO,STATUS,REP_ESPACIAL_DECLARADA
0,MS,Eldorado Brasil Celulose S.A.,,MS0001,MS0001,"CO,MP10,MP25,NO2,O3,PTS",,,50,,...,,,2022,2024,,,,,,
1,MS,Petrobras - UTE,,MS0002,MS0002,"CO,NO2,O3",,,50,,...,,,2022,2024,,,,,,
2,MS,Suzano TLS1-VCPTL - Três Lagoas,,MS0003,MS0003,"CO,H2S,MP10,NO2,O3,PTS,SO2",,,50,,...,,,2022,2024,,,,,,
3,MS,Suzano - Ribas do Rio Pardo,,MS0004,MS0004,"MP10,MP25,NO2,O3,PTS,SO2",,,50,,...,,,2024,2024,,,,,,


In [382]:
cols = {
    "CATEGORIA": "Referencia",
}

targets = list(cols)
# normalize empty strings and whitespace to NA
ms_station[targets] = ms_station[targets].replace(r"^\s*$", pd.NA, regex=True)
# fill only where missing
for c, v in cols.items():
    ms_station.loc[ms_station[c].isna(), c] = v

In [383]:
save_UF_estacoes_csv(ms_station, uf)

Saved to: /home/nobre/Notebooks/RQAR_2025_book/data/DADOS_ESTACOES/MS_estacoes.csv


##### Bibliotecas

In [158]:
import os, time, math, requests, pandas as pd
from datetime import datetime, timedelta, timezone
import pandas as pd
from collections import defaultdict
import re
import numpy as np
from pathlib import Path
import difflib
import unicodedata
from typing import Callable, Any, Optional
from functools import partial
from pandas.api.types import is_object_dtype, is_string_dtype
from collections import Counter
import unicodedata as ud

##### Dicionários e listas 

In [110]:
# Colunas que precisam conter nas files de dados de ESTAÇÃO para cada estado
mqar_campos = [
        "UF","ID_OEMA","CIDADE","ID_MMA","ID_MMA_COMPLETO","POLUENTE","COD_POLUENTE",
        "CD_MUN","COD_UF_IBGE","PROPRIETARIO","PROP_ENTIDADE","OPERADOR","OP_ENTIDADE",
        "LATITUDE","LONGITUDE","MOBILIDADE","CATEGORIA","FUNCIONAMENTO","METODO",
        "MARCA",'INICIO', 'FIM',"FINALIDADE","MONITORAR","FONTE","CALIBRACAO","REALOCACAO",
        "OBS_CALIBRACAO","DADOS_MONITORAMENTO","RECONHECIDA","OBS_GERAIS",
        "STATUS","CERTIFICACAO","REP_ESPACIAL_DECLARADA"
    ]

In [111]:
name_to_uf = {
    "acre":"AC","alagoas":"AL","amapa":"AP","amazonas":"AM","bahia":"BA","ceara":"CE",
    "distrito federal":"DF","espirito santo":"ES","goias":"GO","maranhao":"MA",
    "mato grosso":"MT","mato grosso do sul":"MS","minas gerais":"MG","para":"PA",
    "paraiba":"PB","parana":"PR","pernambuco":"PE","piaui":"PI","rio de janeiro":"RJ",
    "rio grande do norte":"RN","rio grande do sul":"RS","rondonia":"RO","roraima":"RR",
    "santa catarina":"SC","sao paulo":"SP","sergipe":"SE","tocantins":"TO"
}

In [112]:
UF_TO_IBGE = {
    "AC":12,"AL":27,"AP":16,"AM":13,"BA":29,"CE":23,"DF":53,"ES":32,"GO":52,"MA":21,
    "MT":51,"MS":50,"MG":31,"PA":15,"PB":25,"PR":41,"PE":26,"PI":22,"RJ":33,"RN":24,
    "RS":43,"RO":11,"RR":14,"SC":42,"SP":35,"SE":28,"TO":17
}

def sigla_to_ibge(uf): return UF_TO_IBGE[uf.upper()]

In [113]:
# Colunas que precisam conter nas files de dados de MONITORAMENTO para cada estado
mon_cols = ['DATETIME','ANO','MES','DIA','HORA','UNIDADE','QAQC_INTERNO','VALOR']

In [114]:
# Importar planilha com os códigos de poluentes
base = Path.cwd().parent  
out_dir = base / "data" / "dicionarios" 
out_dir.mkdir(parents=True, exist_ok=True)

df_cod = pd.read_csv(out_dir / 'CODIGO_POLUENTES.csv')

In [115]:
# Importar planilha com as respostas do formulário das UFs
base = Path.cwd().parent  
fr_dir = base / "data" 
fr_dir.mkdir(parents=True, exist_ok=True)

forms = pd.read_csv(fr_dir / '2025_Formulário_Coleta_Respostas_UFs.csv')

#Indice das colunas com respostas sobre rede de monitoramento
# for i, c in enumerate(forms.columns):
#    print(f"[{i}] {c}")

idxs = [6,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22] 

##### Concatenar planilhas de estações por UF e Monitoramento_QAr_BR

In [21]:
base = Path.cwd().parent 
df_dir = base / "data" / "DADOS_ESTACOES" 
df_dir.mkdir(parents=True, exist_ok=True)
ufs_dfs = load_csvs(df_dir, prefix=None, recursive=False, limit=None)

Loaded BA_estacoes.csv: (14, 31)
Loaded CE_estacoes.csv: (4, 34)
Loaded DF_estacoes.csv: (9, 31)
Loaded ES_estacoes.csv: (19, 31)
Loaded MA_estacoes.csv: (6, 31)
Loaded MG_estacoes.csv: (67, 28)
Loaded MT_estacoes.csv: (5, 31)
Loaded PB_estacoes.csv: (3, 34)
Loaded PE_estacoes.csv: (4, 34)
Loaded PR_estacoes.csv: (28, 31)
Loaded RJ_estacoes.csv: (100, 35)
Loaded RS_estacoes.csv: (19, 31)
Loaded SC_estacoes.csv: (4, 31)
Loaded SP_estacoes.csv: (101, 31)


In [22]:
for name, d in ufs_dfs.items(): clean_df_all_text(d, in_place=True)

NameError: name 'clean_df_all_text' is not defined

In [ ]:
uf_frame, uf_conflicts = merge_by_id_multi(
    ufs_dfs.copy(),
    id_cols=['ID_OEMA','POLUENTE'],
    source_priority=priority,
    drop_empty_strings=True,
)
uf_frame

In [ ]:
def explode_pollutants(df, col="POLUENTE"):
    out = df.copy()

    # turn "VOC,PM1, PM25 ,PM10" into ["VOC","PM1","PM25","PM10"]
    out[col] = (
        out[col]
        .astype("string")
        .fillna("")
        .apply(lambda s: [p.strip() for p in s.split(",") if p.strip()])
    )

    # explode to one row per pollutant
    out = out.explode(col, ignore_index=True)

    return out.reset_index(drop=True)

In [ ]:
uf_frame = explode_pollutants(uf_frame, col="POLUENTE")

In [ ]:
base = Path.cwd().parent  
out_dir = base / "data" 
out_dir.mkdir(parents=True, exist_ok=True)

df_mma = pd.read_csv(out_dir / "Monitoramento_QAr_BR.csv")

In [ ]:
clean_df_all_text(df_mma, in_place=True)

In [ ]:
print(len(df_mma['ID_OEMA'].unique()))
print(len(uf_frame))

In [ ]:
def replace_vals(df):
    # colunas alvo, só aplica se existirem
    cols = [c for c in ["CATEGORIA", "FUNCIONAMENTO", "STATUS"] if c in df.columns]

    # regex para strings "vazias" comuns
    null_like = r'^\s*(na|n/a|none|null|nan|nat)?\s*$'

    # 1) normaliza vazios em todas as colunas alvo
    for c in cols:
        df[c] = df[c].replace(null_like, pd.NA, regex=True).fillna("Nao declarado")

    # 2) mapeamentos específicos
    if "CATEGORIA" in df.columns:
        df["CATEGORIA"] = df["CATEGORIA"].replace({
            "N": "Nao declarado",
            "D": "Nao declarado",
            "Naodeclarado": "Nao declarado",
            "CertificadaEPA": "Referencia",
            "Certificada EPA": "Referencia",
            "Equivalente": "Referencia"
        })

    if "FUNCIONAMENTO" in df.columns:
        df["FUNCIONAMENTO"] = df["FUNCIONAMENTO"].replace({
            "N": "Nao declarado",
            "D": "Nao declarado",
            "Autmatica": "Automatica",
            "Automatico": "Automatica"
        })

    if "STATUS" in df.columns:
        df["STATUS"] = df["STATUS"].replace({
            "Sim": "Ativa",
            "sim": "Ativa",
            "Não": "Inativa",
            "Nao": "Inativa",
            "nao": "Inativa",
            "NAO": "Inativa",
        })
        # se após o mapeamento ainda sobrou algo vazio, garante "Nao declarado"
        df["STATUS"] = df["STATUS"].replace(null_like, pd.NA, regex=True).fillna("Nao declarado")

    return df

In [ ]:
df_mma = replace_vals(df_mma)
uf_frame = replace_vals(uf_frame)
dfs_dict = {
    'df_mma': df_mma,
    'uf_frame': uf_frame
}

In [ ]:
df_merged, conflicts = merge_by_id_multi(
    dfs_dict.copy(),
    id_cols=['ID_OEMA','POLUENTE'],
    source_priority=priority,
    drop_empty_strings=True,
)
df_merged

In [ ]:
def _erase(s: str) -> str:
    return "".join(c for c in unicodedata.normalize("NFKD", s) if not unicodedata.combining(c))

def _norm(s: str) -> str:
    s = str(s)
    s = re.sub(r"\(.*?\)|\[.*?\]", "", s).replace("µ", "u")
    s = _erase(s)
    s = re.sub(r"\s+", " ", s).strip().lower()
    return s

def build_cod_map(df_cod, col_sinonimos=None):
    m = {}
    for _, r in df_cod.iterrows():
        m[_norm(r["POLUENTE"])] = r["NOME_PASTA"]
        if col_sinonimos and col_sinonimos in df_cod.columns and pd.notna(r[col_sinonimos]):
            for alt in str(r[col_sinonimos]).split("|"):
                alt = alt.strip()
                if alt:
                    m[_norm(alt)] = r["NOME_PASTA"]
    return m

def normalize_pols_cell(text, cod_map, sep=r"[;,/|]+"):
    if pd.isna(text):
        return ""
    parts = re.split(sep, str(text))
    seen, out = set(), []
    for p in parts:
        k = _norm(p)
        name = cod_map.get(k)
        if name and name not in seen:
            seen.add(name); out.append(name)
    return ",".join(sorted(out))

# uso linha a linha
cod_map = build_cod_map(df_cod)  # faça uma vez
df_merged["POLUENTE"] = df_merged["POLUENTE"].apply(lambda s: normalize_pols_cell(s, cod_map))

In [ ]:
def limpar_ipynb_poluente(df, col="POLUENTE"):
    s = df[col].astype(str)

    # remove tokens contendo ".ipynb_checkpoints"
    s = s.str.replace(r'(?i)(^|,)\s*[^,]*ipynb[_-]?checkpoints[^,]*(?=,|$)', '', regex=True)
    # remove tokens contendo ".ipynb"
    s = s.str.replace(r'(?i)(^|,)\s*[^,]*\.ipynb[^,]*(?=,|$)', '', regex=True)

    # compacta vírgulas e espaços
    s = s.str.replace(r'\s*,\s*', ',', regex=True)
    s = s.str.replace(r',+', ',', regex=True).str.strip(', ')

    # dedup dos itens mantendo a ordem
    def _dedup(v):
        if not v:
            return v
        parts = [p for p in v.split(',') if p]
        seen = set()
        out = []
        for p in parts:
            if p not in seen:
                seen.add(p)
                out.append(p)
        return ",".join(out)

    df[col] = s.map(_dedup)
    return df

In [ ]:
limpar_ipynb_poluente(df_merged, col="POLUENTE")

In [ ]:
df_merged = explode_pollutants(df_merged, col="POLUENTE")

In [ ]:
# def replace_POL(df):
#     # colunas alvo, só aplica se existirem
#     cols = [c for c in ["POLUENTE"] if c in df.columns]

#     # regex para strings "vazias" comuns
#     null_like = r'^\s*(na|n/a|none|null|nan|nat)?\s*$'

#     # 2) mapeamentos específicos
#     if "POLUENTE" in df.columns:
#         df["POLUENTE"] = df["POLUENTE"].replace({
#             "PM1": "MP1",
#             "PM1O": "MP10",
#             "5": "O3",
#             "5 E O3": "O3",
#             "ETIL E ORTO": "ETIL",
#             "HCNM E ODORES": "HCNM",
#             "MP10 E MP2": "PTS",
#             "MP10": "MP10",
#             "MP2.5": "MP25",
#             "NO E NOX": "NOX",
#             "NOX E ODORES": "NOX",
#             "PTS E MP10": "PTS",
#             "NOx e Odores": "NOX",
#             "nan": "Nao declarado",
#             "HCnM e Odores": "HCNM",
#             "nan": "Nao declarado"
#         })

#     return df 

In [ ]:
def _to_ascii_upper(s: str) -> str:
    # remove acentos e normaliza subscrito/sobrescrito para dígitos
    subs = str.maketrans("₀₁₂₃₄₅₆₇₈₉", "0123456789")
    sups = str.maketrans("⁰¹²³⁴⁵⁶⁷⁸⁹", "0123456789")
    s = unicodedata.normalize("NFKD", str(s))
    s = "".join(c for c in s if not unicodedata.combining(c))
    s = s.translate(subs).translate(sups).upper().strip()
    s = re.sub(r"\s+", " ", s)
    return s

def _map_token(t: str) -> list[str]:
    """Mapeia um token para 0..n nomes padronizados."""
    t0 = _to_ascii_upper(t)

    # descartes
    if t0 in {"", "NAN", "NA", "NONE", "NULL", "ODOR", "ODORES"}:
        return []

    # PM/MP normalização
    t1 = t0.replace("PM", "MP")  # PM10 -> MP10, PM2.5 -> MP2.5
    t1 = t1.replace("MP1O", "MP10")  # O no lugar de zero

    # MP2.5 variações -> MP25
    if re.fullmatch(r"MP2([.,/]?5)?|MP25|PM2([.,/]?5)?", t0):
        return ["MP25"]

    if t1 in {"MP10"}:
        return ["MP10"]
    if t1 in {"MP1"}:
        return ["MP1"]

    # abreviações comuns
    direct = {
        "NOX": "NOX",
        "NO": "NO",
        "NO2": "NO2",
        "O3": "O3",
        "SO2": "SO2",
        "CO": "CO",
        "CH4": "CH4",
        "PTS": "PTS",
        "VOC": "VOC",
        "HCT": "HCT",
        "HCNM": "HCNM",
        "ERT": "ERT",
        "FMC": "FMC",
        "H2S": "H2S",
        "BENZENO": "BENZENO",
        "TOLUENO": "TOLUENO",
        "ETILBENZENO": "ETILBENZENO",
        "XILENO": "XILENO",
        "OXILENO": "OXILENO",
        "MPXILENO": "MPXILENO",
        "ACETAL": "ACETAL",
        "FORMAL": "FORMAL",
        "CH2O": "FORMAL",   # formaldeído
        "BEN": "BENZENO",
        "BENZ": "BENZENO",
        "TOL": "TOLUENO",
        "ETBEN": "ETILBENZENO",
        "ETILBEN": "ETILBENZENO",
        "MPXIL": "MPXILENO",
        "OXIL": "OXILENO",
        "XIL": "XILENO",
    }
    if t1 in direct:
        return [direct[t1]]

    # caso especial “ETIL,ORTO” já será quebrado pelo split; mas se vier colado:
    if t1 in {"ETIL,ORTO", "ETIL ORTO"}:
        return ["ETILBENZENO", "OXILENO"]

    # fallback: mantemos o token bruto normalizado
    return [t1]

def normalizar_coluna_poluente(df: pd.DataFrame, df_cod: pd.DataFrame, col="POLUENTE") -> pd.DataFrame:
    if col not in df.columns:
        return df.copy()

    # vocabulário permitido
    allowed = set(df_cod["NOME_PASTA"].astype(str).str.upper().str.strip())

    # separadores: vírgula, ;, /, |, e “ e ”
    sep_re = re.compile(r"\s*(?:,|;|/|\||\se\s)\s*", flags=re.I)

    def process_cell(val) -> str:
        if pd.isna(val):
            return ""
        tokens = []
        seen = set()
        # quebra
        parts = [p for p in sep_re.split(str(val)) if p.strip()]
        for p in parts:
            mapped = _map_token(p)
            for m in mapped:
                if m in allowed and m not in seen:
                    seen.add(m)
                    tokens.append(m)
        return ",".join(tokens)

    out = df.copy()
    out[col] = out[col].apply(process_cell)
    return out

def listar_tokens_restantes(df: pd.DataFrame, col="POLUENTE") -> list[str]:
    parts = df[col].astype(str).str.split(",").explode().dropna().str.strip()
    return sorted([t for t in parts.unique() if t])

# -------------------- uso --------------------
# df_cod precisa ter a coluna NOME_PASTA com os nomes oficiais
# df_merged é o seu DF com a coluna POLUENTE
df_merged = normalizar_coluna_poluente(df_merged, df_cod, col="POLUENTE")
print(listar_tokens_restantes(df_merged, "POLUENTE"))

In [ ]:
base = Path.cwd().parent  # .../RQAR_2025_book
out_dir = base / "data" 
out_dir.mkdir(parents=True, exist_ok=True)

out_file = out_dir / "Monitoramento_QAr_BR.csv"
df_merged.to_csv(out_file, index=False, encoding="utf-8")
print("Saved to:", out_file.resolve())

##### Ler e abrir arquivos CSV

In [116]:
def _read_csv(path, sep=None, decimal=None, encoding=None, **kw):
    head = path.read_bytes()[:4096]
    txt  = head.decode(encoding or "utf-8", errors="ignore")
    sep_guess = sep or (";" if txt.count(";") > txt.count(",") else ",")
    dec_guess = decimal or ("," if re.search(r"\d+,\d+", txt) and txt.count(",") > txt.count(".") else ".")
    enc_guess = encoding or ("utf-8-sig" if txt.startswith("\ufeff")
                             else ("latin-1" if ("Ã" in txt or "�" in txt) else "utf-8"))
    try:
        return pd.read_csv(path, sep=sep_guess, decimal=dec_guess, encoding=enc_guess, engine="c", **kw)
    except Exception:
        return pd.read_csv(path, sep=None, engine="python", decimal=dec_guess, encoding=enc_guess, **kw)

def load_csvs(dir_path, prefix=None, recursive=False, limit=None, **read_csv_kwargs):
    pattern = "**/*.csv" if recursive else "*.csv"
    files = sorted(dir_path.glob(pattern))
    if prefix:
        p = prefix.upper()
        files = [f for f in files if f.name.upper().startswith(p)]
    if limit:
        files = files[:int(limit)]
    dfs = {}
    for f in files:
        df = _read_csv(f, **read_csv_kwargs)
        key = f.stem
        # avoid key clashes
        if key in dfs:
            key = f"{f.stem}__{len(dfs)}"
        dfs[key] = df
        print(f"Loaded {f.name}: {df.shape}")
    return dfs

##### Ler e abrir arquivos TXT

In [117]:
def _read_txt(path, sep=None, decimal=None, encoding=None, header=None, **kw):
    head = path.read_bytes()[:4096]
    txt  = head.decode(encoding or "utf-8", errors="ignore")

    enc_guess = encoding or ("utf-8-sig" if txt.startswith("\ufeff")
                             else ("latin-1" if ("Ã" in txt or "�" in txt) else "utf-8"))
    dec_guess = decimal or ("," if re.search(r"\d+,\d+", txt) and txt.count(",") > txt.count(".") else ".")

    if sep is not None:
        sep_guess = sep
    else:
        candidates = ["\t", ";", "|", ","]
        counts = {d: txt.count(d) for d in candidates}
        sep_guess = max(counts, key=counts.get) if max(counts.values()) >= 2 else None

    try:
        if sep_guess is None:
            return pd.read_csv(path, sep=None, engine="python",
                               decimal=dec_guess, encoding=enc_guess,
                               header=header, **kw)
        else:
            return pd.read_csv(path, sep=sep_guess, engine="c",
                               decimal=dec_guess, encoding=enc_guess,
                               header=header, **kw)
    except Exception:
        full = path.read_text(encoding=enc_guess, errors="ignore")
        return pd.DataFrame({"text": full.splitlines()})

def load_txts(dir_path, prefix=None, recursive=False, limit=None, **read_txt_kwargs):
    pattern = "**/*.txt" if recursive else "*.txt"
    files = sorted(dir_path.glob(pattern))
    if prefix:
        p = prefix.upper()
        files = [f for f in files if f.name.upper().startswith(p)]
    if limit:
        files = files[:int(limit)]
    dfs = {}
    for f in files:
        df = _read_txt(f, **read_txt_kwargs)
        key = f.stem
        if key in dfs:
            key = f"{f.stem}__{len(dfs)}"
        dfs[key] = df
        print(f"Loaded {f.name}: {df.shape}")
    return dfs

##### Ler e abrir arquivos XLSX

In [333]:
EXCEL_EXTS = {".xlsx", ".xls", ".xlsm", ".xltx", ".xltm"}

def _excel_engine(ext: str) -> str | None:
    """
    Retorna o engine recomendado para cada extensão.
    .xlsx, .xlsm, .xltx, .xltm -> openpyxl
    .xls -> xlrd (necessita instalar xlrd)
    """
    ext = ext.lower()
    if ext in {".xlsx", ".xlsm", ".xltx", ".xltm"}:
        return "openpyxl"
    if ext == ".xls":
        return "xlrd"  
    return None

In [334]:
def load_excels(dir_path, sheets=0, prefix=None, recursive=False, limit=None, **read_excel_kwargs):
    """
    Lê arquivos Excel de uma pasta e retorna um dicionário {chave: DataFrame}.

    Parâmetros:
      - dir_path: str | Path
        Caminho da pasta onde estão os arquivos.
      - sheets: int | str | list | "all"
        Qual planilha carregar.
        0 carrega a primeira planilha. Pode ser índice (int) ou nome (str).
        Lista para múltiplas planilhas. "all" para todas as planilhas.
      - prefix: str | None
        Se definido, carrega apenas arquivos cujo nome começa com este prefixo.
      - recursive: bool
        Se True, busca nas subpastas.
      - limit: int | None
        Limita a quantidade de arquivos lidos.
      - **read_excel_kwargs:
        Parâmetros extras repassados na função

    Retorno:
      - Dicionário com chave nome base do arquivo
    """
    dir_path = Path(dir_path)
    files = []
    if recursive:
        for ext in EXCEL_EXTS:
            files += list(dir_path.rglob(f"*{ext}"))
    else:
        for ext in EXCEL_EXTS:
            files += list(dir_path.glob(f"*{ext}"))
    files = sorted(files)

    if prefix:
        p = prefix.upper()
        files = [f for f in files if f.name.upper().startswith(p)]
    if limit:
        files = files[:int(limit)]

    dfs = {}
    for f in files:
        eng = _excel_engine(f.suffix)
        try:
            with pd.ExcelFile(f, engine=eng) as xls:
                if sheets == "all":
                    wanted = xls.sheet_names
                elif isinstance(sheets, (list, tuple)):
                    wanted = sheets
                else:
                    wanted = [sheets]

                for s in wanted:
                    df = pd.read_excel(xls, sheet_name=s, **read_excel_kwargs)
                    if sheets == "all" or isinstance(sheets, (list, tuple)):
                        sheet_label = s if isinstance(s, str) else xls.sheet_names[s]
                        key = f"{f.stem}__{sheet_label}"
                    else:
                        key = f.stem
                    if key in dfs:
                        i = 1
                        new_key = f"{key}__{i}"
                        while new_key in dfs:
                            i += 1
                            new_key = f"{key}__{i}"
                        key = new_key

                    dfs[key] = df
                    print(f"Loaded {f.name} [sheet {s}]: {df.shape}")
        except Exception as e:
            print(f"Skip {f.name} due to read error: {e}")
            continue
    return dfs

##### Remove e substitui caracteres específicos 

In [120]:
CLEAN_TABLE = {
    ord("ç"): "c",
    ord("Ç"): "C",
    ord("´"): None,
    ord("~"): None,
    ord("ˆ"): None,
    ord("°"): None,
    ord("`"): None,
    0x0302: None,  # combining ^
}

In [121]:
def clean_text_full(s: object) -> object:
    if pd.isna(s):
        return s
    t = str(s)
    # sua tabela primeiro
    t = t.translate(CLEAN_TABLE)
    # remover acentos de letras precompostas (ex.: ã, á, í)
    t = unicodedata.normalize("NFKD", t)
    t = "".join(ch for ch in t if unicodedata.category(ch) != "Mn")
    return t

def clean_df_all_text(df: pd.DataFrame, cols=None, in_place=False) -> pd.DataFrame:
    out = df if in_place else df.copy()
    if cols is None:
        cols = [c for c in out.columns
                if is_object_dtype(out[c].dtype) or is_string_dtype(out[c].dtype)
                   or isinstance(out[c].dtype, pd.CategoricalDtype)]
    for c in cols:
        s = out[c]
        was_cat = isinstance(s.dtype, pd.CategoricalDtype)
        s2 = s.astype("string").map(lambda v: clean_text_full(v) if pd.notna(v) else v)
        out[c] = s2.astype("category") if was_cat else s2
    return out

##### Transformar coluna Datetime

In [122]:
def convert_column_to_datetime(df, column_name, format=None):
    out = df.copy()

    if (out.index.name or "").upper() == column_name.upper():
        if column_name in out.columns:
            out.reset_index(drop=True, inplace=True)
        else:
            out.reset_index(inplace=True)

    if column_name not in out.columns:
        if pd.api.types.is_datetime64_any_dtype(out.index):
            out[column_name] = out.index
            if not keep_index:
                out.reset_index(drop=True, inplace=True)
        else:
            raise ValueError(f"Column or datetime index '{column_name}' not found.")

    out[column_name] = pd.to_datetime(out[column_name], format=format,  errors="coerce")

    # if out[column_name].notna().any():
    #     out["INICIO"] = int(out[column_name].min().year)
    #     out["FIM"] = int(out[column_name].max().year)
    # else:
    #     out["INICIO"] = None
    #     out["FIM"] = None

    return out

##### Inspecionar colunas de mesmo nome dentro de um dicionário

In [123]:
def _flatten_columns(df):
    """
    Achata colunas MultiIndex em strings simples.
    Exemplo: ("A","B") vira "A | B".
    """
    cols = df.columns
    if isinstance(cols, pd.MultiIndex):
        return [" | ".join([str(x) for x in tup if pd.notna(x)]) for tup in cols]
    return [str(c) for c in cols]

In [124]:
def _normalize(names, lower=True, strip=True):
    """
    Normaliza nomes de colunas para comparação justa.
    - lower: converte para minúsculas
    - strip: remove espaços extras
    """
    out = []
    for n in names:
        s = str(n)
        if strip:
            s = s.strip()
        if lower:
            s = s.lower()
        out.append(s)
    return out

In [125]:
def summarize_columns(dfs: dict, normalize=True):
    """
    Resume colunas presentes em um dicionário {nome_df: DataFrame}.

    Parâmetros
    - dfs: dict[str, pd.DataFrame]
      Dicionário onde a chave é o nome e o valor é um DataFrame.
    - normalize: bool
      Se True, compara usando nomes normalizados (minúsculas e trim).

    Retorno
    - summary_df: pd.DataFrame
      Tabela com uma linha por coluna distinta:
        col            nome da coluna considerada na comparação
        n_present      em quantos DataFrames ela aparece
        n_missing      em quantos não aparece
        is_common      True se aparece em todos
        is_unique      True se aparece em apenas um
        present_in     lista de DataFrames onde aparece
        missing_in     lista de DataFrames onde não aparece
        examples       exemplos de rótulos originais vistos para esta coluna
    - per_df_stats: pd.DataFrame
      Uma linha por DataFrame com:
        n_cols                 quantidade de colunas
        n_common_present       quantas colunas comuns ele possui
        n_common_missing       quantas colunas comuns faltam nele
        extras_count           colunas que só existem em alguns e estão nele
        extras                 lista dessas colunas extras
    """
    # 1) coletar nomes por df
    name_to_cols = {}
    name_to_rawmap = {}
    for name, df in dfs.items():
        cols_raw = _flatten_columns(df)
        cols_cmp = _normalize(cols_raw) if normalize else cols_raw
        name_to_cols[name] = set(cols_cmp)
        # mapeia versão normalizada para exemplos originais
        rawmap = {}
        for raw, cmp in zip(cols_raw, cols_cmp):
            rawmap.setdefault(cmp, set()).add(raw)
        name_to_rawmap[name] = rawmap

    all_cols = set().union(*name_to_cols.values()) if name_to_cols else set()
    df_names = list(name_to_cols.keys())
    n_dfs = len(df_names)

    # 2) frequências por coluna
    records = []
    for col in sorted(all_cols):
        present_in = [n for n in df_names if col in name_to_cols[n]]
        missing_in = [n for n in df_names if col not in name_to_cols[n]]
        # exemplos de rótulos originais
        examples = sorted(set().union(*[name_to_rawmap[n].get(col, set()) for n in df_names]))
        records.append({
            "col": col,
            "n_present": len(present_in),
            "n_missing": n_dfs - len(present_in),
            "is_common": len(present_in) == n_dfs,
            "is_unique": len(present_in) == 1,
            "present_in": present_in,
            "missing_in": missing_in,
            "examples": examples[:5],  # mostra até 5 exemplos
        })
    summary_df = pd.DataFrame(records).sort_values(
        ["is_common", "n_present", "col"], ascending=[False, False, True]
    ).reset_index(drop=True)

    # 3) colunas comuns e extras
    common_set = set(summary_df.loc[summary_df["is_common"], "col"])
    per_df_rows = []
    for name in df_names:
        cols_set = name_to_cols[name]
        n_cols = len(cols_set)
        n_common_present = len(cols_set & common_set)
        n_common_missing = len(common_set - cols_set)
        extras = sorted(c for c in cols_set if c not in common_set)
        per_df_rows.append({
            "df": name,
            "n_cols": n_cols,
            "n_common_present": n_common_present,
            "n_common_missing": n_common_missing,
            "extras_count": len(extras),
            "extras": extras
        })
    per_df_stats = pd.DataFrame(per_df_rows).sort_values("df").reset_index(drop=True)

    return summary_df, per_df_stats

##### Renomear colunas - substituir campos conforme desejado

In [126]:
# manual_map = {"fecha":  "DATETIME",
#              "Date":  "DATETIME",
#              "Fecha":  "DATETIME",
#}

##### Selecionar estações únicas no conjunto de dados

Nota: Quando há junção de mais de uma base de dados de fontes diferentes, utilizar função para manter estações não repetidas e garantir que a maioria das colunas com dados seja mantida.

In [386]:
56+14+4+9+19+6+67+4+5+3+4+28+100+2+19+4+101
445-422

23

In [127]:
def merge_by_id_multi(
    dfs_dict: dict,
    id_cols,              # pode ser "ID_OEMA" ou ["ID_OEMA","UF"]
    source_col="__source",
    source_priority: list[str] | None = None,
    drop_empty_strings: bool = True,
):
    """
    Une vários DataFrames (em um dict) e mantém 1 linha por chave composta.

    Parâmetros:
      - dfs_dict: dict[str, pd.DataFrame]  dicionário nome->DataFrame.
      - id_cols: str | list/tuple          coluna(s) que formam a chave. Ex.: "ID_OEMA" ou ["ID_OEMA","UF"].
      - source_col: str                    coluna auxiliar com o nome da fonte.
      - source_priority: list[str] | None  prioridade entre fontes para resolver conflitos.
      - drop_empty_strings: bool           converte strings vazias em NaN antes de mesclar.

    Retorna:
      - df_merged: DataFrame final com 1 linha por chave.
      - df_conflicts: DataFrame listando conflitos por coluna e chave.
    """
    if isinstance(id_cols, str):
        id_cols = (id_cols,)
    id_set = set(id_cols)

    # 1) concatena e marca fonte
    frames = []
    for name, df in dfs_dict.items():
        if isinstance(df, pd.DataFrame):
            tmp = df.copy()
            tmp[source_col] = name
            frames.append(tmp)
    if not frames:
        return pd.DataFrame(), pd.DataFrame()
    big = pd.concat(frames, ignore_index=True, sort=False)

    # garante que todas as colunas de chave existam
    for c in id_cols:
        if c not in big.columns:
            big[c] = pd.NA

    # 2) normalização leve
    if drop_empty_strings:
        for c in big.columns:
            if big[c].dtype.kind in "OUS":
                big[c] = big[c].astype("string").str.strip().replace({"": pd.NA})

    def pick_value(series, sources):
        mask = series.notna()
        vals = series[mask].astype(object).tolist()
        srcs = sources[mask].astype(str).tolist()
        if not vals:
            return pd.NA, None, []
        if len(set(map(str, vals))) == 1:
            return vals[0], srcs[0], list(dict.fromkeys(srcs))
        if source_priority:
            pri = {s: i for i, s in enumerate(source_priority)}
            best = min(range(len(srcs)), key=lambda i: pri.get(srcs[i], 10**9))
            return vals[best], srcs[best], list(dict.fromkeys(srcs))
        return vals[0], srcs[0], list(dict.fromkeys(srcs))

    # 3) reduz por chave composta
    data_cols = [c for c in big.columns if c not in id_set | {source_col}]
    rows, conflicts = [], []
    for key_vals, g in big.groupby(list(id_cols), dropna=True, sort=False):
        if len(id_cols) == 1:
            key_vals = (key_vals,)
        out = {c: v for c, v in zip(id_cols, key_vals)}

        for c in data_cols:
            s = g[c] if c in g.columns else pd.Series([pd.NA] * len(g))
            winner, winner_src, all_srcs = pick_value(s, g[source_col])
            out[c] = winner

            non_null = [str(v) for v in s.dropna().tolist()]
            if len(set(non_null)) > 1:
                confl = {col: val for col, val in zip(id_cols, key_vals)}
                confl.update({
                    "col": c,
                    "values": sorted(set(non_null)),
                    "sources": all_srcs,
                    "chosen": str(winner),
                    "chosen_source": winner_src,
                })
                conflicts.append(confl)

        rows.append(out)

    df_merged = pd.DataFrame(rows)
    ordered = list(id_cols) + [c for c in df_merged.columns if c not in id_set]
    df_merged = df_merged[ordered]

    df_conflicts = pd.DataFrame(conflicts)
    return df_merged, df_conflicts

##### Unir estações quando dentro de um dict - Múltiplas planilhas

Nota: Maneira mais simplificada de unir diferentes dfs dentro de um dict, quando sabemos que não existem linhas duplicadas ou informações repetidas. Quando há dúvida, rodar a função anterior.

In [128]:
def merge_station_dfs(dfs_dict, uf):
    frames = []
    for station_name, d in dfs_dict.items():
        df = d.copy()

        # station
        df["ESTACAO"] = station_name # LEMBRAR DE DAR UM DROP
        frames.append(df)

    if not frames:
        raise ValueError("No data frames to merge.")
    return pd.concat(frames, ignore_index=True, sort=False)

##### Corrigir e identificar problemas com Datetime column

In [129]:
TARGET_FMT = "%d/%m/%Y %H:%M"

def fix_datetime_df(df, col, keep_dt_col=True):
    out = df.copy()
    out.rename(columns=lambda c: str(c).strip().lstrip("\ufeff"), inplace=True)

    s = (out[col].astype(str)
                  .str.strip()
                  .str.replace("\xa0", " ", regex=False)   # nbsp
                  .str.replace("T", " ", regex=False)
                  .str.replace("Z", "", regex=False))

    dt = pd.Series(pd.NaT, index=out.index, dtype="datetime64[ns]")

    # put your actual input formats first
    tries = [
        ("%H:%M %m/%d/%Y", False),       # 02:00 1/1/2019
        ("%H:%M:%S %m/%d/%Y", False),    # 02:00:00 1/1/2019
        ("%d/%m/%Y %H:%M", True),
        ("%d/%m/%Y %H:%M:%S", True),
        ("%Y-%m-%d %H:%M", False),
        ("%Y-%m-%d %H:%M:%S", False),
        ("%d-%b-%Y %H:%M", True),
        ("%d-%b-%Y %H:%M:%S", True),
    ]

    for fmt, dayfirst in tries:
        m = dt.isna()
        if not m.any():
            break
        dt.loc[m] = pd.to_datetime(s[m], format=fmt, errors="coerce", dayfirst=dayfirst)

    # final fallback
    m = dt.isna()
    if m.any():
        try:
            dt.loc[m] = pd.to_datetime(s[m], format="mixed", dayfirst=True, errors="coerce")
        except TypeError:
            dt.loc[m] = pd.to_datetime(s[m], dayfirst=True, errors="coerce")

    if keep_dt_col:
        out[f"{col}_DT"] = dt

    out[col] = np.where(dt.notna(), dt.dt.strftime(TARGET_FMT), np.nan)
    return out

In [130]:
# Diagnóstico das colunas ou linhas com problema
def diag_bad_stations(df_all, col="DATETIME", top=5):
    out = []
    for st, g in df_all.groupby("ESTACAO"):
        rate = g[col].notna().mean()
        if rate < 0.99:
            bad = g.loc[g[col].isna(), col].astype(str).head(top).tolist()
            out.append((st, rate, bad))
    for st, rate, examples in out:
        print(f"{st}: parsed {rate:.1%}  examples not parsed -> {examples}")

##### Criar início e fim das medições por estação

In [131]:
# def station_year_bounds(g, datetime_col: str | None):
#     if not datetime_col or datetime_col not in g.columns:
#         return None, None
#     ts = g[datetime_col].dropna()
#     ini = int(ts.min().year) if not ts.empty else None
#     fim = int(ts.max().year) if not ts.empty else None
#     return ini, fim

def station_year_bounds(g: pd.DataFrame, datetime_col: Optional[str]):
    if not datetime_col or datetime_col not in g.columns:
        return pd.NA, pd.NA
    s = pd.to_datetime(g[datetime_col], dayfirst=True)
    if s.notna().any():
        return int(s.min().year), int(s.max().year)
    return pd.NA, pd.NA

##### Criar coluna de poluentes mapeando os nomes

In [132]:
# Função para mapear e converter apenas os nomes dos poluentes quando no CABEÇALHO DAS COLUNAS
def id_pol(df, df_cod, column_name, drop_after_underscore=True):
    nomes = []
    for col in df.columns:
        if col == column_name:
            continue  
        
        # limpa parênteses -> "CO(µg/m³)" -> "CO"
        new_col = re.sub(r"\(.*?\)", "", col).strip()
        # keep only left side before first underscore if you want
        if drop_after_underscore and "_" in new_col:
            new_col = new_col.split("_", 1)[0]

        # verifica no dicionário
        linha = df_cod[df_cod["POLUENTE"].str.strip() == new_col]
        if not linha.empty:
            nomes.append(linha["NOME_PASTA"].values[0])
    
    return ",".join(sorted(set(nomes)))  # retorna só os nomes únicos

In [133]:
# Função para mapear e converter apenas os nomes dos poluentes quando NAS LINHAS DOS DFS
def id_pol_from_rows(
    df,
    df_cod,
    cols_cand=("POLUENTE","POLUENTES","Poluente","pollutant","nome_poluente","NomePoluente"),
    find_all=True,
    col_sinonimos=None,
    sep=r"[;,/|]+",
):
    """
    Identifica poluentes quando os nomes estão nas LINHAS e retorna nomes únicos
    padronizados conforme df_cod['NOME_PASTA'], separados por vírgula.

    Parâmetros:
      - df: DataFrame de entrada com as respostas.
      - df_cod: DataFrame de dicionário com colunas obrigatórias:
          'POLUENTE'  nome canônico
          'NOME_PASTA' nome padronizado desejado
        Opcional:
          col_sinonimos  nome de coluna em df_cod com sinônimos separados por '|'
      - cols_cand: possíveis nomes de coluna em df que contêm o texto dos poluentes.
      - find_all: se True, quando não encontrar cols_cand, varre todas as colunas de texto.
      - sep: regex para dividir listas no texto, ex "CO, NO2; PM10".

    Retorno:
      - str com nomes únicos padronizados, separados por vírgula.
    """

    # Normalização simples de rótulos para comparação
    def _erase(s: str) -> str:
        return "".join(c for c in unicodedata.normalize("NFKD", s) if not unicodedata.combining(c))

    def _norm(s: str) -> str:
        s = str(s)
        s = re.sub(r"\(.*?\)|\[.*?\]", "", s)  # remove unidades entre () ou []
        s = s.replace("µ", "u")
        s = _erase(s)
        s = re.sub(r"\s+", " ", s).strip().lower()
        return s

    # Mapa normalizado -> NOME_PASTA, incluindo sinônimos se houver
    cod_map = {}
    for _, row in df_cod.iterrows():
        cod_map[_norm(row["POLUENTE"])] = row["NOME_PASTA"]
        if col_sinonimos and col_sinonimos in df_cod.columns and pd.notna(row[col_sinonimos]):
            for alt in str(row[col_sinonimos]).split("|"):
                alt = alt.strip()
                if alt:
                    cod_map[_norm(alt)] = row["NOME_PASTA"]

    # Coleta de text candidatos
    text = []
    col_pol = next((c for c in df.columns if str(c).strip() in cols_cand), None)
    if col_pol is not None:
        text = df[col_pol].dropna().astype(str).tolist()
    elif find_all:
        for c in df.columns:
            if df[c].dtype.kind in "OUS":
                text += df[c].dropna().astype(str).tolist()

    if not text:
        return ""

    # Quebra por sep e mapeia para nomes padronizados
    seen = set()
    out = []
    sep_re = re.compile(sep)
    for t in text:
        partes = [p.strip() for p in sep_re.split(t)] if sep_re.search(t) else [t.strip()]
        for p in partes:
            if not p:
                continue
            k = _norm(p)
            nome = cod_map.get(k)
            if nome and nome not in seen:
                seen.add(nome)
                out.append(nome)

    return ",".join(sorted(out))

##### Criar planilha final com os campos desejados para DADOS ESTAÇÕES, quando df final está incompleto

In [134]:
def build_inventory_rows(df_all,
                         df_cod,
                         uf,
                         datetime_col: str | None=None,
                         drop_after_underscore=True,
                         mode: str = "rows"):
    """
    Monta linhas do inventário por estação.

    Parâmetros:
      - df_all: DataFrame já unido, com colunas 'ESTACAO' e opcionalmente a coluna de data-hora.
      - df_cod: dicionário de poluentes com colunas 'POLUENTE' e 'NOME_PASTA' (e opcional 'SINONIMOS').
      - uf: sigla UF, ex. 'RJ'.
      - datetime_col: nome da coluna de data-hora ou None (quando não há).
      - mode:
          'columns' -> detectar poluentes no CABEÇALHO (usa sua função id_pol(...))
          'rows'    -> detectar poluentes nas LINHAS (usa id_pol_from_rows(...))

    Retorno:
      - DataFrame com uma linha por estação, incluindo POLUENTE, INICIO e FIM.
    """ 
    rows = []
    for station, g in df_all.groupby("ESTACAO", sort=True):
        inicio, fim = station_year_bounds(g, datetime_col)
            
        if mode == "columns":
            # usa função que lê do cabeçalho; passar um nome para ignorar
            col_to_ignore = datetime_col 
            pols = id_pol(g, df_cod, column_name=col_to_ignore, drop_after_underscore=True)
        else:
            # lê poluentes a partir das LINHAS; NÃO passe datetime_col aqui
            pols = id_pol_from_rows(g, df_cod)
            
        rows.append({
            "UF": uf,
            "ID_OEMA": station,
            "CIDADE": "",
            "ID_MMA": "",  # será preenchido depois
            "ID_MMA_COMPLETO": "",
            "POLUENTE": pols,
            "COD_POLUENTE": "",  # deixamos vazio
            "CD_MUN": "",
            "COD_UF_IBGE": sigla_to_ibge(uf),
            "PROPRIETARIO": "",
            "PROP_ENTIDADE": "",
            "OPERADOR": "",
            "OP_ENTIDADE": "",
            "LATITUDE": "",
            "LONGITUDE": "",
            "MOBILIDADE": "",
            "CATEGORIA": "",
            "FUNCIONAMENTO": "",
            "METODO": "",
            "MARCA": "",
            "FINALIDADE": "",
            "MONITORAR": "",
            "FONTE": "",
            "CALIBRACAO": "",
            "REALOCACAO": "",
            "OBS_CALIBRACAO": "",
            "INICIO": inicio,
            "FIM": fim,
            "DADOS_MONITORAMENTO": "",
            "RECONHECIDA": "",
            "OBS_GERAIS": "",
            "CERTIFICACAO": "",
            "STATUS": "",
            "REP_ESPACIAL_DECLARADA": ""
        })

    return pd.DataFrame(rows)

##### Criar planilha final com os campos desejados para DADOS ESTAÇÕES, quando df final contém colunas desejadas

In [135]:
def pick_group_value(s: pd.Series, strategy="first_non_null", sep=" | "):
    """
    Escolhe um valor representativo dentro de um grupo.
    - s: Série com valores do grupo
    - strategy: "first_non_null" | "mode" | "concat_unique"
    - sep: separador para concatenação
    """
    ss = s.dropna()
    if ss.empty:
        return None
    if strategy == "mode":
        m = ss.mode()
        return m.iloc[0] if not m.empty else ss.iloc[0]
    if strategy == "concat_unique":
        vals = pd.unique(ss.astype(str).str.strip())
        return sep.join([v for v in vals if v])
    return ss.iloc[0]  # padrão

In [136]:
def build_inventory_flexible(
    df_all: pd.DataFrame,
    uf: str,
    df_cod,                         # usado pelas funções de poluentes
    group_col: str = "ID_OEMA",
    datetime_col: Optional[str] = None,
    mode: str = "rows",
    field_map: Optional[dict[str, Any]] = None,
    drop_after_underscore=True
) -> pd.DataFrame:
    """
    PT-BR:
      - df_all: DataFrame com ao menos group_col.
      - uf: sigla da UF.
      - df_cod: dicionário de poluentes.
      - group_col: chave do agrupamento.
      - datetime_col: coluna de data-hora para INICIO/FIM (opcional).
      - mode: "rows" usa id_pol_from_rows, "columns" usa id_pol.
      - field_map: regras extras, ex. {"CIDADE": ("col","CIDADE",{"strategy":"mode"})}
    """
    if group_col not in df_all.columns:
        raise ValueError(f"Coluna de agrupamento '{group_col}' não existe.")

    # funções de poluentes por modo
    if mode == "columns":
        pol_func = lambda g: id_pol(g, df_cod, column_name=datetime_col, drop_after_underscore=True)
    else:
        pol_func = lambda g: id_pol_from_rows(g, df_cod)

    # regras base: valores por grupo usando ("func", ...)
    base_rules = {
        "UF": uf,
        "ID_OEMA": ("group_key",),
        "POLUENTE": ("func", pol_func),
        "COD_UF_IBGE": sigla_to_ibge(uf),
        "INICIO": ("func", lambda g: station_year_bounds(g, datetime_col)[0]) if datetime_col else "",
        "FIM":    ("func", lambda g: station_year_bounds(g, datetime_col)[1]) if datetime_col else "",
        "CIDADE": "",
        "ID_MMA": "",
        "ID_MMA_COMPLETO": "",
        "COD_POLUENTE": "",
        "CD_MUN": "",
        "PROPRIETARIO": "",
        "PROP_ENTIDADE": "",
        "OPERADOR": "",
        "OP_ENTIDADE": "",
        "LATITUDE": "",
        "LONGITUDE": "",
        "MOBILIDADE": "",
        "CATEGORIA": "",
        "FUNCIONAMENTO": "",
        "METODO": "",
        "MARCA": "",
        "FINALIDADE": "",
        "MONITORAR": "",
        "FONTE": "",
        "CALIBRACAO": "",
        "REALOCACAO": "",
        "OBS_CALIBRACAO": "",
        "DADOS_MONITORAMENTO": "",
        "RECONHECIDA": "",
        "OBS_GERAIS": "",
        "CERTIFICACAO": "",
        "STATUS": "",
        "REP_ESPACIAL_DECLARADA": "",
    }

    rules = {**base_rules, **(field_map or {})}

    rows = []
    for group_key, g in df_all.groupby(group_col, sort=True):
        out = {}
        for field, rule in rules.items():
            # literal
            if not isinstance(rule, tuple):
                out[field] = rule
                continue

            kind = rule[0]
            if kind == "group_key":
                out[field] = group_key
            elif kind == "col":
                colname = rule[1]
                opts = rule[2] if len(rule) > 2 and isinstance(rule[2], dict) else {}
                out[field] = pick_group_value(g[colname], **opts) if colname in g.columns else None
            elif kind == "func":
                func = rule[1] if len(rule) > 1 else None
                out[field] = func(g) if callable(func) else None
            else:
                out[field] = None

        rows.append(out)

    return pd.DataFrame(rows)

##### Criar ID_MMA e ID_MMA_COMPLETO com base nas datas de inicio e fim das operações

NOTA: Sempre antes de criar ID_MMA COMPLETO na planilha final dos DADOS ESTAÇÕES, precisa garantir que as estações estejam com os campos início e fim preenchidos, para criar a sequência dos códigos - que depende do período de funcionamento

In [156]:
# def assign_id_mma(df, uf_col="UF"):
#     out = df.sort_values([uf_col, "INICIO", "ID_OEMA"], na_position="last").reset_index(drop=True)
#     seq = out.groupby(uf_col).cumcount().add(1).astype(str).str.zfill(4)
#     out["ID_MMA"] = out[uf_col] + seq
#     out["ID_MMA_COMPLETO"] = out["ID_MMA"]
#     return out

def _ascii_lower(s: str) -> str:
    s = str(s)
    s = ud.normalize("NFKD", s)
    s = "".join(ch for ch in s if not ud.combining(ch))
    return s.lower()

def assign_id_mma(df, uf_col="UF", date_col="INICIO"):
    out = df.copy()
    name_key = out["ID_OEMA"].astype(str).map(_ascii_lower)

    out = (
        out.assign(_name_key=name_key)
           .sort_values([uf_col, date_col, "_name_key"], kind="mergesort", na_position="last")
           .drop(columns="_name_key")
           .reset_index(drop=True)
    )

    seq = out.groupby(uf_col).cumcount().add(1).astype(str).str.zfill(4)
    out["ID_MMA"] = out[uf_col] + seq
    out["ID_MMA_COMPLETO"] = out["ID_MMA"]
    return out

##### Conferir após criação da planilha principal, se não permaneceram linhas duplicadas 

In [138]:
# Linhas duplicadas por chave (ex: "ID_OEMA")
def get_duplicate_rows(df: pd.DataFrame, key: str = "ID_OEMA") -> pd.DataFrame:
    '''  - df: DataFrame final
         - key: nome da coluna que quer usar como filtro'''
    
    s = df[key].astype("string").str.strip()
    mask = s.duplicated(keep=False)
    return df.loc[mask].sort_values(key)

# Linhas que correspondem a um valor específico da chave
# - value: valor a filtrar (ex.: "RJ_est_0123")
# def get_rows_for_id(df: pd.DataFrame, value, key: str = "ID_OEMA") -> pd.DataFrame:
#     s = df[key].astype("string").str.strip()
#     return df.loc[s.eq(str(value).strip())]

# Relatório por chave com número de linhas e colunas diferentes
def get_duplicates_report(df: pd.DataFrame, key: str = "ID_OEMA") -> pd.DataFrame:
    s = df[key].astype("string").str.strip()
    dups = df.loc[s.duplicated(keep=False)]
    rows = []
    for k, g in dups.groupby(key):
        nun = g.nunique(dropna=False)
        diff_cols = nun[nun > 1].index.tolist()
        rows.append({"ID": k, "n_rows": len(g), "diff_cols": diff_cols})
    rep = pd.DataFrame(rows).sort_values(["n_rows","ID"], ascending=[False, True])
    return rep

# Linhas duplicadas por chave dupla (ex: ["ID_OEMA","UF"])
def get_duplicate_rows_multi(df: pd.DataFrame, keys: list[str]) -> pd.DataFrame:
    # normaliza espaços nas chaves
    norm = df.copy()
    for k in keys:
        norm[k] = norm[k].astype("string").str.strip()
    mask = norm.duplicated(subset=keys, keep=False)
    return df.loc[mask].sort_values(keys)

# Todas as linhas de uma chave composta específica
def get_rows_for_keys(df: pd.DataFrame, keys: list[str], values: list) -> pd.DataFrame:
    m = pd.Series(True, index=df.index)
    for c, v in zip(keys, values):
        m &= df[c].astype("string").str.strip().eq(str(v).strip())
    return df.loc[m]

##### Salvar planilha com DADOS ESTAÇÕES

In [139]:
def save_UF_estacoes_csv(df_final, uf):
    base = Path.cwd().parent  
    out_dir = base / "data" / "DADOS_ESTACOES"
    out_dir.mkdir(parents=True, exist_ok=True)
    
    out_file = out_dir / f"{uf}_estacoes.csv"
    df_final.to_csv(out_file, index=False, encoding="utf-8")
    print("Saved to:", out_file.resolve())
    return

##### a) Ceará

In [399]:
# Substituir pelo estado desejado
uf = "CE" 

In [400]:
# Conferir respostas do formulário 
ce_forms = forms[forms["Unidade da Federação: "].str.contains(uf, case=False, na=False)]
ce_forms.iloc[:, idxs].head()

,Unidade da Federação:,Em sua UF há estações de monitoramento da qualidade do ar com equipamentos certificados como de referência ou equivalentes? \n,"Em caso afirmativo, a UF é responsável pela operação de algumas dessas estações?",Informe a quantidade de estações de operação própria:,Em sua UF há estações instaladas como condicionante de licença ambiental?,Informe a quantidade de estações instaladas por licença ambiental:,Em sua UF há estações de monitoramento indicativas (com sensores de baixo custo)?,Informe a quantidade de estações indicativas:,"Em 2024, houve mudanças na rede de monitoramento da qualidade do ar na UF?","Em caso afirmativo, indique quais mudanças ocorreram:",Quais são as perspectivas de curto prazo (1 ano) para a rede de monitoramento?,Deseja incluir observações sobre a rede de monitoramento da qualidade do ar? \n,Os dados de monitoramento da qualidade do ar são disponibilizados ao público?,"Em caso afirmativo, informe o endereço eletrônico para acesso:",Os dados de monitoramento passam por algum tratamento e formatação antes de serem disponibilizados?,"Em caso afirmativo, informe o método utilizado: \n(Exemplo: validação automática com base em padrões da CETESB)"
5,Ceará (CE),Sim,Sim,2,Não,0,Sim,8,Sim,Inativação de estações,Não há perspectivas de curto prazo,A Semace tem 2 estações de monitoramento da qu...,Não,NaN,Não,X


In [414]:
# Importar arquivos do estado
base = Path.cwd().parent 
df_dir = base / "data" / "DADOS_BRUTOS" 
df_dir.mkdir(parents=True, exist_ok=True)
print(df_dir)

ce_dir = df_dir / uf

/home/nobre/Notebooks/RQAR_2025_book/data/DADOS_BRUTOS


In [424]:
ce_dfs = load_csvs(ce_dir)

Loaded CIPP.csv: (21114, 25)
Loaded UM Parada Pecem.csv: (692, 24)
Loaded UM Parque Alto Alegre.csv: (557, 24)
Loaded UM UFC Reitoria.csv: (1922, 25)


In [425]:
for name, df in ce_dfs.items():
    print(f"\n=== {name} ===")
    display(df.head())


=== CIPP ===


,Date,BEN(),CH3-C6H5-CH3(µg/m³),CH4(ppm),CO(µg/m³),DD(grados),EBE(),HC(ppm),HR(%),LL(l/m²),...,PM10(µg/m³),PM10-MAN(µg/m³),PRB(hPa),RS(W/m²),SO2(µg/m³),SPM(),TIN(ºC),TMP(ºC),TOL(),VV(m/s)
0,25/08/2016 14:00,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,25/08/2016 15:00,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,25/08/2016 16:00,1.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,25/08/2016 17:00,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,25/08/2016 18:00,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN



=== UM Parada Pecem ===


,Date,BEN(),CH4(ppm),CO(µg/m³),DD(grados),EBE(),HC(ppm),HR(%),LL(l/m²),NMH(),...,PM10(µg/m³),PM25(),PRB(hPa),RS(W/m²),SO2(µg/m³),SPM(),TIN(ºC),TMP(ºC),TOL(),VV(m/s)
0,2019-07-25 07:00,0.1,2.0,276.0,188.0,0.2,2.5,74.0,0.0,0.52,...,7.0,NaN,1011.0,135.0,14.0,31.0,25.0,27.0,0.2,3.0
1,2019-07-25 08:00,0.2,2.0,276.0,193.0,0.3,2.6,69.0,0.0,0.53,...,7.0,NaN,1012.0,362.0,14.0,34.0,25.0,28.0,0.2,4.2
2,2019-07-25 09:00,0.1,1.9,228.0,200.0,0.1,2.4,60.0,0.0,0.52,...,4.0,NaN,1012.0,566.0,11.0,31.0,26.0,31.0,0.2,4.7
3,2019-07-25 10:00,0.1,1.9,228.0,190.0,0.1,2.4,56.0,0.0,0.53,...,16.0,NaN,1012.0,760.0,8.6,62.0,26.0,32.0,0.2,5.5
4,2019-07-25 11:00,0.1,1.9,252.0,192.0,0.1,2.4,54.0,0.0,0.50,...,13.0,NaN,1011.0,898.0,8.7,60.0,26.0,33.0,0.2,5.8



=== UM Parque Alto Alegre ===


,Date,BEN(),CH4(ppm),CO(µg/m³),DD(grados),EBE(),HC(ppm),HR(%),LL(l/m²),NMH(),...,PM10(µg/m³),PM25(),PRB(hPa),RS(W/m²),SO2(µg/m³),SPM(),TIN(ºC),TMP(ºC),TOL(),VV(m/s)
0,13/09/2019 15:00,0.12,2.2,927,120,0.10,3.6,42,0.0,1.4,...,153,NaN,1006,736,7.1,239,28,32,0.12,5.2
1,13/09/2019 16:00,0.12,2.2,1042,124,0.09,3.6,49,0.0,1.4,...,166,48.0,1006,438,10.0,253,28,31,0.12,4.5
2,13/09/2019 17:00,0.11,2.2,1099,130,0.09,3.7,54,0.0,1.5,...,150,52.0,1006,184,5.4,231,29,30,0.11,4.2
3,13/09/2019 18:00,0.11,2.2,1030,148,0.08,3.6,65,0.0,1.4,...,79,31.0,1006,15,4.0,137,29,28,0.10,2.3
4,13/09/2019 19:00,0.10,2.2,1042,138,0.08,3.6,75,0.0,1.4,...,53,20.0,1006,1,3.3,86,29,27,0.10,2.1



=== UM UFC Reitoria ===


,Fecha,BEN(),CH4(ppm),CO(µg/m³),DD(grados),EBE(),HC(ppm),HR(%),LL(l/m²),NMH(),...,PM10(µg/m³),PM25(),PRB(hPa),RS(W/m²),SO2(µg/m³),SPM(),TIN(ºC),TMP(ºC),TOL(),VV(m/s)
0,2019-02-04 08:00,0.0,0.0,0.0,208.0,0.0,0.0,72.0,0.1,0.0,...,3.0,3.0,1010,132.0,5.4,0.0,29,29.0,0.0,0.87
1,2019-02-04 09:00,0.0,0.0,792.0,210.0,0.0,0.0,75.0,0.0,0.0,...,3.0,3.0,1010,82.0,5.4,0.0,27,28.0,0.0,1.10
2,2019-02-04 10:00,0.0,0.0,1008.0,216.0,0.0,0.0,68.0,0.0,0.0,...,3.0,3.0,1010,333.0,5.4,0.0,25,29.0,0.0,0.79
3,2019-02-04 11:00,0.0,0.0,0.0,222.0,0.0,0.0,0.0,0.0,0.0,...,3.0,3.0,500,0.0,5.4,0.0,1,-30.0,0.0,1.30
4,2019-02-04 12:00,0.0,0.0,0.0,222.0,0.0,0.0,0.0,0.0,0.0,...,3.0,3.0,500,0.0,5.4,0.0,1,-30.0,0.0,1.30


In [426]:
# Renomear todas as colunas com data e hora para datetime
manual_map = {"fecha":  "DATETIME",
              "Date":  "DATETIME",
              "Fecha":  "DATETIME",
}

ce_dfs = {name: df.rename(columns=manual_map) for name, df in ce_dfs.items()}
ce_dfs

{'CIPP':                DATETIME  BEN()  CH3-C6H5-CH3(µg/m³)  CH4(ppm)  CO(µg/m³)  \
 0      25/08/2016 14:00    1.0                  NaN       NaN        NaN   
 1      25/08/2016 15:00    1.0                  NaN       NaN        NaN   
 2      25/08/2016 16:00    1.0                  0.0       NaN        NaN   
 3      25/08/2016 17:00    1.0                  NaN       NaN        NaN   
 4      25/08/2016 18:00    1.0                  NaN       NaN        NaN   
 ...                 ...    ...                  ...       ...        ...   
 21109  28/08/2023 11:00   25.0                  0.0       2.0      378.0   
 21110  28/08/2023 12:00   25.0                  0.0       1.0        NaN   
 21111  28/08/2023 13:00   24.0                  0.0       1.0        NaN   
 21112  28/08/2023 14:00   24.0                  0.0       1.0        NaN   
 21113  29/08/2023 09:00   25.0                  0.0       1.0        NaN   
 
        DD(grados)  EBE()  HC(ppm)  HR(%)  LL(l/m²)  ...  PM10(µg/

In [427]:
# Transformar coluna datetime para formato desejado
ce_dfs = {
    name: convert_column_to_datetime(d, column_name="DATETIME", format="%d/%m/%Y %H:%M")
    for name, d in ce_dfs.copy().items()
}

In [428]:
# Unir dataframes em único > Precisa ser feito antes de rodar a rotina de criação da planilha de estações
ce_frame = merge_station_dfs(ce_dfs.copy(), uf)
ce_frame.head()

,DATETIME,BEN(),CH3-C6H5-CH3(µg/m³),CH4(ppm),CO(µg/m³),DD(grados),EBE(),HC(ppm),HR(%),LL(l/m²),...,RS(W/m²),SO2(µg/m³),SPM(),TIN(ºC),TMP(ºC),TOL(),VV(m/s),ESTACAO,PM25(),O-XI(µg/m³)
0,2016-08-25 14:00:00,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,CIPP,NaN,NaN
1,2016-08-25 15:00:00,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,CIPP,NaN,NaN
2,2016-08-25 16:00:00,1.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,CIPP,NaN,NaN
3,2016-08-25 17:00:00,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,CIPP,NaN,NaN
4,2016-08-25 18:00:00,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,CIPP,NaN,NaN


In [429]:
# Corrigir problemas com datetime
ce_frame = fix_datetime_df(ce_frame, "DATETIME")
diag_bad_stations(ce_frame, "DATETIME")

UM Parada Pecem: parsed 0.0%  examples not parsed -> ['NaT', 'NaT', 'NaT', 'NaT', 'NaT']
UM UFC Reitoria: parsed 0.0%  examples not parsed -> ['NaT', 'NaT', 'NaT', 'NaT', 'NaT']


In [431]:
ce_dfs = build_inventory_rows(ce_frame.copy(), df_cod, uf, datetime_col='DATETIME', mode="columns")
ce_dfs = assign_id_mma(ce_dfs)
ce_dfs.head()

,UF,ID_OEMA,CIDADE,ID_MMA,ID_MMA_COMPLETO,POLUENTE,COD_POLUENTE,CD_MUN,COD_UF_IBGE,PROPRIETARIO,...,REALOCACAO,OBS_CALIBRACAO,INICIO,FIM,DADOS_MONITORAMENTO,RECONHECIDA,OBS_GERAIS,CERTIFICACAO,STATUS,REP_ESPACIAL_DECLARADA
0,CE,CIPP,,CE0001,CE0001,"BENZENO,CH4,CO,HCNM,HCT,MP10,MP25,NO,NO2,NOX,O...",,,23,,...,,,2016,2023,,,,,,
1,CE,UM Parque Alto Alegre,,CE0002,CE0002,"BENZENO,CH4,CO,HCNM,HCT,MP10,MP25,NO,NO2,NOX,O...",,,23,,...,,,2019,2019,,,,,,
2,CE,UM Parada Pecem,,CE0003,CE0003,"BENZENO,CH4,CO,HCNM,HCT,MP10,MP25,NO,NO2,NOX,O...",,,23,,...,,,<NA>,<NA>,,,,,,
3,CE,UM UFC Reitoria,,CE0004,CE0004,"BENZENO,CH4,CO,HCNM,HCT,MP10,MP25,NO,NO2,NOX,O...",,,23,,...,,,<NA>,<NA>,,,,,,


In [432]:
save_UF_estacoes_csv(ce_dfs, uf)

Saved to: /home/nobre/Notebooks/RQAR_2025_book/data/DADOS_ESTACOES/CE_estacoes.csv


##### b) Rio de Janeiro

In [232]:
# Substituir pelo estado desejado
uf = "RJ" 

In [233]:
# Conferir respostas do formulário 
rj_forms = forms[forms["Unidade da Federação: "].str.contains(uf, case=False, na=False)]
rj_forms.iloc[:, idxs].head()

,Unidade da Federação:,Em sua UF há estações de monitoramento da qualidade do ar com equipamentos certificados como de referência ou equivalentes? \n,"Em caso afirmativo, a UF é responsável pela operação de algumas dessas estações?",Informe a quantidade de estações de operação própria:,Em sua UF há estações instaladas como condicionante de licença ambiental?,Informe a quantidade de estações instaladas por licença ambiental:,Em sua UF há estações de monitoramento indicativas (com sensores de baixo custo)?,Informe a quantidade de estações indicativas:,"Em 2024, houve mudanças na rede de monitoramento da qualidade do ar na UF?","Em caso afirmativo, indique quais mudanças ocorreram:",Quais são as perspectivas de curto prazo (1 ano) para a rede de monitoramento?,Deseja incluir observações sobre a rede de monitoramento da qualidade do ar? \n,Os dados de monitoramento da qualidade do ar são disponibilizados ao público?,"Em caso afirmativo, informe o endereço eletrônico para acesso:",Os dados de monitoramento passam por algum tratamento e formatação antes de serem disponibilizados?,"Em caso afirmativo, informe o método utilizado: \n(Exemplo: validação automática com base em padrões da CETESB)"
22,Rio de Janeiro (RJ),Sim,Não,46,Sim,144,Não,0,Sim,"Expansão da rede (novas estações), Reativação ...",Ampliação da rede estadual,O questionário foi preenchido com base nas est...,Sim,https://portalsigqar.inea.rj.gov.br/,Sim,Validação automática e também pela equipe de a...


In [234]:
# Importar arquivos do estado
base = Path.cwd().parent 
df_dir = base / "data" / "DADOS_ESTACOES" 
df_dir.mkdir(parents=True, exist_ok=True)
print(df_dir)

rj_dir = df_dir / uf 
os.listdir(rj_dir)
#print(rj_dir)

/home/nobre/Notebooks/RQAR_2025_book/data/DADOS_ESTACOES


['RJ_estacoes_nao_preenchidas.xlsx',
 'RJ_estacoes_enviada.xlsx',
 'RJ_estacoes.csv']

In [235]:
''' Neste caso, a inspeção das pastas indiciou que existiam arquivos .XLSX e .CSV, 
    necessitando aplicar duas operações diferentes para leitura dos arquivos'''

rj_exls = load_excels(rj_dir) # Está organizado para ler sempre a primeira planilha
rj_csvs = load_csvs(rj_dir)

Loaded RJ_estacoes_enviada.xlsx [sheet 0]: (64, 29)
Loaded RJ_estacoes_nao_preenchidas.xlsx [sheet 0]: (57, 31)
Loaded RJ_estacoes.csv: (121, 31)


In [236]:
''' Neste caso, como são dois dicionários diferentes - uma para arquivos .XLSX e uma para arquivos .CSV,
    foi necessário fazer um merge entre eles. Esse passo vai existir nessas condições'''

rj_dfs = rj_csvs | rj_exls 

In [237]:
for name, df in rj_dfs.items():
    print(f"\n=== {name} ===")
    display(df.head())


=== RJ_estacoes ===


,UF,COD_UF_IBGE,ID_OEMA,ID_MMA,CIDADE,CD_MUN,CATEGORIA,FUNCIONAMENTO,PROPRIETARIO,PROP_ENTIDADE,...,FIM,STATUS,CALIBRACAO,OBS_CALIBRACAO,MONITORAR,FONTE,OBS_GERAIS,DADOS_MONITORAMENTO,RECONHECIDA,REP_ESPACIAL_DECLARADA
0,RJ,33,RJ - Largo do Bodegão,RJ0012,Rio de Janeiro,3304557.0,Referencia,Automatica,NaN,NaN,...,NaN,Ativa,NaN,NaN,NaN,Consulta 2024,NaN,NaN,NaN,NaN
1,RJ,33,BR - São Bernardo,RJ0018,Belford Roxo,3300456.0,Referencia,Automatica,NaN,NaN,...,NaN,Ativa,NaN,NaN,NaN,Consulta 2024,NaN,NaN,NaN,NaN
2,RJ,33,NI - Monteiro Lobato,RJ0019,Nova Iguaçu,3303500.0,Referencia,Automatica,NaN,NaN,...,NaN,Ativa,NaN,NaN,NaN,Consulta 2024,NaN,NaN,NaN,NaN
3,RJ,33,RJ - Campo dos Afonsos,RJ0020,Rio de Janeiro,3304557.0,Referencia,Automatica,NaN,NaN,...,NaN,Ativa,NaN,NaN,NaN,Consulta 2024,NaN,NaN,NaN,NaN
4,RJ,33,RJ - Taquara,RJ0021,Rio de Janeiro,3304557.0,Referencia,Automatica,NaN,NaN,...,NaN,Ativa,NaN,NaN,NaN,Consulta 2024,NaN,NaN,NaN,NaN



=== RJ_estacoes_enviada ===


,UF,COD_UF_IBGE,ID_OEMA,ID_MMA,CIDADE,CD_MUN,CATEGORIA,FUNCIONAMENTO,PROPRIETARIO,PROP_ENTIDADE,...,FINALIDADE,REP_ESPACIAL,INICIO,FIM,STATUS,CALIBRACAO,OBS_CALIBRACAO,MONITORAR,FONTE,OBS_GERAIS
0,RJ,33,BM - Boa Sorte,RJ0073,Barra Mansa,3300407,Referencia,Automática,Saint Gobain,Privada,...,Licenciamento Ambiental,Bairro,NaN,NaN,Ativa,Sim,NaN,Não,Coleta Interna,NaN
1,RJ,33,BM - Sesi,RJ0074,Barra Mansa,3300407,Referencia,Automática,Saint Gobain,Privada,...,Licenciamento Ambiental,Bairro,NaN,NaN,Ativa,Sim,NaN,Não,Coleta Interna,NaN
2,RJ,33,BM - Bocaininha,RJ0075,Barra Mansa,3300407,Referencia,Automática,Arcelormittal Barra Mansa,Privada,...,Licenciamento Ambiental,Bairro,NaN,NaN,Ativa,Sim,NaN,Não,Coleta Interna,NaN
3,RJ,33,BM - Roberto Silveira,RJ0076,Barra Mansa,3300407,Referencia,Automática,Arcelormittal Barra Mansa,Privada,...,Licenciamento Ambiental,Bairro,NaN,NaN,Ativa,Sim,NaN,Não,Coleta Interna,NaN
4,RJ,33,BM - Vista Alegre,RJ0077,Barra Mansa,3300407,Referencia,Automática,Arcelormittal Barra Mansa,Privada,...,Licenciamento Ambiental,Bairro,NaN,NaN,Ativa,Sim,NaN,Não,Coleta Interna,NaN



=== RJ_estacoes_nao_preenchidas ===


,UF,COD_UF_IBGE,ID_OEMA,ID_MMA,CIDADE,CD_MUN,CATEGORIA,FUNCIONAMENTO,PROPRIETARIO,PROP_ENTIDADE,...,FIM,STATUS,CALIBRACAO,OBS_CALIBRACAO,MONITORAR,FONTE,OBS_GERAIS,DADOS_MONITORAMENTO,RECONHECIDA,REP_ESPACIAL_DECLARADA
0,RJ,33,E. Móvel - Barra Mansa,RJ0299,Barra Mansa,3300407.0,Referencia,Automatica,INEA,Pública,...,NaN,Inativa,x,NaN,Não,NaN,NaN,NaN,NaN,NaN
1,RJ,33,E. Móvel - Belford Roxo,RJ0298,Belford Roxo,3300456.0,Referencia,Automatica,INEA,Pública,...,NaN,Inativa,x,NaN,Não,NaN,NaN,NaN,NaN,NaN
2,RJ,33,Estação Meteorológica - Ute Campos,RJ0613,Campos dos Goytacazes,3301009.0,Referencia,Automatica,Furnas S.A,Privada,...,NaN,Inativa,x,NaN,Não,NaN,NaN,NaN,NaN,NaN
3,RJ,33,Cg - Meteorológica Euclidelândia 1,RJ0053,Cantagalo,3301108.0,Referencia,Automatica,Votorantim Cimentos S.A,Privada,...,NaN,Ativa,x,NaN,Não,NaN,NaN,NaN,NaN,NaN
4,RJ,33,DC - Vila São Luiz,RJ0033,Duque de Caxias,3301702.0,Referencia,Automatica,REDUC,Privada,...,NaN,Inativa,x,NaN,Não,Consulta 2024,NaN,NaN,NaN,NaN


In [238]:
# Encontrar colunas diferentes e iguais dentro dos dicts 
summary, stats = summarize_columns(rj_dfs, normalize=True)

# Colunas iguais em todos
cols_comuns = summary.loc[summary.is_common, "col"].tolist()
print("Qtd colunas comuns:", len(cols_comuns))
print("Algumas comuns:", cols_comuns[:10])

# Colunas diferentes em todos
cols_diff = summary.loc[~ summary.is_common, "col"].tolist()
print("Qtd colunas diferentes:", len(cols_diff))
print("Algumas diferentes:", cols_diff[:10])

# Estatísticas por DataFrame
stats[["df","n_cols","n_common_present","n_common_missing","extras_count"]].head()

Qtd colunas comuns: 28
Algumas comuns: ['calibracao', 'categoria', 'cd_mun', 'cidade', 'cod_uf_ibge', 'fim', 'finalidade', 'fonte', 'funcionamento', 'id_mma']
Qtd colunas diferentes: 4
Algumas diferentes: ['dados_monitoramento', 'reconhecida', 'rep_espacial_declarada', 'rep_espacial']


,df,n_cols,n_common_present,n_common_missing,extras_count
0,RJ_estacoes,31,28,0,3
1,RJ_estacoes_enviada,29,28,0,1
2,RJ_estacoes_nao_preenchidas,31,28,0,3


In [239]:
# Limpar caracteres indesejados
for name, d in rj_dfs.items(): clean_df_all_text(d, in_place=True)

In [240]:
# Depois de rodar pela primeira vez e identificar os conflitos, posso escolher a base de dados prioritária para sobrepor informações
priority = ["RJ_estacoes_enviada", "RJ_estacoes_nao_preenchidas", "RJ_estacoes"]

rj_frame, conflicts = merge_by_id_multi(
    rj_dfs.copy(),
    id_cols=['ID_OEMA','POLUENTE'],
    source_priority=priority,
    drop_empty_strings=True,
)
rj_frame

,ID_OEMA,POLUENTE,UF,COD_UF_IBGE,ID_MMA,CIDADE,CD_MUN,CATEGORIA,FUNCIONAMENTO,PROPRIETARIO,...,STATUS,CALIBRACAO,OBS_CALIBRACAO,MONITORAR,FONTE,OBS_GERAIS,DADOS_MONITORAMENTO,RECONHECIDA,REP_ESPACIAL_DECLARADA,REP_ESPACIAL
0,RJ - Largo do Bodegao,"MP10,NO,.ipynb_checkpoints,HCT,BENZENO,PTS,CH4...",RJ,33,RJ0012,Rio de Janeiro,3304557.0,Referencia,Automatica,<NA>,...,Ativa,<NA>,<NA>,<NA>,Consulta 2024,<NA>,<NA>,<NA>,<NA>,<NA>
1,BR - Sao Bernardo,"MP10,NO,.ipynb_checkpoints,HCT,CO,CH4,O3,NO2,H...",RJ,33,RJ0018,Belford Roxo,3300456.0,Referencia,Automatica,<NA>,...,Ativa,<NA>,<NA>,<NA>,Consulta 2024,<NA>,<NA>,<NA>,<NA>,<NA>
2,NI - Monteiro Lobato,"MP10,NO,HCT,CO,CH4,O3,SO2,NO2,HCNM,NOX",RJ,33,RJ0019,Nova Iguacu,3303500.0,Referencia,Automatica,INEA,...,Ativa,Sim,<NA>,Nao,Coleta Interna,<NA>,<NA>,<NA>,<NA>,Bairro
3,RJ - Campo dos Afonsos,"NO,O3,NO2,NOX",RJ,33,RJ0020,Rio de Janeiro,3304557.0,Referencia,Automatica,INEA,...,Ativa,Sim,<NA>,Nao,Coleta Interna,<NA>,<NA>,<NA>,<NA>,Bairro
4,RJ - Taquara,"MP10,NO,HCT,CO,CH4,O3,SO2,NO2,HCNM,NOX",RJ,33,RJ0021,Rio de Janeiro,3304557.0,Referencia,Automatica,INEA,...,Ativa,Sim,<NA>,Nao,Coleta Interna,<NA>,<NA>,<NA>,<NA>,Bairro
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
102,Mt - Praia Do Saco,"MP10,PTS",RJ,33,RJ0068,Mangaratiba,3302601.0,Referencia,Automatica,Vale S. A,...,Ativa,Sim,<NA>,Nao,Coleta Interna,<NA>,<NA>,<NA>,<NA>,Urbana
103,RJ - Largo do Bodegao,"MP10,NO,HCT,BENZENO,PTS,CH4,O3,ETILBENZENO,SO2...",RJ,33,RJ0012,Rio de Janeiro,3304557.0,Referencia,Automatica,Ternium Brasil,...,Ativa,Sim,<NA>,Nao,Coleta Interna,<NA>,<NA>,<NA>,<NA>,Bairro
104,RJ - Manguinhos,"MP10,MP2,5,NO,HCT,CO,CH4,O3,SO2,NO2,HCNM,NOX,MP25",RJ,33,RJ0029,Rio de Janeiro,3304557.0,Referencia,Automatica,Refinaria de Manguinhos (Refit),...,Ativa,Sim,<NA>,Nao,Coleta Interna,<NA>,<NA>,<NA>,<NA>,Urbana
105,RJ - Joao XXIII (Caminhao),"NO,HCT,BENZENO,CH4,O3,ETILBENZENO,SO2,NO2,TOLU...",RJ,33,RJ0216,Rio de Janeiro,3304557.0,Referencia,Automatica,INEA,...,Inativa,x,<NA>,Nao,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>


In [241]:
rj_frame.columns

Index(['ID_OEMA', 'POLUENTE', 'UF', 'COD_UF_IBGE', 'ID_MMA', 'CIDADE',
       'CD_MUN', 'CATEGORIA', 'FUNCIONAMENTO', 'PROPRIETARIO', 'PROP_ENTIDADE',
       'OPERADOR', 'OP_ENTIDADE', 'LATITUDE', 'LONGITUDE', 'MOBILIDADE',
       'REALOCACAO', 'MARCA', 'METODO', 'FINALIDADE', 'INICIO', 'FIM',
       'STATUS', 'CALIBRACAO', 'OBS_CALIBRACAO', 'MONITORAR', 'FONTE',
       'OBS_GERAIS', 'DADOS_MONITORAMENTO', 'RECONHECIDA',
       'REP_ESPACIAL_DECLARADA', 'REP_ESPACIAL'],
      dtype='object')

In [242]:
# Lista de colunas alvo que já existem no df (na ordem desejada)
cols = [
    "ID_OEMA","ID_MMA","CIDADE","CD_MUN","CATEGORIA",
    "FUNCIONAMENTO","PROPRIETARIO","PROP_ENTIDADE","OPERADOR","OP_ENTIDADE",
    "LATITUDE","LONGITUDE","MOBILIDADE","REALOCACAO","MARCA","METODO","FINALIDADE",
    "STATUS","CALIBRACAO","OBS_CALIBRACAO","MONITORAR","FONTE",
    "OBS_GERAIS","DADOS_MONITORAMENTO","RECONHECIDA","REP_ESPACIAL_DECLARADA",
    "REP_ESPACIAL"
]

def make_field_map(cols,
                   prefer_mode=("CIDADE",),     # PT-BR: colunas que preferem moda
                   use_group_key=("ID_OEMA",)): # PT-BR: colunas que vêm da chave do grupo
    fm = {}
    for c in cols:
        if c in use_group_key:
            fm[c] = ("group_key",)
        elif c in prefer_mode:
            fm[c] = ("col", c, {"strategy": "mode"})
        else:
            fm[c] = ("col", c, {"strategy": "first_non_null"})
    return fm

field_map = make_field_map(cols)
teste = build_inventory_flexible(
    rj_frame.copy(),
    uf,
    df_cod,
    group_col="ID_OEMA",
    datetime_col=None,  # ou "DATETIME" se desejar calcular INICIO/FIM
    mode= "rows",
    field_map=field_map
)
teste

,UF,ID_OEMA,POLUENTE,COD_UF_IBGE,INICIO,FIM,CIDADE,ID_MMA,ID_MMA_COMPLETO,COD_POLUENTE,...,CALIBRACAO,REALOCACAO,OBS_CALIBRACAO,DADOS_MONITORAMENTO,RECONHECIDA,OBS_GERAIS,CERTIFICACAO,STATUS,REP_ESPACIAL_DECLARADA,REP_ESPACIAL
0,RJ,BM - Boa Sorte,"MP10,PTS",33,,,Barra Mansa,RJ0073,,,...,Sim,Nao,None,None,None,None,,Ativa,None,Bairro
1,RJ,BM - Bocaininha,"MP10,PTS",33,,,Barra Mansa,RJ0075,,,...,Sim,Nao,None,None,None,None,,Ativa,None,Bairro
2,RJ,BM - Roberto Silveira,"MP10,PTS",33,,,Barra Mansa,RJ0076,,,...,Sim,Nao,None,None,None,None,,Ativa,None,Bairro
3,RJ,BM - Sesi,"MP10,PTS",33,,,Barra Mansa,RJ0074,,,...,Sim,Nao,None,None,None,None,,Ativa,None,Bairro
4,RJ,BM - Vista Alegre,"MP10,PTS",33,,,Barra Mansa,RJ0077,,,...,Sim,Nao,None,None,None,None,,Ativa,None,Bairro
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,RJ,Sp - Piranema,"CH4,CO,HCNM,HCT,MP10,NO,NO2,NOX,O3,SO2",33,,,Seropedica,RJ0058,,,...,Sim,Nao,None,None,None,None,,Ativa,None,Urbana
96,RJ,VR - Belmonte,"BENZENO,CH4,CO,ETILBENZENO,HCNM,HCT,MP10,MP25,...",33,,,Volta Redonda,RJ0069,,,...,Sim,Nao,None,None,None,None,,Ativa,None,Bairro
97,RJ,VR - Nossa Sra. das Gracas (Van),"MP10,MP25,NO,NO2,NOX,PTS,SO2",33,,,Volta Redonda,RJ0637,,,...,Sim,Nao,None,None,None,None,,Ativa,None,Bairro
98,RJ,VR - Retiro,"BENZENO,CH4,CO,ETILBENZENO,HCNM,HCT,MP10,MP25,...",33,,,Volta Redonda,RJ0070,,,...,Sim,Nao,None,None,None,None,,Ativa,None,Bairro


In [278]:
save_UF_estacoes_csv(teste, uf)

Saved to: /home/nobre/Notebooks/RQAR_2025_book/data/DADOS_ESTACOES/RJ_estacoes.csv


In [244]:
print("IDs orig:", rj_frame["ID_OEMA"].astype("string").str.strip().dropna().nunique())
print("IDs inv :", teste["ID_OEMA"].astype("string").str.strip().dropna().nunique())

# ver se POLUENTE varia por linha (não deve ser igual para todas)
print(teste["POLUENTE"].nunique(), "poluente(s) distintos")


IDs orig: 100
IDs inv : 100
45 poluente(s) distintos


In [243]:
dups_df = get_duplicate_rows(rj_frame, key="ID_OEMA")
dups_df

,ID_OEMA,POLUENTE,UF,COD_UF_IBGE,ID_MMA,CIDADE,CD_MUN,CATEGORIA,FUNCIONAMENTO,PROPRIETARIO,...,STATUS,CALIBRACAO,OBS_CALIBRACAO,MONITORAR,FONTE,OBS_GERAIS,DADOS_MONITORAMENTO,RECONHECIDA,REP_ESPACIAL_DECLARADA,REP_ESPACIAL
1,BR - Sao Bernardo,"MP10,NO,.ipynb_checkpoints,HCT,CO,CH4,O3,NO2,H...",RJ,33,RJ0018,Belford Roxo,3300456.0,Referencia,Automatica,<NA>,...,Ativa,<NA>,<NA>,<NA>,Consulta 2024,<NA>,<NA>,<NA>,<NA>,<NA>
93,BR - Sao Bernardo,"MP10,NO,HCT,CO,CH4,O3,NO2,HCNM,NOX",RJ,33,RJ0018,Belford Roxo,3300456.0,Referencia,Automatica,INEA,...,Ativa,Sim,<NA>,Nao,Coleta Interna,<NA>,<NA>,<NA>,<NA>,Bairro
28,Cg - Macuco,"MP10,NO,PTS,O3,NO2,NOX,MP25",RJ,33,RJ0051,Macuco,3302452.0,Referencia,Automatica,<NA>,...,Ativa,<NA>,<NA>,<NA>,Consulta 2024,<NA>,<NA>,<NA>,<NA>,<NA>
101,Cg - Macuco,"MP10,NO,O3,NO2,NOX,MP25",RJ,33,RJ0051,Macuco,3302452.0,Referencia,Automatica,CSN Cimento - Cantagalo,...,Ativa,Sim,<NA>,Nao,Coleta Interna,<NA>,<NA>,<NA>,<NA>,Urbana
19,Itb - Porto das Caixas,"MP10,NO,HCT,BENZENO,CO,CH4,O3,ETILBENZENO,SO2,...",RJ,33,RJ0038,Itaborai,3301900.0,Referencia,Automatica,<NA>,...,Ativa,<NA>,<NA>,<NA>,Consulta 2024,<NA>,<NA>,<NA>,<NA>,<NA>
96,Itb - Porto das Caixas,"MP10,PTS,NO,HCT,BENZENO,CO,CH4,O3,ETILBENZENO,...",RJ,33,RJ0038,Itaborai,3301900.0,Referencia,Automatica,Complexo de Energias Boaventura,...,Ativa,Sim,<NA>,Nao,Coleta Interna,<NA>,<NA>,<NA>,<NA>,Bairro
37,Itg - Ilha Da Madeira,"MP10,PTS,MP25",RJ,33,RJ0066,Itaguai,3302007.0,Referencia,Automatica,<NA>,...,Ativa,<NA>,<NA>,<NA>,Consulta 2024,<NA>,<NA>,<NA>,<NA>,<NA>
99,Itg - Ilha Da Madeira,"MP10,PTS",RJ,33,RJ0066,Itaguai,3302007.0,Referencia,Automatica,Vale S. A,...,Ativa,Sim,<NA>,Nao,Coleta Interna,<NA>,<NA>,<NA>,<NA>,Urbana
39,Mt - Praia Do Saco,"MP10,PTS,MP25",RJ,33,RJ0068,Mangaratiba,3302601.0,Referencia,Automatica,<NA>,...,Ativa,<NA>,<NA>,<NA>,Consulta 2024,<NA>,<NA>,<NA>,<NA>,<NA>
102,Mt - Praia Do Saco,"MP10,PTS",RJ,33,RJ0068,Mangaratiba,3302601.0,Referencia,Automatica,Vale S. A,...,Ativa,Sim,<NA>,Nao,Coleta Interna,<NA>,<NA>,<NA>,<NA>,Urbana


In [129]:
# # mapeia campos extras que quer pegar das linhas
# field_map = {
#     "ID_OEMA": ("col", "ID_OEMA", {"strategy": "first_non_null"}),
#     "CIDADE":  ("col", "CIDADE", {"strategy": "mode"}),
# }

# teste = build_inventory_flexible(
#     rj_frame.copy(),
#     uf,
#     df_cod,
#     group_col="ID_OEMA",
#     datetime_col=None,
#     mode= "rows",       
#     field_map=field_map
# )
# teste

,UF,ID_OEMA,CIDADE,ID_MMA,ID_MMA_COMPLETO,POLUENTE,COD_POLUENTE,CD_MUN,COD_UF_IBGE,PROPRIETARIO,...,REALOCACAO,OBS_CALIBRACAO,INICIO,FIM,DADOS_MONITORAMENTO,RECONHECIDA,OBS_GERAIS,CERTIFICACAO,STATUS,REP_ESPACIAL_DECLARADA
0,RJ,BM - Boa Sorte,Barra Mansa,,,"BENZENO,CH4,CO,ETILBENZENO,HCNM,HCT,MP10,MP25,...",,,33,,...,,,None,None,,,,,,
1,RJ,BM - Bocaininha,Barra Mansa,,,"BENZENO,CH4,CO,ETILBENZENO,HCNM,HCT,MP10,MP25,...",,,33,,...,,,None,None,,,,,,
2,RJ,BM - Roberto Silveira,Barra Mansa,,,"BENZENO,CH4,CO,ETILBENZENO,HCNM,HCT,MP10,MP25,...",,,33,,...,,,None,None,,,,,,
3,RJ,BM - Sesi,Barra Mansa,,,"BENZENO,CH4,CO,ETILBENZENO,HCNM,HCT,MP10,MP25,...",,,33,,...,,,None,None,,,,,,
4,RJ,BM - Vista Alegre,Barra Mansa,,,"BENZENO,CH4,CO,ETILBENZENO,HCNM,HCT,MP10,MP25,...",,,33,,...,,,None,None,,,,,,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
119,RJ,VR - Belmonte,Volta Redonda,,,"BENZENO,CH4,CO,ETILBENZENO,HCNM,HCT,MP10,MP25,...",,,33,,...,,,None,None,,,,,,
120,RJ,VR - Meteorológica Ilha das Águas Cruas,Volta Redonda,,,"BENZENO,CH4,CO,ETILBENZENO,HCNM,HCT,MP10,MP25,...",,,33,,...,,,None,None,,,,,,
121,RJ,VR - Nossa Sra. das Graças (Van),Volta Redonda,,,"BENZENO,CH4,CO,ETILBENZENO,HCNM,HCT,MP10,MP25,...",,,33,,...,,,None,None,,,,,,
122,RJ,VR - Retiro,Volta Redonda,,,"BENZENO,CH4,CO,ETILBENZENO,HCNM,HCT,MP10,MP25,...",,,33,,...,,,None,None,,,,,,


In [301]:
#save_UF_estacoes_csv(rj_dfs, uf)

Saved to: /home/nobre/Notebooks/RQAR_2025_book/data/DADOS_ESTACOES/CE_estacoes.csv


##### c) Paraíba

In [433]:
# Substituir pelo estado desejado
uf = "PB" 

In [434]:
# Conferir respostas do formulário 
pb_forms = forms[forms["Unidade da Federação: "].str.contains(uf, case=False, na=False)]
pb_forms.iloc[:, idxs].head()

,Unidade da Federação:,Em sua UF há estações de monitoramento da qualidade do ar com equipamentos certificados como de referência ou equivalentes? \n,"Em caso afirmativo, a UF é responsável pela operação de algumas dessas estações?",Informe a quantidade de estações de operação própria:,Em sua UF há estações instaladas como condicionante de licença ambiental?,Informe a quantidade de estações instaladas por licença ambiental:,Em sua UF há estações de monitoramento indicativas (com sensores de baixo custo)?,Informe a quantidade de estações indicativas:,"Em 2024, houve mudanças na rede de monitoramento da qualidade do ar na UF?","Em caso afirmativo, indique quais mudanças ocorreram:",Quais são as perspectivas de curto prazo (1 ano) para a rede de monitoramento?,Deseja incluir observações sobre a rede de monitoramento da qualidade do ar? \n,Os dados de monitoramento da qualidade do ar são disponibilizados ao público?,"Em caso afirmativo, informe o endereço eletrônico para acesso:",Os dados de monitoramento passam por algum tratamento e formatação antes de serem disponibilizados?,"Em caso afirmativo, informe o método utilizado: \n(Exemplo: validação automática com base em padrões da CETESB)"
20,Paraíba (PB),Sim,Sim,4,Não,0,Sim,14,Sim,Expansão da rede (novas estações),Início do monitoramento com estações certificadas,A Paraíba possui quatorze monitores indicativo...,Não,NaN,Não,X


In [559]:
# Importar arquivos do estado
base = Path.cwd().parent 
df_dir = base / "data" / "DADOS_BRUTOS" 
df_dir.mkdir(parents=True, exist_ok=True)

pb_dir = df_dir / uf
print(pb_dir)

/home/nobre/Notebooks/RQAR_2025_book/data/DADOS_BRUTOS/PB


In [560]:
pb_dfs = load_txts(pb_dir)

Loaded Estação 1.txt: (2810, 8)
Loaded Estação 2.txt: (2395, 8)
Loaded Estação 3.txt: (3175, 8)


In [561]:
for name, df in pb_dfs.items():
    print(f"\n=== {name} ===")
    display(df.head())


=== Estação 1 ===


,0,1,2,3,4,5,6,7
0,2024-08-22T20:00:00,23040019,SUDEMA,-999999.00,-999999.00,-999999.00,-999999.00,-999999.0
1,2024-08-22T21:00:00,23040019,SUDEMA,54.25,42.17,17.42,15.33,12.8
2,2024-08-22T22:00:00,23040019,SUDEMA,54.92,43.92,19.67,17.58,12.8
3,2024-08-22T23:00:00,23040019,SUDEMA,47.75,39.75,19.50,17.92,12.7
4,2024-08-23T00:00:00,23040019,SUDEMA,79.67,60.00,21.08,18.08,12.7



=== Estação 2 ===


,0,1,2,3,4,5,6,7
0,2024-08-22T19:00:00,23040016,SUDEMA,46.40,35.60,16.40,14.40,12.5
1,2024-08-22T20:00:00,23040016,SUDEMA,63.67,47.33,19.67,16.67,12.5
2,2024-08-22T21:00:00,23040016,SUDEMA,57.17,43.58,19.67,17.00,12.4
3,2024-08-22T22:00:00,23040016,SUDEMA,53.42,41.67,19.92,17.75,12.4
4,2024-08-22T23:00:00,23040016,SUDEMA,53.33,41.75,20.25,17.42,12.4



=== Estação 3 ===


,0,1,2,3,4,5,6,7
0,2024-08-21T16:00:00,23040023,SUDEMA,22.22,21.22,18.78,14.56,12.7
1,2024-08-21T17:00:00,23040023,SUDEMA,28.92,27.25,24.08,18.50,12.5
2,2024-08-21T18:00:00,23040023,SUDEMA,31.58,29.75,26.58,20.50,12.4
3,2024-08-21T19:00:00,23040023,SUDEMA,36.42,34.50,30.25,22.83,12.4
4,2024-08-21T20:00:00,23040023,SUDEMA,41.42,38.50,32.92,24.58,12.3


NOTA: Informações fornecidas pela UF

Localização:

Estação 1: 7°2'4.75"S 34°50'34.73"W

Estação 2: 7°5'11.20"S 34°50'56.19"W

Estação 3: 7°2'50.65"S 34°57'23.56"W

Formato do arquivo de dados:[Data]T[Hora];[Serial];[NOME DO ÓRGÃO];[PTS];[PM10];[PM2.5];[PM1];[VOLTAGEM DA BATERIA];

In [562]:
# Adicionar informações de cabeçalho 
# Manual_map foi montado com base nas informações fornecidas pelo estado sobre os arquivos .txt
manual_map = [
    "DATETIME", "serial", "PROPRIETARIO", "PTS",
    "MP10", "MP25", "MP1", "voltagem_bateria"
]

pb_dfs = {name: df.set_axis(manual_map, axis=1) for name, df in pb_dfs.items()}

In [563]:
pb_frame = merge_station_dfs(pb_dfs.copy(), uf)

In [564]:
# Limpar linhas de datetime para aceitar o formato
pb_frame["DATETIME"] = (pb_frame["DATETIME"].astype(str).str.replace("T", " "))

In [569]:
# Criar planilha final 
pb_dfs = build_inventory_rows(pb_frame.copy(), df_cod, uf, datetime_col='DATETIME', mode="columns")
pb_dfs["PROPRIETARIO"] = pb_frame["PROPRIETARIO"]

/tmp/ipykernel_70933/4086501568.py:12: UserWarning: Parsing dates in %Y-%m-%d %H:%M:%S format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  s = pd.to_datetime(g[datetime_col], dayfirst=True)
/tmp/ipykernel_70933/4086501568.py:12: UserWarning: Parsing dates in %Y-%m-%d %H:%M:%S format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  s = pd.to_datetime(g[datetime_col], dayfirst=True)
/tmp/ipykernel_70933/4086501568.py:12: UserWarning: Parsing dates in %Y-%m-%d %H:%M:%S format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  s = pd.to_datetime(g[datetime_col], dayfirst=True)


In [574]:
# Adicionar colunas de latitude e longitude
lat_map = {
    "Estação 1": -7.034652778,
    "Estação 2": -7.086444444,
    "Estação 3": -7.047402778,
}
lon_map = {
    "Estação 1": -34.842980556,
    "Estação 2": -34.848941667,
    "Estação 3": -34.956544444,
}

pb_dfs["LATITUDE"]  = pb_dfs["ID_OEMA"].map(lat_map)
pb_dfs["LONGITUDE"] = pb_dfs["ID_OEMA"].map(lon_map)

In [572]:
pb_dfs = assign_id_mma(pb_dfs)
pb_dfs.head()

,UF,ID_OEMA,CIDADE,ID_MMA,ID_MMA_COMPLETO,POLUENTE,COD_POLUENTE,CD_MUN,COD_UF_IBGE,PROPRIETARIO,...,REALOCACAO,OBS_CALIBRACAO,INICIO,FIM,DADOS_MONITORAMENTO,RECONHECIDA,OBS_GERAIS,CERTIFICACAO,STATUS,REP_ESPACIAL_DECLARADA
0,PB,Estação 1,,PB0001,PB0001,"MP1 ,MP10,MP25,PTS",,,25,SUDEMA,...,,,2024,2024,,,,,,
1,PB,Estação 2,,PB0002,PB0002,"MP1 ,MP10,MP25,PTS",,,25,SUDEMA,...,,,2024,2024,,,,,,
2,PB,Estação 3,,PB0003,PB0003,"MP1 ,MP10,MP25,PTS",,,25,SUDEMA,...,,,2024,2024,,,,,,


In [573]:
save_UF_estacoes_csv(pb_dfs, uf)

Saved to: /home/nobre/Notebooks/RQAR_2025_book/data/DADOS_ESTACOES/PB_estacoes.csv


##### d) Pernambuco

1. Dados enviados pela UF

In [42]:
# Substituir pelo estado desejado
uf = "PE" 

In [43]:
# Conferir respostas do formulário 
pe_forms = forms[forms["Unidade da Federação: "].str.contains(uf, case=False, na=False)]
pe_forms.iloc[:, idxs].head()

,Unidade da Federação:,Em sua UF há estações de monitoramento da qualidade do ar com equipamentos certificados como de referência ou equivalentes? \n,"Em caso afirmativo, a UF é responsável pela operação de algumas dessas estações?",Informe a quantidade de estações de operação própria:,Em sua UF há estações instaladas como condicionante de licença ambiental?,Informe a quantidade de estações instaladas por licença ambiental:,Em sua UF há estações de monitoramento indicativas (com sensores de baixo custo)?,Informe a quantidade de estações indicativas:,"Em 2024, houve mudanças na rede de monitoramento da qualidade do ar na UF?","Em caso afirmativo, indique quais mudanças ocorreram:",Quais são as perspectivas de curto prazo (1 ano) para a rede de monitoramento?,Deseja incluir observações sobre a rede de monitoramento da qualidade do ar? \n,Os dados de monitoramento da qualidade do ar são disponibilizados ao público?,"Em caso afirmativo, informe o endereço eletrônico para acesso:",Os dados de monitoramento passam por algum tratamento e formatação antes de serem disponibilizados?,"Em caso afirmativo, informe o método utilizado: \n(Exemplo: validação automática com base em padrões da CETESB)"
16,Pernambuco (PE),Sim,Não,0,Sim,4,Sim,11,Não,Não se aplica – não existem estações na UF,Ampliação da rede estadual,As 11 estações de baixo custo são operadas pel...,Sim,https://monitorar.mma.gov.br/mapa,Sim,Validação automática realizada na concepção do...
33,Sergipe (SE),Não,Não se aplica - não existem estações de referê...,0,Não,0,Não,0,Não se aplica - não existem estações na UF,Não se aplica – não existem estações na UF,Início do monitoramento com estações de baixo ...,Existem monitoramento nas saídas das chaminés ...,Não se aplica - não existem estações na UF,NaN,Não se aplica - não existem estações na UF,X


In [44]:
# Importar arquivos do estado
base = Path.cwd().parent 
df_dir = base / "data" / "DADOS_BRUTOS" 
df_dir.mkdir(parents=True, exist_ok=True)

pe_dir = df_dir / uf
print(pe_dir)

/home/nobre/Notebooks/RQAR_2025_book/data/DADOS_BRUTOS/PE


In [89]:
pe_dfs = load_excels(pe_dir,sheets=0)

Loaded Multi Station Report _2019.xlsx [sheet 0]: (8774, 25)
Loaded Multi Station Report _2020.xlsx [sheet 0]: (8798, 25)
Loaded Multi Station Report _2021.xlsx [sheet 0]: (8774, 25)
Loaded Multi Station Report _2022.xlsx [sheet 0]: (8774, 25)
Loaded Multi Station Report _2023.xlsx [sheet 0]: (8774, 25)
Loaded Multi Station Report_2024.xlsx [sheet 0]: (8798, 25)


In [90]:
from IPython.display import display as ipy_display

for name, df in pe_dfs.items():
    print(f"\n=== {name} ===")
    ipy_display(df.head())


=== Multi Station Report _2019 ===


,"Type: Average, Time Base: Hourly, Yearly: 1/1/2019 - 12/31/2019",Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,...,Unnamed: 15,Unnamed: 16,Unnamed: 17,Unnamed: 18,Unnamed: 19,Unnamed: 20,Unnamed: 21,Unnamed: 22,Unnamed: 23,Unnamed: 24
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Date Time,RNEST ESCOLA IPOJUCA,NaN,NaN,NaN,NaN,NaN,RNEST EDCUPE,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,RNEST CPRH,NaN,NaN,NaN,NaN
2,NaN,PM10_1Hr,O3_ug/m3,NO2_ug/m3,NH3_ug/m3,SO2_ug/m3,CO_ppm,PM10_1Hr,NO2_ug/m3,SO2_ug/m3,...,CO_ppm,SO2_ug/m3,O3_ug/m3,NO2_ug/m3,H2S_ug/m3,CO_ppm,PM10,SO2_ug/m3,NO2_ug/m3,O3_ug/m3
3,NaN,ug/m3,ug/m3,ug/m3,ug/m3,ug/m3,ppm,ug/m3,ug/m3,ug/m3,...,ppm,ug/m3,ug/m3,ug/m3,ug/m3,ppm,ug/m3,ug/m3,ug/m3,ug/m3
4,01:00 1/1/2019,25,23.95,10.45,4.7,14.54,0.41,2,0.27,5.14,...,NoData,NoData,NoData,NaN,NaN,NaN,NaN,NaN,NaN,NaN



=== Multi Station Report _2020 ===


,"Type: Average, Time Base: Hourly, Yearly: 1/1/2020 - 12/31/2020",Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,...,Unnamed: 15,Unnamed: 16,Unnamed: 17,Unnamed: 18,Unnamed: 19,Unnamed: 20,Unnamed: 21,Unnamed: 22,Unnamed: 23,Unnamed: 24
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Date Time,RNEST ESCOLA IPOJUCA,NaN,NaN,NaN,NaN,NaN,RNEST EDCUPE,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,RNEST CPRH,NaN,NaN,NaN,NaN
2,NaN,PM10_1Hr,O3_ug/m3,NO2_ug/m3,NH3_ug/m3,SO2_ug/m3,CO_ppm,PM10_1Hr,NO2_ug/m3,SO2_ug/m3,...,CO_ppm,SO2_ug/m3,O3_ug/m3,NO2_ug/m3,H2S_ug/m3,CO_ppm,PM10,SO2_ug/m3,NO2_ug/m3,O3_ug/m3
3,NaN,ug/m3,ug/m3,ug/m3,ug/m3,ug/m3,ppm,ug/m3,ug/m3,ug/m3,...,ppm,ug/m3,ug/m3,ug/m3,ug/m3,ppm,ug/m3,ug/m3,ug/m3,ug/m3
4,01:00 1/1/2020,17,13.24,15.28,7.93,4.16,0.4,NoData,NoData,NoData,...,0.32,5.16,32.63,NoData,4.22,0.24,NoData,NoData,NoData,NoData



=== Multi Station Report _2021 ===


...

,"Type: Average, Time Base: Hourly, Yearly: 1/1/2022 - 12/31/2022",Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,...,Unnamed: 15,Unnamed: 16,Unnamed: 17,Unnamed: 18,Unnamed: 19,Unnamed: 20,Unnamed: 21,Unnamed: 22,Unnamed: 23,Unnamed: 24
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Date Time,RNEST ESCOLA IPOJUCA,NaN,NaN,NaN,NaN,NaN,RNEST EDCUPE,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,RNEST CPRH,NaN,NaN,NaN,NaN
2,NaN,PM10_1Hr,O3_ug/m3,NO2_ug/m3,NH3_ug/m3,SO2_ug/m3,CO_ppm,PM10_1Hr,NO2_ug/m3,SO2_ug/m3,...,CO_ppm,SO2_ug/m3,O3_ug/m3,NO2_ug/m3,H2S_ug/m3,CO_ppm,PM10,SO2_ug/m3,NO2_ug/m3,O3_ug/m3
3,NaN,ug/m3,ug/m3,ug/m3,ug/m3,ug/m3,ppm,ug/m3,ug/m3,ug/m3,...,ppm,ug/m3,ug/m3,ug/m3,ug/m3,ppm,ug/m3,ug/m3,ug/m3,ug/m3
4,01:00 1/1/2022,75.26,34.507,4.873,1.1,1.03,0.383,21.09,2.098,1.799,...,0.099,BelowR,30.842,6.939,BelowR,0,9,3.97,0.817,22.229



=== Multi Station Report _2023 ===


,"Type: Average, Time Base: Hourly, Yearly: 1/1/2023 - 12/31/2023",Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,...,Unnamed: 15,Unnamed: 16,Unnamed: 17,Unnamed: 18,Unnamed: 19,Unnamed: 20,Unnamed: 21,Unnamed: 22,Unnamed: 23,Unnamed: 24
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Date Time,RNEST ESCOLA IPOJUCA,NaN,NaN,NaN,NaN,NaN,RNEST EDCUPE,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,RNEST CPRH,NaN,NaN,NaN,NaN
2,NaN,PM10_1Hr,O3_ug/m3,NO2_ug/m3,NH3_ug/m3,SO2_ug/m3,CO_ppm,PM10_1Hr,NO2_ug/m3,SO2_ug/m3,...,CO_ppm,SO2_ug/m3,O3_ug/m3,NO2_ug/m3,H2S_ug/m3,CO_ppm,PM10,SO2_ug/m3,NO2_ug/m3,O3_ug/m3
3,NaN,ug/m3,ug/m3,ug/m3,ug/m3,ug/m3,ppm,ug/m3,ug/m3,ug/m3,...,ppm,ug/m3,ug/m3,ug/m3,ug/m3,ppm,ug/m3,ug/m3,ug/m3,ug/m3
4,01:00 1/1/2023,12.34,14.437,5.783,0.436,13.797,0.311,6.7,2.935,0.581,...,0.096,0.956,27.068,7.399,1.38,NoData,18.1,3.577,6.019,32.458



=== Multi Station Report_2024 ===


,"Type: Average, Time Base: Hourly, Yearly: 1/1/2024 - 12/31/2024",Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,...,Unnamed: 15,Unnamed: 16,Unnamed: 17,Unnamed: 18,Unnamed: 19,Unnamed: 20,Unnamed: 21,Unnamed: 22,Unnamed: 23,Unnamed: 24
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Date Time,RNEST ESCOLA IPOJUCA,NaN,NaN,NaN,NaN,NaN,RNEST EDCUPE,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,RNEST CPRH,NaN,NaN,NaN,NaN
2,NaN,PM10_1Hr,O3_ug/m3,NO2_ug/m3,NH3_ug/m3,SO2_ug/m3,CO_ppm,PM10_1Hr,NO2_ug/m3,SO2_ug/m3,...,CO_ppm,SO2_ug/m3,O3_ug/m3,NO2_ug/m3,H2S_ug/m3,CO_ppm,PM10,SO2_ug/m3,NO2_ug/m3,O3_ug/m3
3,NaN,ug/m3,ug/m3,ug/m3,ug/m3,ug/m3,ppm,ug/m3,ug/m3,ug/m3,...,ppm,ug/m3,ug/m3,ug/m3,ug/m3,ppm,ug/m3,ug/m3,ug/m3,ug/m3
4,01:00 1/1/2024,NoData,NoData,NoData,NoData,NoData,NoData,27.9,3.643,1.934,...,0.226,2.679,4.656,10.74,1.919,NoData,25,2.545,InVld,38.996


In [91]:
# Limpar dataframes pois contém informações extras nas linhas que não são utilizadas

bad_rows = {
    "Data Precent","Avg","STD","Num",
    "Maximum","Max Date","Max Time",
    "Minimum","Min Date","Min Time",
}

for name, df in pe_dfs.items():
    first_col = df.columns[0]
    mask = df[first_col].astype(str).str.strip().isin(bad_rows)
    pe_dfs[name] = df.loc[~mask].copy()

In [92]:
# Criar um dataframe por df dentro do dict
pe_out = {} 

for name, df in pe_dfs.copy().items():

    # 1) find the header start row
    idx = df.index[df.iloc[:, 0].astype(str).str.strip().eq("Date Time")]
    start = int(idx[0]) if len(idx) else 0
    sub = df.iloc[start:].reset_index(drop=True)
    
    # df is your raw frame
    station = df.iloc[1].ffill()          # row 1, forward-fill across columns
    pollutant = df.iloc[2].astype(str)                # row 2
    
    new_cols = ["DATETIME"] + [f"{st}|{po}" for st, po in zip(station[1:], pollutant[1:])]

    out = sub.iloc[4:].copy()                       # data rows
    out.columns = new_cols

    pe_out[name] = out  

In [93]:
pe_frame = {
    name: fix_datetime_df(d, col="DATETIME", keep_dt_col=False)
    for name, d in pe_out.copy().items()
}

Nota: Nesse caso, as planilhas de estações seguem ordem cronológica de datetime, então o merge precisa conferir se a estação já existia ou se foi criada em um novo ano.

In [65]:
def coalesce_duplicate_columns(df: pd.DataFrame) -> pd.DataFrame:
    """If the same column name appears multiple times, keep one and fill with first non-null."""
    counts = Counter(df.columns)
    dups = [c for c, n in counts.items() if n > 1]
    for c in dups:
        cols = [k for k in df.columns if k == c]
        df[c] = df[cols].bfill(axis=1).iloc[:, 0]  # take first non-null left-to-right
        df.drop(columns=cols[1:], inplace=True)
    return df

def merge_by_datetime_union(dfs_dict: dict, keep_source=True):
    frames = []
    for name, df in dfs_dict.items():
        t = df.copy()
        if keep_source:
            t["__source"] = name  # year or filename

        frames.append(t)

    if not frames:
        return pd.DataFrame()

    out = pd.concat(frames, axis=0, ignore_index=True, sort=True)  # union of columns
    out = coalesce_duplicate_columns(out)

    # keep DATETIME first if present
    if "DATETIME" in out.columns:
        cols = ["DATETIME"] + [c for c in out.columns if c != "DATETIME"]
        out = out[cols]

    return out

In [95]:
pe_all = merge_by_datetime_union(pe_frame.copy(), keep_source=True)
pe_all

,DATETIME,RNEST CPRH|CO_ppm,RNEST CPRH|NO2_ug/m3,RNEST CPRH|O3_ug/m3,RNEST CPRH|PM10,RNEST CPRH|SO2_ug/m3,RNEST EDCUPE|CO_ppm,RNEST EDCUPE|H2S_ug/m3,RNEST EDCUPE|NH3_ug/m3,RNEST EDCUPE|NO2_ug/m3,...,RNEST ESCOLA IPOJUCA|O3_ug/m3,RNEST ESCOLA IPOJUCA|PM10_1Hr,RNEST ESCOLA IPOJUCA|SO2_ug/m3,RNEST IFPE|CO_ppm,RNEST IFPE|H2S_ug/m3,RNEST IFPE|NO2_ug/m3,RNEST IFPE|O3_ug/m3,RNEST IFPE|PM10,RNEST IFPE|SO2_ug/m3,__source
0,01/01/2019 02:00,NaN,NaN,NaN,NaN,NaN,1.92,28.49,0.31,0.27,...,24.22,17,15.58,NoData,NaN,NaN,NoData,NoData,NoData,Multi Station Report _2019
1,01/01/2019 03:00,NaN,NaN,NaN,NaN,NaN,4.79,23.78,0.31,0.29,...,26.02,21,15.1,NoData,NaN,NaN,NoData,NoData,NoData,Multi Station Report _2019
2,01/01/2019 04:00,NaN,NaN,NaN,NaN,NaN,8.99,18.19,0.33,0.3,...,24.89,17,14.95,NoData,NaN,NaN,NoData,NoData,NoData,Multi Station Report _2019
3,01/01/2019 05:00,NaN,NaN,NaN,NaN,NaN,9.99,10.68,0.33,0.35,...,23.13,16,14.69,NoData,NaN,NaN,NoData,NoData,NoData,Multi Station Report _2019
4,01/01/2019 06:00,NaN,NaN,NaN,NaN,NaN,10.54,6.37,0.32,0.41,...,16.94,22,14.65,NoData,NaN,NaN,NoData,NoData,NoData,Multi Station Report _2019
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
52597,31/12/2024 20:00,0.536,5.977,33.563,27.2,2.271,0.23,7.792,1.086,3.497,...,13.115,42.06,5.282,0.07,0.679,Maintain,13.1,26.93,2.124,Multi Station Report_2024
52598,31/12/2024 21:00,0.586,4.141,24.413,32.6,1.626,0.226,7.185,0.777,3.899,...,12.967,34.04,5.09,0.054,1.034,Maintain,13.244,18.33,1.97,Multi Station Report_2024
52599,31/12/2024 22:00,0.583,4.532,18.867,23.9,1.637,0.218,5.131,0.757,3.39,...,13.802,31.75,4.937,0.061,2.259,Maintain,14.253,11.31,1.864,Multi Station Report_2024
52600,31/12/2024 23:00,0.624,5.767,19.587,22.1,1.453,0.246,5.039,0.79,3.582,...,12.195,31.09,4.722,0.051,3.026,Maintain,13.887,10.38,1.927,Multi Station Report_2024


In [66]:
def make_station_layout(df_or_dict, keep_source=True):
  # accept dicts of DataFrames too
    df = (pd.concat(df_or_dict.values(), ignore_index=True, sort=False)
          if isinstance(df_or_dict, dict) else df_or_dict)

    id_vars = ["DATETIME"]
    if keep_source and "__source" in df.columns:
        id_vars.append("__source")

    long = df.melt(id_vars=id_vars, var_name="pair", value_name="VALUE")
    long = long[long["pair"].astype(str).str.contains(r"\|", na=False)].copy()

    # critical line: force string and split on literal "|"
    long["pair"] = long["pair"].astype("string")
    long[["ESTACAO", "POLLUTANT"]] = long["pair"].str.split(
        pat="|", n=1, expand=True, regex=False
    )
    long.drop(columns=["pair"], inplace=True)
    long["ESTACAO"] = long["ESTACAO"].str.strip()
    long["POLLUTANT"] = long["POLLUTANT"].str.strip()

    index_cols = ["DATETIME", "ESTACAO"] + (["__source"] if "__source" in id_vars else [])
    wide = (
        long.pivot_table(index=index_cols, columns="POLLUTANT", values="VALUE", aggfunc="first")
            .reset_index()
            .sort_values(["DATETIME", "ESTACAO"], kind="stable")
    )
    wide.columns.name = None
    return wide

In [97]:
pe_all = make_station_layout(pe_all, keep_source=True)  # keeps "NoData"
pe_all.head()

,DATETIME,ESTACAO,__source,CO_ppm,H2S_ug/m3,NH3_ug/m3,NO2_ug/m3,O3_ug/m3,PM10,PM10_1Hr,SO2_ug/m3
0,01/01/2019 02:00,RNEST EDCUPE,Multi Station Report _2019,1.92,28.49,0.31,0.27,2.83,NaN,18,4.83
1,01/01/2019 02:00,RNEST ESCOLA IPOJUCA,Multi Station Report _2019,0.41,NaN,4.71,11.13,24.22,NaN,17,15.58
2,01/01/2019 02:00,RNEST IFPE,Multi Station Report _2019,NoData,NaN,NaN,NaN,NoData,NoData,NaN,NoData
3,01/01/2019 03:00,RNEST EDCUPE,Multi Station Report _2019,4.79,23.78,0.31,0.29,4.92,NaN,8,5.04
4,01/01/2019 03:00,RNEST ESCOLA IPOJUCA,Multi Station Report _2019,0.35,NaN,4.92,9.91,26.02,NaN,21,15.1


Nota: Nota-se que existem anos que não há medição por poluente, discriminado como NoData. Neste caso, não podemos contabilizar o poluente na estação, por isso deve-se aplicar um filtro.

In [106]:
pe_station = build_inventory_rows(pe_all.copy(), df_cod, uf, datetime_col='DATETIME', mode="columns", drop_after_underscore=True)
pe_station

,UF,ID_OEMA,CIDADE,ID_MMA,ID_MMA_COMPLETO,POLUENTE,COD_POLUENTE,CD_MUN,COD_UF_IBGE,PROPRIETARIO,...,REALOCACAO,OBS_CALIBRACAO,INICIO,FIM,DADOS_MONITORAMENTO,RECONHECIDA,OBS_GERAIS,CERTIFICACAO,STATUS,REP_ESPACIAL_DECLARADA
0,PE,RNEST CPRH,,,,"CO,H2S,MP10,NH3,NO2,O3,SO2",,,26,,...,,,2020,2024,,,,,,
1,PE,RNEST EDCUPE,,,,"CO,H2S,MP10,NH3,NO2,O3,SO2",,,26,,...,,,2019,2024,,,,,,
2,PE,RNEST ESCOLA IPOJUCA,,,,"CO,H2S,MP10,NH3,NO2,O3,SO2",,,26,,...,,,2019,2024,,,,,,
3,PE,RNEST IFPE,,,,"CO,H2S,MP10,NH3,NO2,O3,SO2",,,26,,...,,,2019,2024,,,,,,


In [109]:
pe_station = assign_id_mma(pe_station)

In [108]:
save_UF_estacoes_csv(pe_station, uf)

Saved to: /home/nobre/Notebooks/RQAR_2025_book/data/DADOS_ESTACOES/PE_estacoes.csv


##### e) Roraima

In [140]:
# Substituir pelo estado desejado
uf = "RR" 

In [141]:
# Conferir respostas do formulário 
rr_forms = forms[forms["Unidade da Federação: "].str.contains("RR" , case=False, na=False)]
rr_forms.iloc[:, idxs].head()

,Unidade da Federação:,Em sua UF há estações de monitoramento da qualidade do ar com equipamentos certificados como de referência ou equivalentes? \n,"Em caso afirmativo, a UF é responsável pela operação de algumas dessas estações?",Informe a quantidade de estações de operação própria:,Em sua UF há estações instaladas como condicionante de licença ambiental?,Informe a quantidade de estações instaladas por licença ambiental:,Em sua UF há estações de monitoramento indicativas (com sensores de baixo custo)?,Informe a quantidade de estações indicativas:,"Em 2024, houve mudanças na rede de monitoramento da qualidade do ar na UF?","Em caso afirmativo, indique quais mudanças ocorreram:",Quais são as perspectivas de curto prazo (1 ano) para a rede de monitoramento?,Deseja incluir observações sobre a rede de monitoramento da qualidade do ar? \n,Os dados de monitoramento da qualidade do ar são disponibilizados ao público?,"Em caso afirmativo, informe o endereço eletrônico para acesso:",Os dados de monitoramento passam por algum tratamento e formatação antes de serem disponibilizados?,"Em caso afirmativo, informe o método utilizado: \n(Exemplo: validação automática com base em padrões da CETESB)"
27,Roraima (RR),Não,Não se aplica - não existem estações de referê...,0,Não,0,Sim,2,Não se aplica - não existem estações na UF,Não se aplica – não existem estações na UF,Não há perspectivas de curto prazo,NaN,Não se aplica - não existem estações na UF,NaN,Não se aplica - não existem estações na UF,x


In [142]:
# Importar arquivos do estado
base = Path.cwd().parent 
df_dir = base / "data" / "DADOS_BRUTOS" 
df_dir.mkdir(parents=True, exist_ok=True)
print(df_dir)

rr_dir = df_dir / uf
os.listdir(rr_dir)

/home/nobre/Notebooks/RQAR_2025_book/data/DADOS_BRUTOS


['FEMARH-ENEVA.xlsx', 'Medições (29).xlsx', 'FAZENDA-ENEVA.xlsx']

In [143]:
rr_dfs = load_excels(rr_dir)

Loaded FAZENDA-ENEVA.xlsx [sheet 0]: (6576, 31)
Loaded FEMARH-ENEVA.xlsx [sheet 0]: (7296, 37)
Loaded Medições (29).xlsx [sheet 0]: (30542, 101)


In [144]:
for name, df in rr_dfs.items():
    print(f"\n=== {name} ===")
    display(df.head())


=== FAZENDA-ENEVA ===


,Data,PM25,Status_PM25,PM10,Status_PM10,CO,Status_CO,SO2,Status_SO2,O3,...,DirVento,Status_DirVento,RH,Status_RH,Rain,Status_Rain,Radiação,Status_Radiação,Press_Ar,Status_Press_Ar
0,2024-12-01 00:00:00,0.009,Ok,-9999.0,InVld,0.119,Ok,0.182,Ok,6.25,...,-9999.0,InVld,-9999.0,InVld,-9999.0,InVld,-9999.0,InVld,-9999.0,InVld
1,2024-12-01 01:00:00,7.000,Ok,-9999.0,InVld,0.124,Ok,0.095,Ok,5.93,...,-9999.0,InVld,-9999.0,InVld,-9999.0,InVld,-9999.0,InVld,-9999.0,InVld
2,2024-12-01 02:00:00,6.000,Ok,-9999.0,InVld,0.132,Ok,0.290,Ok,6.02,...,-9999.0,InVld,-9999.0,InVld,-9999.0,InVld,-9999.0,InVld,-9999.0,InVld
3,2024-12-01 03:00:00,17.000,Ok,-9999.0,InVld,0.153,Ok,0.310,Ok,5.57,...,-9999.0,InVld,-9999.0,InVld,-9999.0,InVld,-9999.0,InVld,-9999.0,InVld
4,2024-12-01 04:00:00,22.000,Ok,-9999.0,InVld,0.100,Ok,0.300,Ok,3.82,...,-9999.0,InVld,-9999.0,InVld,-9999.0,InVld,-9999.0,InVld,-9999.0,InVld



=== FEMARH-ENEVA ===


,Data,TempAr,Status_TempAr,VelVento,Status_VelVento,DirVento,Status_DirVento,RH,Status_RH,Rain,...,NMHC,Status_NMHC,CH4,Status_CH4,NO,Status_NO,NO2,Status_NO2,NOX,Status_NOX
0,2024-11-01 00:00:00,28.49,Ok,0.00,Ok,79.78,Ok,82.47,Ok,0.0,...,0.93,Ok,1.21,Ok,-9999.00,NoData,-9999.00,NoData,-9999.00,NoData
1,2024-11-01 01:00:00,28.53,Ok,0.00,Ok,108.00,Ok,85.42,Ok,0.0,...,0.94,Ok,1.25,Ok,2.98,Ok,0.25,Ok,3.23,Ok
2,2024-11-01 02:00:00,28.94,Ok,0.07,Ok,86.32,Ok,77.41,Ok,0.0,...,0.94,Ok,1.18,Ok,2.47,Ok,0.31,Ok,2.78,Ok
3,2024-11-01 03:00:00,28.25,Ok,0.13,Ok,79.69,Ok,79.34,Ok,0.0,...,0.93,Ok,1.17,Ok,2.84,Ok,0.25,Ok,3.09,Ok
4,2024-11-01 04:00:00,27.43,Ok,0.15,Ok,78.89,Ok,82.10,Ok,0.0,...,0.94,Ok,1.16,Ok,2.61,Ok,0.12,Ok,2.73,Ok



=== Medições (29) ===


,Data e Hora,AZULÃO GERAÇÃO DE ENERGIA S.A. - ENEVA S.A. - UTE Jaguatirica II,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,...,Unnamed: 91,Unnamed: 92,Unnamed: 93,Unnamed: 94,Unnamed: 95,Unnamed: 96,Unnamed: 97,Unnamed: 98,Unnamed: 99,Unnamed: 100
0,NaT,Estação Fazenda Carolina,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaT,Qualidade do Ar,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaT,Ar Ambiente,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaT,"Partículas Respiráveis (<2,5µm)",NaN,Monóxido de Carbono,NaN,NaN,NaN,Dióxido de Nitrogênio,NaN,NaN,...,Direção Escalar do Vento,NaN,Umidade Relativa,NaN,Radiação Solar Global,NaN,Pressão Atmosférica,NaN,Precipitação Pluviométrica,NaN
4,NaT,Média:1 h/ Freq.: horária/ Alt.: 4.0 m,NaN,Média:1 h/ Freq.: horária/ Alt.: 3.0 m,NaN,Média:1 h/ Freq.: horária/ Alt.: 3.0 m,NaN,Média:1 h/ Freq.: horária/ Alt.: 3.0 m,NaN,Média:1 h/ Freq.: horária/ Alt.: 3.0 m,...,Média:1 h/ Freq.: horária/ Alt.: 10.0 m,NaN,Média:1 h/ Freq.: horária/ Alt.: 1.0 m,NaN,Média:1 h/ Freq.: horária/ Alt.: 10.0 m,NaN,Média:1 h/ Freq.: horária/ Alt.: 1.0 m,NaN,Média:1 h/ Freq.: horária/ Alt.: 3.0 m,NaN


Nota: Como os 3 dataframes possuem formatos diferentes, é necessário tratá-los conforme suas particularidades

In [145]:
# 1) Fazer alterações somente no df Medições (29) dentro de rr_dfs dict
data = rr_dfs.copy()
key = "Medições (29)"              # exact key in rr_dfs
df = data[key].copy()

# 1) find the header start row
idx = df.index[df.iloc[:, 0].astype(str).str.strip().eq("Data e Hora")]
start = int(idx[0]) if len(idx) else 0
sub = df.iloc[start:].reset_index(drop=True)

# df is your raw frame
station = df.iloc[0].ffill()          # row 1, forward-fill across columns
pollutant = df.iloc[3].astype(str)                # row 2

new_cols = ["DATETIME"] + [f"{st}|{po}" for st, po in zip(station[1:], pollutant[1:])]

out = sub.iloc[7:].copy()                       # data rows
out.columns = new_cols
out["__source"] = key

data[key]=out
out

,DATETIME,"Estação Fazenda Carolina|Partículas Respiráveis (<2,5µm)",Estação Fazenda Carolina|nan,Estação Fazenda Carolina|Monóxido de Carbono,Estação Fazenda Carolina|nan,Estação Fazenda Carolina|nan,Estação Fazenda Carolina|nan,Estação Fazenda Carolina|Dióxido de Nitrogênio,Estação Fazenda Carolina|nan,Estação Fazenda Carolina|nan,...,Estação FEMARH|nan,Estação FEMARH|Umidade Relativa,Estação FEMARH|nan,Estação FEMARH|Radiação Solar Global,Estação FEMARH|nan,Estação FEMARH|Pressão Atmosférica,Estação FEMARH|nan,Estação FEMARH|Precipitação Pluviométrica,Estação FEMARH|nan,__source
7,2021-06-13 19:30:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Medições (29)
8,2021-06-14 16:30:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,IE,113.7,IE,1298,IE,1016.9,IE,0,IE,Medições (29)
9,2021-06-14 17:30:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,IU,69.6,,34.6,,1000,,0,,Medições (29)
10,2021-06-14 18:30:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,IU,69.1,,2.6,,1000.5,,0,,Medições (29)
11,2021-06-14 19:30:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,IU,68.1,,1.7,,1001.4,,0,,Medições (29)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
30537,2024-12-19 11:30:00,5,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Medições (29)
30538,2024-12-19 12:30:00,7,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Medições (29)
30539,2024-12-19 13:30:00,5,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Medições (29)
30540,2024-12-19 14:30:00,2,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Medições (29)


In [146]:
# 2) Fazer alterações nos dfs "FAZENDA-ENEVA" e "FEMARH-ENEVA" dentro de rr_dfs dict
manual_map = {"Data": "DATETIME"}

for key in ["FAZENDA-ENEVA", "FEMARH-ENEVA"]:
    val = data.get(key)
    if val is None:
        continue

    if isinstance(val, dict):
        # rename inside nested dict
        new = {}
        for name, obj in val.items():
            if isinstance(obj, pd.DataFrame):
                new[name] = obj.rename(columns=manual_map)
        data[key] = new

    elif isinstance(val, pd.DataFrame):
        data[key] = val.rename(columns=manual_map)

    elif isinstance(val, pd.Series):
        data[key] = val.rename("DATETIME") if val.name == "Data" else val

In [147]:
for name, df in data.items():
    print(f"\n=== {name} ===")
    display(df.head())


=== FAZENDA-ENEVA ===


,DATETIME,PM25,Status_PM25,PM10,Status_PM10,CO,Status_CO,SO2,Status_SO2,O3,...,DirVento,Status_DirVento,RH,Status_RH,Rain,Status_Rain,Radiação,Status_Radiação,Press_Ar,Status_Press_Ar
0,2024-12-01 00:00:00,0.009,Ok,-9999.0,InVld,0.119,Ok,0.182,Ok,6.25,...,-9999.0,InVld,-9999.0,InVld,-9999.0,InVld,-9999.0,InVld,-9999.0,InVld
1,2024-12-01 01:00:00,7.000,Ok,-9999.0,InVld,0.124,Ok,0.095,Ok,5.93,...,-9999.0,InVld,-9999.0,InVld,-9999.0,InVld,-9999.0,InVld,-9999.0,InVld
2,2024-12-01 02:00:00,6.000,Ok,-9999.0,InVld,0.132,Ok,0.290,Ok,6.02,...,-9999.0,InVld,-9999.0,InVld,-9999.0,InVld,-9999.0,InVld,-9999.0,InVld
3,2024-12-01 03:00:00,17.000,Ok,-9999.0,InVld,0.153,Ok,0.310,Ok,5.57,...,-9999.0,InVld,-9999.0,InVld,-9999.0,InVld,-9999.0,InVld,-9999.0,InVld
4,2024-12-01 04:00:00,22.000,Ok,-9999.0,InVld,0.100,Ok,0.300,Ok,3.82,...,-9999.0,InVld,-9999.0,InVld,-9999.0,InVld,-9999.0,InVld,-9999.0,InVld



=== FEMARH-ENEVA ===


,DATETIME,TempAr,Status_TempAr,VelVento,Status_VelVento,DirVento,Status_DirVento,RH,Status_RH,Rain,...,NMHC,Status_NMHC,CH4,Status_CH4,NO,Status_NO,NO2,Status_NO2,NOX,Status_NOX
0,2024-11-01 00:00:00,28.49,Ok,0.00,Ok,79.78,Ok,82.47,Ok,0.0,...,0.93,Ok,1.21,Ok,-9999.00,NoData,-9999.00,NoData,-9999.00,NoData
1,2024-11-01 01:00:00,28.53,Ok,0.00,Ok,108.00,Ok,85.42,Ok,0.0,...,0.94,Ok,1.25,Ok,2.98,Ok,0.25,Ok,3.23,Ok
2,2024-11-01 02:00:00,28.94,Ok,0.07,Ok,86.32,Ok,77.41,Ok,0.0,...,0.94,Ok,1.18,Ok,2.47,Ok,0.31,Ok,2.78,Ok
3,2024-11-01 03:00:00,28.25,Ok,0.13,Ok,79.69,Ok,79.34,Ok,0.0,...,0.93,Ok,1.17,Ok,2.84,Ok,0.25,Ok,3.09,Ok
4,2024-11-01 04:00:00,27.43,Ok,0.15,Ok,78.89,Ok,82.10,Ok,0.0,...,0.94,Ok,1.16,Ok,2.61,Ok,0.12,Ok,2.73,Ok



=== Medições (29) ===


,DATETIME,"Estação Fazenda Carolina|Partículas Respiráveis (<2,5µm)",Estação Fazenda Carolina|nan,Estação Fazenda Carolina|Monóxido de Carbono,Estação Fazenda Carolina|nan,Estação Fazenda Carolina|nan,Estação Fazenda Carolina|nan,Estação Fazenda Carolina|Dióxido de Nitrogênio,Estação Fazenda Carolina|nan,Estação Fazenda Carolina|nan,...,Estação FEMARH|nan,Estação FEMARH|Umidade Relativa,Estação FEMARH|nan,Estação FEMARH|Radiação Solar Global,Estação FEMARH|nan,Estação FEMARH|Pressão Atmosférica,Estação FEMARH|nan,Estação FEMARH|Precipitação Pluviométrica,Estação FEMARH|nan,__source
7,2021-06-13 19:30:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Medições (29)
8,2021-06-14 16:30:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,IE,113.7,IE,1298,IE,1016.9,IE,0,IE,Medições (29)
9,2021-06-14 17:30:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,IU,69.6,,34.6,,1000,,0,,Medições (29)
10,2021-06-14 18:30:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,IU,69.1,,2.6,,1000.5,,0,,Medições (29)
11,2021-06-14 19:30:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,IU,68.1,,1.7,,1001.4,,0,,Medições (29)


In [148]:
rr_frame = {
    name: fix_datetime_df(d, col="DATETIME", keep_dt_col=False)
    for name, d in data.copy().items()
}

In [149]:
# 1) Fazer alterações somente no df Medições (29) dentro de rr_dfs dict
key = "Medições (29)"              
df = rr_frame[key].copy()

medicoes = make_station_layout(df, keep_source=True)

In [151]:
medicoes

,DATETIME,ESTACAO,__source,Direção Escalar do Vento,Dióxido de Enxofre,Dióxido de Nitrogênio,Frequência Elétrica de Entrada,Hidrocarbonetos Não Metano,Hidrocarbonetos Totais,Metano,...,"Partículas Respiráveis (<2,5µm)",Precipitação Pluviométrica,Pressão Atmosférica,Radiação Solar Global,Temperatura,Tensão Elétrica,Umidade Relativa,Velocidade Escalar do Vento,nan,Óxidos de Nitrogênio
0,01/01/2022 00:30,Estação FEMARH,Medições (29),72,0.00107,0.0001,60,-9999,-9999,-9999,...,17,0,1001.4,0,30,217.9,74.9,2.5,,0.0007
1,01/01/2022 00:30,Estação Fazenda Carolina,Medições (29),72.8,0.00067,0.0004,60,0.44,2.79,2.35,...,18,0,1004.1,0,25.8,NaN,78,2.4,,0.0026
2,01/01/2022 01:30,Estação FEMARH,Medições (29),71.9,0.00109,0,60,-9999,-9999,-9999,...,17,0,1001.4,0,29.4,218.3,76.7,2.3,VU,0.0006
3,01/01/2022 01:30,Estação Fazenda Carolina,Medições (29),73.8,0.00072,0.0004,60,0.44,2.8,2.36,...,15,0,1004.1,0,26,NaN,80.1,1.4,,0.0027
4,01/01/2022 02:30,Estação FEMARH,Medições (29),71.5,0.00108,0,60,-9999,-9999,-9999,...,13,0,1000.9,0,29.1,217.8,78.5,2.4,,0.0007
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
58356,31/12/2023 21:30,Estação Fazenda Carolina,Medições (29),64.5,0.00023,0.0006,60,NaN,NaN,NaN,...,985,0,1004.5,4.3,26.3,NaN,57.6,2.2,IU,0.0018
58357,31/12/2023 22:30,Estação FEMARH,Medições (29),69.1,0.00116,NaN,60,NaN,NaN,NaN,...,9,0,1001.7,0,24.5,215.1,70,1.7,,NaN
58358,31/12/2023 22:30,Estação Fazenda Carolina,Medições (29),56.4,0.00028,0.0007,60,NaN,NaN,NaN,...,985,0,1005.3,4.3,26.2,NaN,59.1,3.6,IU,0.0019
58359,31/12/2023 23:30,Estação FEMARH,Medições (29),69.5,0.00107,NaN,60,NaN,NaN,NaN,...,6,0,1001.5,0,24.9,214.2,73.1,1.8,,NaN


In [160]:
rr_station = build_inventory_rows(medicoes.copy(), df_cod, uf, datetime_col='DATETIME', mode="columns", drop_after_underscore=True)

In [161]:
rr_station = assign_id_mma(rr_station)
rr_station

,UF,ID_OEMA,CIDADE,ID_MMA,ID_MMA_COMPLETO,POLUENTE,COD_POLUENTE,CD_MUN,COD_UF_IBGE,PROPRIETARIO,...,REALOCACAO,OBS_CALIBRACAO,INICIO,FIM,DADOS_MONITORAMENTO,RECONHECIDA,OBS_GERAIS,CERTIFICACAO,STATUS,REP_ESPACIAL_DECLARADA
0,RR,Estação Fazenda Carolina,,RR0001,RR0001,"CH4,CO,HCT,NO,NO2,NOX,O3,SO2",,,14,,...,,,2021,2024,,,,,,
1,RR,Estação FEMARH,,RR0002,RR0002,"CH4,CO,HCT,NO,NO2,NOX,O3,SO2",,,14,,...,,,2021,2024,,,,,,


In [162]:
save_UF_estacoes_csv(rr_station, uf)

Saved to: /home/nobre/Notebooks/RQAR_2025_book/data/DADOS_ESTACOES/RR_estacoes.csv


##### f) Acre

In [164]:
# Substituir pelo estado desejado
uf = "AC" 

In [165]:
# Conferir respostas do formulário 
ac_forms = forms[forms["Unidade da Federação: "].str.contains(uf , case=False, na=False)]
ac_forms.iloc[:, idxs].head()

,Unidade da Federação:,Em sua UF há estações de monitoramento da qualidade do ar com equipamentos certificados como de referência ou equivalentes? \n,"Em caso afirmativo, a UF é responsável pela operação de algumas dessas estações?",Informe a quantidade de estações de operação própria:,Em sua UF há estações instaladas como condicionante de licença ambiental?,Informe a quantidade de estações instaladas por licença ambiental:,Em sua UF há estações de monitoramento indicativas (com sensores de baixo custo)?,Informe a quantidade de estações indicativas:,"Em 2024, houve mudanças na rede de monitoramento da qualidade do ar na UF?","Em caso afirmativo, indique quais mudanças ocorreram:",Quais são as perspectivas de curto prazo (1 ano) para a rede de monitoramento?,Deseja incluir observações sobre a rede de monitoramento da qualidade do ar? \n,Os dados de monitoramento da qualidade do ar são disponibilizados ao público?,"Em caso afirmativo, informe o endereço eletrônico para acesso:",Os dados de monitoramento passam por algum tratamento e formatação antes de serem disponibilizados?,"Em caso afirmativo, informe o método utilizado: \n(Exemplo: validação automática com base em padrões da CETESB)"
7,Acre (AC),Não,Não,0,Não,0,Sim,26,Não,Inativação de estações,Não há perspectivas de curto prazo,NaN,Sim,https://map.purpleair.com/air-quality-raw-pm25...,Sim,Apenas seleção do Sensor que possui dados comp...
29,Acre (AC),Não,Não se aplica - não existem estações de referê...,0,Sim,1,Não,0,Não se aplica - não existem estações na UF,Não se aplica – não existem estações na UF,Não há perspectivas de curto prazo,NaN,Não se aplica - não existem estações na UF,NaN,Não se aplica - não existem estações na UF,X


In [172]:
# Importar arquivos do estado
base = Path.cwd().parent 
df_dir = base / "data" / "DADOS_BRUTOS" 
df_dir.mkdir(parents=True, exist_ok=True)
print(df_dir)

ac_dir = df_dir / uf
os.listdir(ac_dir)

/home/nobre/Notebooks/RQAR_2025_book/data/DADOS_BRUTOS


['.ipynb_checkpoints',
 'AC_MONITOR DE  QUALIDADE DO AR MÉDIA - BANCO DE DADOS - SENSOR PURPLE AIR BAIXO CUSTO - Ylza Lima.xlsx',
 'Erro',
 'AC_PM2.5_sensoresAB_2023 - Ylza Lima.csv']

In [177]:
# Como foram apresentadas particularidades nesse arquivo csv, foi necessário fazer uma alteração na função read,
# para transformar manualmente o tipo de encoding

def _read_csv(path, sep=None, decimal=None, encoding="cp1252", **kw):
    head = path.read_bytes()[:4096]
    txt  = head.decode(encoding or "utf-8", errors="ignore")
    sep_guess = sep or (";" if txt.count(";") > txt.count(",") else ",")
    dec_guess = decimal or ("," if re.search(r"\d+,\d+", txt) and txt.count(",") > txt.count(".") else ".")
    enc_guess = encoding or ("utf-8-sig" if txt.startswith("\ufeff")
                             else ("latin-1" if ("Ã" in txt or "�" in txt) else "utf-8"))
    try:
        return pd.read_csv(path, sep=sep_guess, decimal=dec_guess, encoding=enc_guess, engine="c", **kw)
    except Exception:
        return pd.read_csv(path, sep=None, engine="python", decimal=dec_guess, encoding=enc_guess, **kw)

In [256]:
ac_csv = load_csvs(ac_dir)
for name, df in ac_csv.items():
    print(f"\n=== {name} ===")
    display(df.head())

Loaded AC_PM2.5_sensoresAB_2023 - Ylza Lima.csv: (394, 57)

=== AC_PM2.5_sensoresAB_2023 - Ylza Lima ===


,BD - QUALIDADE DO AR - MÉDIA DIÁRIA PM2.5 (?g/m3) - 2023 (limite OMS: 15 ?g/m3),Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,...,Unnamed: 47,Unnamed: 48,Unnamed: 49,Unnamed: 50,Unnamed: 51,Unnamed: 52,Unnamed: 53,Unnamed: 54,Unnamed: 55,Unnamed: 56
0,NaN,Rio Branco,Rio Branco,Rio Branco,Rio Branco,Rio Branco,Rio Branco,Rio Branco,Rio Branco,Bujari,...,Epitaciolândia,Epitaciolândia,Brasiléia,Brasiléia,Brasiléia,Brasiléia,Santa Rosa do Purus,Santa Rosa do Purus,Porto Walter,Porto Walter
1,NaN,Ministério Público do Estado do Acre (SEDE) A,Ministério Público do Estado do Acre (SEDE) B,UFAC A,UFAC B,RB-BACKUP (Estação particular FB) A,RB-BACKUP (Estação particular FB) B,AcreBioClima - UFAC A,AcreBioClima - UFAC B,MPAC_BJR_01_promotoria A,...,MPAC_EPL_02_escola.joao.pedro A,MPAC_EPL_02_escola.joao.pedro B,MPAC_BRL_02_radio fm 90.3 A,MPAC_BRL_02_radio fm 90.3 B,MPAC_BRL_01_promotoria A,MPAC_BRL_01_promotoria B,MPAC_SRP_01_prefeitura A,MPAC_SRP_01_prefeitura B,MPAC_PTW_01_prefeitura A,MPAC_PTW_01_prefeitura B
2,01/01/2023,NaN,NaN,"1,09","1,15",NaN,NaN,NaN,NaN,"0,00",...,"1,88","2,22",NaN,NaN,"1,34","1,41","2,10","1,81",NaN,NaN
3,02/01/2023,NaN,NaN,"1,43","1,51",NaN,NaN,NaN,NaN,"0,00",...,"3,03","3,36",NaN,NaN,"2,26","2,22","3,25","2,87",NaN,NaN
4,03/01/2023,NaN,NaN,"0,73","0,78",NaN,NaN,NaN,NaN,"0,00",...,"2,09","2,27",NaN,NaN,"1,56","1,49","1,53","1,46",NaN,NaN


In [290]:
ac_long = pd.DataFrame(list(ac_csv.values())[0])
ac_long.head()

,BD - QUALIDADE DO AR - MÉDIA DIÁRIA PM2.5 (?g/m3) - 2023 (limite OMS: 15 ?g/m3),Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,...,Unnamed: 47,Unnamed: 48,Unnamed: 49,Unnamed: 50,Unnamed: 51,Unnamed: 52,Unnamed: 53,Unnamed: 54,Unnamed: 55,Unnamed: 56
0,NaN,Rio Branco,Rio Branco,Rio Branco,Rio Branco,Rio Branco,Rio Branco,Rio Branco,Rio Branco,Bujari,...,Epitaciolândia,Epitaciolândia,Brasiléia,Brasiléia,Brasiléia,Brasiléia,Santa Rosa do Purus,Santa Rosa do Purus,Porto Walter,Porto Walter
1,NaN,Ministério Público do Estado do Acre (SEDE) A,Ministério Público do Estado do Acre (SEDE) B,UFAC A,UFAC B,RB-BACKUP (Estação particular FB) A,RB-BACKUP (Estação particular FB) B,AcreBioClima - UFAC A,AcreBioClima - UFAC B,MPAC_BJR_01_promotoria A,...,MPAC_EPL_02_escola.joao.pedro A,MPAC_EPL_02_escola.joao.pedro B,MPAC_BRL_02_radio fm 90.3 A,MPAC_BRL_02_radio fm 90.3 B,MPAC_BRL_01_promotoria A,MPAC_BRL_01_promotoria B,MPAC_SRP_01_prefeitura A,MPAC_SRP_01_prefeitura B,MPAC_PTW_01_prefeitura A,MPAC_PTW_01_prefeitura B
2,01/01/2023,NaN,NaN,"1,09","1,15",NaN,NaN,NaN,NaN,"0,00",...,"1,88","2,22",NaN,NaN,"1,34","1,41","2,10","1,81",NaN,NaN
3,02/01/2023,NaN,NaN,"1,43","1,51",NaN,NaN,NaN,NaN,"0,00",...,"3,03","3,36",NaN,NaN,"2,26","2,22","3,25","2,87",NaN,NaN
4,03/01/2023,NaN,NaN,"0,73","0,78",NaN,NaN,NaN,NaN,"0,00",...,"2,09","2,27",NaN,NaN,"1,56","1,49","1,53","1,46",NaN,NaN


In [299]:
# Como a planilha de dados possui diversas particularidades, precisa passar por uma limpeza

df = ac_long.copy()

# 1) find the header start row
idx = df.index[(df.iloc[:, 0].isna()) & (df.notna().sum(axis=1) >= 3)]
start = int(idx[0]) if len(idx) else 0
sub = df.iloc[start:].reset_index(drop=True)

# 2) build column names: row 1 has stations from col 1 onward
stations = (
    sub.iloc[1, 1:]           # row with station names, skip first col
       .astype(str).str.strip()
       .tolist()
)

new_cols = ["DATETIME"] + stations 

ac_dfc = sub.iloc[2:].copy() 
ac_dfc.columns = new_cols 
ac_dfc.head()

,DATETIME,Ministério Público do Estado do Acre (SEDE) A,Ministério Público do Estado do Acre (SEDE) B,UFAC A,UFAC B,RB-BACKUP (Estação particular FB) A,RB-BACKUP (Estação particular FB) B,AcreBioClima - UFAC A,AcreBioClima - UFAC B,MPAC_BJR_01_promotoria A,...,MPAC_EPL_02_escola.joao.pedro A,MPAC_EPL_02_escola.joao.pedro B,MPAC_BRL_02_radio fm 90.3 A,MPAC_BRL_02_radio fm 90.3 B,MPAC_BRL_01_promotoria A,MPAC_BRL_01_promotoria B,MPAC_SRP_01_prefeitura A,MPAC_SRP_01_prefeitura B,MPAC_PTW_01_prefeitura A,MPAC_PTW_01_prefeitura B
2,01/01/2023,NaN,NaN,"1,09","1,15",NaN,NaN,NaN,NaN,"0,00",...,"1,88","2,22",NaN,NaN,"1,34","1,41","2,10","1,81",NaN,NaN
3,02/01/2023,NaN,NaN,"1,43","1,51",NaN,NaN,NaN,NaN,"0,00",...,"3,03","3,36",NaN,NaN,"2,26","2,22","3,25","2,87",NaN,NaN
4,03/01/2023,NaN,NaN,"0,73","0,78",NaN,NaN,NaN,NaN,"0,00",...,"2,09","2,27",NaN,NaN,"1,56","1,49","1,53","1,46",NaN,NaN
5,04/01/2023,NaN,NaN,"1,57","1,66",NaN,NaN,NaN,NaN,"0,00",...,"1,21","1,37",NaN,NaN,"1,08","1,05","1,78","1,68",NaN,NaN
6,05/01/2023,NaN,NaN,"0,00","0,00",NaN,NaN,NaN,NaN,"0,00",...,"0,62","1,05",NaN,NaN,"0,00","0,02","0,00","0,00",NaN,NaN


In [300]:
station_cols = ['Ministério Público do Estado do Acre (SEDE) A',
       'Ministério Público do Estado do Acre (SEDE) B', 'UFAC A', 'UFAC B',
       'RB-BACKUP (Estação particular FB) A',
       'RB-BACKUP (Estação particular FB) B', 'AcreBioClima - UFAC A',
       'AcreBioClima - UFAC B', 'MPAC_BJR_01_promotoria A',
       'MPAC_BJR_01_promotoria B', 'MPAC_SNG_01_promotoria A',
       'MPAC_SNG_01_promotoria B', 'MPAC_PTA_01_Sec.infraestrutura A',
       'MPAC_PTA_01_Sec.infraestrutura B', 'MPAC_ACL_01_promotoria A',
       'MPAC_ACL_01_promotoria B', 'MPAC_CPX_01_qpm A', 'MPAC_CPX_01_qpm B',
       'MPAC_XAP_02_promotoria A', 'MPAC_XAP_02_promotoria B',
       'MPAC_ABR_01_promotoria A', 'MPAC_ABR_01_promotoria B',
       'MPAC_ABR_02_SEMSA A', 'MPAC_ABR_02_SEMSA B',
       'MPAC_PLC_01_promotoria A', 'MPAC_PLC_01_promotoria B',
       'MPAC_SNM_01_ifac A', 'MPAC_SNM_01_ifac B', 'MPAC_SNM_02_promotoria A',
       'MPAC_SNM_02_promotoria B', 'MPAC_MNU_01_promotoria A',
       'MPAC_MNU_01_promotoria B', 'MPAC_FIJ_01_promotoria A',
       'MPAC_FIJ_01_promotoria B', 'MPAC_TRC_02_ifac A', 'MPAC_TRC_02_ifac B',
       'MPAC_JRD_01_prefeitura A', 'MPAC_JRD_01_prefeitura B',
       'MPAC_RDA_01_prefeitura A', 'MPAC_RDA_01_prefeitura B',
       'MPAC_CZS_02_ciosp A', 'MPAC_CZS_02_ciosp B', 'UFACFloresta A',
       'UFACFloresta B', 'MPAC_MTH_01_semec A', 'MPAC_MTH_01_semec B',
       'MPAC_EPL_02_escola.joao.pedro A', 'MPAC_EPL_02_escola.joao.pedro B',
       'MPAC_BRL_02_radio fm 90.3 A', 'MPAC_BRL_02_radio fm 90.3 B',
       'MPAC_BRL_01_promotoria A', 'MPAC_BRL_01_promotoria B',
       'MPAC_SRP_01_prefeitura A', 'MPAC_SRP_01_prefeitura B',
       'MPAC_PTW_01_prefeitura A', 'MPAC_PTW_01_prefeitura B']

ac_full = ac_dfc.melt(
    id_vars=['DATETIME'],   # keep these as is
    value_vars=station_cols,                      # columns to unpivot
    var_name='ESTACAO',                           # new col with station names
    value_name='VALOR'                            # new col with numeric values
)
ac_full

,DATETIME,ESTACAO,VALOR
0,01/01/2023,Ministério Público do Estado do Acre (SEDE) A,NaN
1,02/01/2023,Ministério Público do Estado do Acre (SEDE) A,NaN
2,03/01/2023,Ministério Público do Estado do Acre (SEDE) A,NaN
3,04/01/2023,Ministério Público do Estado do Acre (SEDE) A,NaN
4,05/01/2023,Ministério Público do Estado do Acre (SEDE) A,NaN
...,...,...,...
21947,NaN,MPAC_PTW_01_prefeitura B,NaN
21948,NaN,MPAC_PTW_01_prefeitura B,NaN
21949,NaN,MPAC_PTW_01_prefeitura B,NaN
21950,NaN,MPAC_PTW_01_prefeitura B,NaN


In [318]:
ac_station = build_inventory_rows(ac_full.copy(), df_cod, uf, datetime_col='DATETIME', mode="rows", drop_after_underscore=True)
ac_station.head()

,UF,ID_OEMA,CIDADE,ID_MMA,ID_MMA_COMPLETO,POLUENTE,COD_POLUENTE,CD_MUN,COD_UF_IBGE,PROPRIETARIO,...,REALOCACAO,OBS_CALIBRACAO,INICIO,FIM,DADOS_MONITORAMENTO,RECONHECIDA,OBS_GERAIS,CERTIFICACAO,STATUS,REP_ESPACIAL_DECLARADA
0,AC,AcreBioClima - UFAC A,,,,,,,12,,...,,,2023,2023,,,,,,
1,AC,AcreBioClima - UFAC B,,,,,,,12,,...,,,2023,2023,,,,,,
2,AC,MPAC_ABR_01_promotoria A,,,,,,,12,,...,,,2023,2023,,,,,,
3,AC,MPAC_ABR_01_promotoria B,,,,,,,12,,...,,,2023,2023,,,,,,
4,AC,MPAC_ABR_02_SEMSA A,,,,,,,12,,...,,,2023,2023,,,,,,


In [319]:
df = ac_long.copy()

# 1) find the header start row
idx = df.index[(df.iloc[:, 0].isna()) & (df.notna().sum(axis=1) >= 3)]
start = int(idx[0]) if len(idx) else 0
sub = df.iloc[start:].reset_index(drop=True)

# From your cleaned 'sub' with two header rows:
cities   = sub.iloc[0, 1:].astype(str).tolist()      
stations = sub.iloc[1, 1:].astype(str).tolist()     

# 1) station -> city
station_to_city = dict(zip(stations, cities))        

# 2) map onto your dataframe
ac_station["CIDADE"] = ac_station["ID_OEMA"].map(station_to_city)

In [322]:
cols = {
    "POLUENTE": "MP25",
    "CATEGORIA": "Indicativa",
    "MARCA": "PurpleAir",
}

targets = list(cols)

# normalize empty strings and whitespace to NA
ac_station[targets] = ac_station[targets].replace(r"^\s*$", pd.NA, regex=True)

# fill only where missing
for c, v in cols.items():
    ac_station.loc[ac_station[c].isna(), c] = v

In [323]:
ac_station = assign_id_mma(ac_station)
ac_station.head()

,UF,ID_OEMA,CIDADE,ID_MMA,ID_MMA_COMPLETO,POLUENTE,COD_POLUENTE,CD_MUN,COD_UF_IBGE,PROPRIETARIO,...,REALOCACAO,OBS_CALIBRACAO,INICIO,FIM,DADOS_MONITORAMENTO,RECONHECIDA,OBS_GERAIS,CERTIFICACAO,STATUS,REP_ESPACIAL_DECLARADA
0,AC,AcreBioClima - UFAC A,Rio Branco,AC0001,AC0001,MP25,,,12,,...,,,2023,2023,,,,,,
1,AC,AcreBioClima - UFAC B,Rio Branco,AC0002,AC0002,MP25,,,12,,...,,,2023,2023,,,,,,
2,AC,Ministério Público do Estado do Acre (SEDE) A,Rio Branco,AC0003,AC0003,MP25,,,12,,...,,,2023,2023,,,,,,
3,AC,Ministério Público do Estado do Acre (SEDE) B,Rio Branco,AC0004,AC0004,MP25,,,12,,...,,,2023,2023,,,,,,
4,AC,MPAC_ABR_01_promotoria A,Assis Brasil,AC0005,AC0005,MP25,,,12,,...,,,2023,2023,,,,,,


In [324]:
save_UF_estacoes_csv(ac_station, uf)

Saved to: /home/nobre/Notebooks/RQAR_2025_book/data/DADOS_ESTACOES/AC_estacoes.csv


##### g) Mato Grosso do Sul

In [325]:
# Substituir pelo estado desejado
uf = "MS" 

In [326]:
# Conferir respostas do formulário 
ms_forms = forms[forms["Unidade da Federação: "].str.contains(uf, case=False, na=False)]
ms_forms.iloc[:, idxs].head()

,Unidade da Federação:,Em sua UF há estações de monitoramento da qualidade do ar com equipamentos certificados como de referência ou equivalentes? \n,"Em caso afirmativo, a UF é responsável pela operação de algumas dessas estações?",Informe a quantidade de estações de operação própria:,Em sua UF há estações instaladas como condicionante de licença ambiental?,Informe a quantidade de estações instaladas por licença ambiental:,Em sua UF há estações de monitoramento indicativas (com sensores de baixo custo)?,Informe a quantidade de estações indicativas:,"Em 2024, houve mudanças na rede de monitoramento da qualidade do ar na UF?","Em caso afirmativo, indique quais mudanças ocorreram:",Quais são as perspectivas de curto prazo (1 ano) para a rede de monitoramento?,Deseja incluir observações sobre a rede de monitoramento da qualidade do ar? \n,Os dados de monitoramento da qualidade do ar são disponibilizados ao público?,"Em caso afirmativo, informe o endereço eletrônico para acesso:",Os dados de monitoramento passam por algum tratamento e formatação antes de serem disponibilizados?,"Em caso afirmativo, informe o método utilizado: \n(Exemplo: validação automática com base em padrões da CETESB)"
19,Mato Grosso do Sul (MS),Sim,Não,0,Sim,4,Não,0,Não,Não se aplica – não existem estações na UF,Início do monitoramento com estações de baixo ...,Existe proposta de instalação de equipamentos ...,Sim,https://monitorar.mma.gov.br/mapa e https://ww...,Sim,Validação automática e revisão de segurança se...


In [329]:
# Importar arquivos do estado
base = Path.cwd().parent 
df_dir = base / "data" / "DADOS_BRUTOS" 
df_dir.mkdir(parents=True, exist_ok=True)
print(df_dir)

ms_dir = df_dir / uf

/home/nobre/Notebooks/RQAR_2025_book/data/DADOS_BRUTOS


In [335]:
ms_dfs = load_excels(ms_dir)

Loaded MS_2024.xlsx [sheet 0]: (212048, 6)
Loaded MS_Dados RMQAR 2022.xlsx [sheet 0]: (130273, 6)
Loaded MS_monitoramento_2023.xlsx [sheet 0]: (212048, 6)


In [366]:
for name, df in ms_dfs.items():
    print(f"\n=== {name} ===")
    display(df.head())


=== MS_2024 ===


,Estação,Parâmetro Nome,Sigla,Valor Medido,Data,Hora
0,Suzano - Ribas do Rio Pardo,Partículas Totais em Suspensão,PTS,718.0,08/10/2024,18:00
1,Suzano - Ribas do Rio Pardo,Partículas Totais em Suspensão,PTS,717.0,06/07/2024,19:00
2,Suzano - Ribas do Rio Pardo,Partículas Totais em Suspensão,PTS,714.0,02/09/2024,19:00
3,Suzano - Ribas do Rio Pardo,Partículas Totais em Suspensão,PTS,707.0,23/09/2024,18:00
4,Suzano - Ribas do Rio Pardo,Partículas Totais em Suspensão,PTS,705.0,26/07/2024,21:00



=== MS_Dados RMQAR 2022 ===


,Estação,Sigla,Parâmetro Nome,Valor Medido,Data,Hora
0,Eldorado Brasil Celulose S.A.,NaN,Chuva,1.368663,01/01/2022,00:00
1,Eldorado Brasil Celulose S.A.,NaN,Chuva,0.815571,01/01/2022,01:00
2,Eldorado Brasil Celulose S.A.,NaN,Chuva,1.535701,01/01/2022,02:00
3,Eldorado Brasil Celulose S.A.,NaN,Chuva,1.438458,01/01/2022,03:00
4,Eldorado Brasil Celulose S.A.,NaN,Chuva,0.885628,01/01/2022,04:00



=== MS_monitoramento_2023 ===


,Estação,Parâmetro Nome,Sigla,Valor Medido,Data,Hora
0,Suzano - Ribas do Rio Pardo,Partículas Totais em Suspensão,PTS,718.0,08/10/2024,18:00
1,Suzano - Ribas do Rio Pardo,Partículas Totais em Suspensão,PTS,717.0,06/07/2024,19:00
2,Suzano - Ribas do Rio Pardo,Partículas Totais em Suspensão,PTS,714.0,02/09/2024,19:00
3,Suzano - Ribas do Rio Pardo,Partículas Totais em Suspensão,PTS,707.0,23/09/2024,18:00
4,Suzano - Ribas do Rio Pardo,Partículas Totais em Suspensão,PTS,705.0,26/07/2024,21:00


In [370]:
for name, df in ms_dfs.items():
    df["DATETIME"] = pd.to_datetime(df["Data"] + " " + df["Hora"], format="%d/%m/%Y %H:%M")

In [373]:
ms_full = merge_by_datetime_union(ms_dfs, keep_source=True)
ms_full.head()

,DATETIME,Data,Estação,Hora,Parâmetro Nome,Sigla,Valor Medido,__source
0,2024-10-08 18:00:00,08/10/2024,Suzano - Ribas do Rio Pardo,18:00,Partículas Totais em Suspensão,PTS,718.0,MS_2024
1,2024-07-06 19:00:00,06/07/2024,Suzano - Ribas do Rio Pardo,19:00,Partículas Totais em Suspensão,PTS,717.0,MS_2024
2,2024-09-02 19:00:00,02/09/2024,Suzano - Ribas do Rio Pardo,19:00,Partículas Totais em Suspensão,PTS,714.0,MS_2024
3,2024-09-23 18:00:00,23/09/2024,Suzano - Ribas do Rio Pardo,18:00,Partículas Totais em Suspensão,PTS,707.0,MS_2024
4,2024-07-26 21:00:00,26/07/2024,Suzano - Ribas do Rio Pardo,21:00,Partículas Totais em Suspensão,PTS,705.0,MS_2024


In [374]:
manual_map = {"Estação":  "ESTACAO",
              "Sigla":  "POLUENTE"
}

ms_full = ms_full.rename(columns=manual_map) 

In [375]:
ms_full = convert_column_to_datetime(ms_full, column_name="DATETIME", format="%d/%m/%Y %H:%M")
# ms_full = ms_full.sort_values("DATETIME", ascending=False).reset_index(drop=True)
ms_full.head()

,DATETIME,Data,ESTACAO,Hora,Parâmetro Nome,POLUENTE,Valor Medido,__source
0,2024-10-08 18:00:00,08/10/2024,Suzano - Ribas do Rio Pardo,18:00,Partículas Totais em Suspensão,PTS,718.0,MS_2024
1,2024-07-06 19:00:00,06/07/2024,Suzano - Ribas do Rio Pardo,19:00,Partículas Totais em Suspensão,PTS,717.0,MS_2024
2,2024-09-02 19:00:00,02/09/2024,Suzano - Ribas do Rio Pardo,19:00,Partículas Totais em Suspensão,PTS,714.0,MS_2024
3,2024-09-23 18:00:00,23/09/2024,Suzano - Ribas do Rio Pardo,18:00,Partículas Totais em Suspensão,PTS,707.0,MS_2024
4,2024-07-26 21:00:00,26/07/2024,Suzano - Ribas do Rio Pardo,21:00,Partículas Totais em Suspensão,PTS,705.0,MS_2024


In [378]:
ms_station = build_inventory_rows(ms_full.copy(), df_cod, uf, datetime_col='DATETIME', mode="rows")
ms_station = assign_id_mma(ms_station)
ms_station

,UF,ID_OEMA,CIDADE,ID_MMA,ID_MMA_COMPLETO,POLUENTE,COD_POLUENTE,CD_MUN,COD_UF_IBGE,PROPRIETARIO,...,REALOCACAO,OBS_CALIBRACAO,INICIO,FIM,DADOS_MONITORAMENTO,RECONHECIDA,OBS_GERAIS,CERTIFICACAO,STATUS,REP_ESPACIAL_DECLARADA
0,MS,Eldorado Brasil Celulose S.A.,,MS0001,MS0001,"CO,MP10,MP25,NO2,O3,PTS",,,50,,...,,,2022,2024,,,,,,
1,MS,Petrobras - UTE,,MS0002,MS0002,"CO,NO2,O3",,,50,,...,,,2022,2024,,,,,,
2,MS,Suzano TLS1-VCPTL - Três Lagoas,,MS0003,MS0003,"CO,H2S,MP10,NO2,O3,PTS,SO2",,,50,,...,,,2022,2024,,,,,,
3,MS,Suzano - Ribas do Rio Pardo,,MS0004,MS0004,"MP10,MP25,NO2,O3,PTS,SO2",,,50,,...,,,2024,2024,,,,,,


In [382]:
cols = {
    "CATEGORIA": "Referencia",
}

targets = list(cols)
# normalize empty strings and whitespace to NA
ms_station[targets] = ms_station[targets].replace(r"^\s*$", pd.NA, regex=True)
# fill only where missing
for c, v in cols.items():
    ms_station.loc[ms_station[c].isna(), c] = v

In [383]:
save_UF_estacoes_csv(ms_station, uf)

Saved to: /home/nobre/Notebooks/RQAR_2025_book/data/DADOS_ESTACOES/MS_estacoes.csv
